In [ ]:
# CELL 1 — Environment setup, imports, and global configuration

# This cell loads the libraries used throughout the notebook and defines
# the global paths, plotting settings, and model parameters shared by all
# subsequent analyses.

!pip install yfinance statsmodels seaborn scipy --quiet

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf
from statsmodels.stats.diagnostic import acorr_ljungbox

drive.mount('/content/drive')


# Base path on Google Drive

BASE_DIR = "/content/drive/MyDrive/tesi_volatilita"
os.makedirs(BASE_DIR, exist_ok=True)


# Global parameters shared across all cells

EPS            = 1e-12   # to avoid log(0)
L_PERSIST_BASE = 120     # minimum span of the filter EWM persistent
C_SPAN         = 40      # span_p = max(L_PERSIST_BASE, C_SPAN * h)

print("BASE_DIR:", BASE_DIR)
print(f"L_PERSIST_BASE: {L_PERSIST_BASE}  |  C_SPAN: {C_SPAN}")


In [ ]:
# CELL 2 — Download of daily prices from Yahoo Finance

# Daily closing prices are retrieved for five major international equity
# indices over the 2000–2026 sample and then saved locally for the
# subsequent empirical analysis.
import yfinance as yf

tickers = {
    "FTSE":  "^FTSE",
    "GSPC":  "^GSPC",
    "N225":  "^N225",
    "HSI":   "^HSI",
    "GDAXI": "^GDAXI"
}

start_date = "2000-03-06"
end_date   = "2026-03-07"

all_close = pd.DataFrame()

for name, ticker in tickers.items():
    tmp = yf.download(
        ticker,
        start=start_date,
        end=end_date,
        interval="1d",
        auto_adjust=False,
        progress=False
    )
    all_close[name] = tmp["Close"]

all_close.index.name = "Date"
all_close = all_close.sort_index()

print("Shape before dropping NaNs:", all_close.shape)
print("\nNaN per index:")
print(all_close.isna().sum())

# Keep only dates common to all indices
all_close = all_close.dropna().copy()

print("\nShape after dropna:", all_close.shape)
print("Temporal order correct:", all_close.index.is_monotonic_increasing)

out = f"{BASE_DIR}/indici_daily_2000_2026.csv"
all_close.to_csv(out, encoding="utf-8")
print("\nSaved to:", out)
all_close.head()


In [ ]:
# CELL 3 — Data cleaning and alignment on common trading dates

# The raw price panel is cleaned by retaining only the dates for which all
# indices have valid observations in order to produce a balanced dataset that is
# directly comparable across markets.

daily_csv = f"{BASE_DIR}/indici_daily_2000_2026.csv"
clean_csv = f"{BASE_DIR}/indici_clean_allineati.csv"

df = pd.read_csv(daily_csv, parse_dates=["Date"], index_col="Date").sort_index()

print("Raw dataset shape:", df.shape)
print("\nNaN per index:")
print(df.isna().sum())


df_clean = df.dropna(how="any").copy() # Keep the dates common to all indexes

print("\nClean dataset shape:", df_clean.shape)
print("Index sorted:", df_clean.index.is_monotonic_increasing)
print("Remaining NaNs:", df_clean.isna().sum().sum())

df_clean.to_csv(clean_csv, encoding="utf-8")
print("\nSaved:", clean_csv)
df_clean.head()


In [ ]:
# CELL 4 — Normalised price series (base 100 at 2000-03-06)
#
# This plot shows the evolution of the five equity indices over the full
# sample period, rebased to 100 at the start date. The normalisation allows
# for a direct visual comparison of cumulative performance across markets
# with very different absolute price levels.
#
# Key episodes visible in the chart:
#   - Dot-com bust (2000-2002)
#   - Global Financial Crisis (2007-2009)
#   - European sovereign debt crisis (2011-2012)
#   - COVID-19 shock (February-March 2020)
#   - Monetary tightening cycle (2022-2023)

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os

df_clean = pd.read_csv(
    f"{BASE_DIR}/indici_clean_allineati.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

# Rebase to 100 at the first available date
df_norm = df_clean.div(df_clean.iloc[0]) * 100

# ── plot ──────────────────────────────────────────────────────────────────────
colors = {
    "FTSE":  "#1f77b4",   # blue
    "GSPC":  "#d62728",   # red
    "N225":  "#2ca02c",   # green
    "HSI":   "#ff7f0e",   # orange
    "GDAXI": "#9467bd"    # purple
}

labels = {
    "FTSE":  "FTSE 100 (UK)",
    "GSPC":  "S&P 500 (US)",
    "N225":  "Nikkei 225 (JP)",
    "HSI":   "Hang Seng (HK)",
    "GDAXI": "DAX (DE)"
}

fig, ax = plt.subplots(figsize=(14, 6))

for col in df_norm.columns:
    ax.plot(df_norm.index, df_norm[col],
            color=colors[col], label=labels[col],
            linewidth=1.2, alpha=0.9)

# Annotate key crisis episodes
crises = {
    "GFC\n(2008)":  "2008-09-15",
    "COVID-19\n(2020)": "2020-03-16",
    "Rate hike\ncycle (2022)": "2022-01-03",
}

for label, date in crises.items():
    ax.axvline(pd.Timestamp(date), color="grey",
               linestyle="--", linewidth=0.8, alpha=0.6)
    ax.text(pd.Timestamp(date), ax.get_ylim()[1] * 0.97,
            label, fontsize=7.5, color="grey",
            ha="center", va="top")

ax.set_title("Equity indices — normalised price series (base 100, March 2000)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Index value (base 100)", fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
ax.legend(loc="upper left", fontsize=9, framealpha=0.7)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()

OUT_DIR = f"{BASE_DIR}/graphics_final_thesis"
os.makedirs(OUT_DIR, exist_ok=True)
fig.savefig(f"{OUT_DIR}/grafico_prezzi_normalizzati.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: grafico_prezzi_normalizzati.png")

In [ ]:
# CELL 5 — Log-returns and volatility proxies

# This cell computes daily log-returns together with the standard
# volatility proxies used in the thesis, namely |r_t|, r_t^2, and
# log(r_t^2). The resulting datasets are exported for later use.

df_clean = pd.read_csv(
    f"{BASE_DIR}/indici_clean_allineati.csv",
    parse_dates=["Date"],
    index_col="Date"
).sort_index()

# Log-returns: r_t = log(P_t / P_{t-1})
log_returns = np.log(df_clean).diff().dropna()
log_returns.columns = [f"{c}_ret" for c in log_returns.columns]

# Proxy of volatility
abs_returns  = log_returns.abs()
sq_returns   = log_returns.pow(2)
logsq_returns = np.log(sq_returns.clip(lower=EPS))

abs_returns.columns   = [c.replace("_ret", "_absret")   for c in abs_returns.columns]
sq_returns.columns    = [c.replace("_ret", "_sqret")    for c in sq_returns.columns]
logsq_returns.columns = [c.replace("_ret", "_logsqret") for c in logsq_returns.columns]


log_returns.to_csv(f"{BASE_DIR}/indici_log_returns.csv",         encoding="utf-8")
abs_returns.to_csv(f"{BASE_DIR}/indici_abs_returns.csv",          encoding="utf-8")
sq_returns.to_csv(f"{BASE_DIR}/indici_squared_returns.csv",       encoding="utf-8")
logsq_returns.to_csv(f"{BASE_DIR}/indici_log_squared_returns.csv", encoding="utf-8")

print("Saved returns and volatility proxies to:", BASE_DIR)
print("\nLog returns shape:", log_returns.shape)
print("NaNs in returns:", log_returns.isna().sum().sum())
log_returns.head()


In [ ]:
# CELL 6 — Identification of market regimes (growth vs crisis)

# For each index, market regimes are classified using a rolling Sharpe-type
# score. The resulting labels distinguish expansionary, neutral, and crisis
# phases, while the drawdown from the historical peak is shown on the
# secondary axis to improve economic interpretation.

df = pd.read_csv(
    f"{BASE_DIR}/indici_clean_allineati.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

OUT_DIR = f"{BASE_DIR}/grafici_regimi_colorati"
os.makedirs(OUT_DIR, exist_ok=True)

TH_UP, TH_DOWN = 0.50, -0.50
colors = {"Growth": "#2ca02c", "Neutral": "#f2c14e", "Crisis": "#d62728"}

def shade(ax, state, alpha=0.12):
    grp = (state != state.shift()).cumsum()
    for _, seg in state.groupby(grp):
        ax.axvspan(seg.index[0], seg.index[-1],
                   color=colors[seg.iloc[0]], alpha=alpha, lw=0)

for col in df.columns:
    p      = df[col].dropna()
    growth = p.rolling(252, min_periods=60).mean()
    crisis = -(p / p.cummax() - 1.0) * 100.0

    r       = np.log(p).diff()
    mu_ann  = r.rolling(126, min_periods=40).mean() * 252
    vol_ann = r.rolling(126, min_periods=40).std() * np.sqrt(252)
    score   = (mu_ann / (vol_ann + EPS)).ewm(span=20, min_periods=20).mean()

    state = pd.Series("Neutral", index=p.index)
    state.loc[score > TH_UP]  = "Growth"
    state.loc[score < TH_DOWN] = "Crisis"

    fig, ax1 = plt.subplots(figsize=(14, 6))
    ax2 = ax1.twinx()
    shade(ax1, state)

    ax1.plot(p.index, p,       lw=1.8, color="steelblue", label=f"{col} Price")
    ax1.plot(growth.index, growth, lw=2, color="green",  label="MA252")
    ax2.plot(crisis.index, crisis, lw=1.4, color="red",  label="Drawdown %")

    ax1.set_title(f"{col} — Market regimes")
    ax1.grid(True, alpha=0.25)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/{col}_market_regimes.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
# CELL 7 — Extreme return spikes and high-volatility episodes

# This visualization highlights two relevant types of events:
#   - extreme return realizations (|r_t| above the 99th percentile);
#   - persistent stress periods (30-day volatility above the 90th percentile).
# The goal is to relate large return movements to broader volatility regimes.
ret = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

OUT_DIR = f"{BASE_DIR}/grafici_returns"
os.makedirs(OUT_DIR, exist_ok=True)

def shade_crisis(ax, mask):
    grp = (mask != mask.shift()).cumsum()
    for _, seg in mask.groupby(grp):
        if seg.iloc[0]:
            ax.axvspan(seg.index[0], seg.index[-1],
                       color="red", alpha=0.10, lw=0)

for col in ret.columns:
    r      = ret[col].dropna()
    vol30  = r.rolling(30, min_periods=15).std()
    crisis = vol30 > vol30.quantile(0.90)
    spikes = r.abs() > r.abs().quantile(0.99)

    fig, ax = plt.subplots(figsize=(14, 5))
    shade_crisis(ax, crisis)
    ax.plot(r.index, r, color="steelblue", lw=0.8, label="Log-returns")
    ax.axhline(0, color="black", lw=1, ls="--")
    ax.scatter(r.index[spikes], r[spikes], color="darkred", s=15, label="Extreme spikes")
    ax.set_title(f"{col} — Log-returns")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/{col}_returns_crisi.png", dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


In [ ]:
# CELL 8 — Empirical return distribution versus Gaussian benchmark

# The empirical density of returns is compared with a fitted Gaussian
# density. In particular, excess kurtosis is used to assess the presence of
# fat tails and to motivate the use of Student-t innovations in the model.

ret = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

OUT_DIR = f"{BASE_DIR}/grafici_returns"
os.makedirs(OUT_DIR, exist_ok=True)

n = len(ret.columns)
fig, axes = plt.subplots(n, 1, figsize=(12, 3.2 * n))
if n == 1:
    axes = [axes]

for ax, col in zip(axes, ret.columns):
    r  = ret[col].dropna()
    mu = r.mean()
    sg = r.std(ddof=1)
    x  = np.linspace(r.quantile(0.001), r.quantile(0.999), 1000)
    gauss = (1 / (sg * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sg) ** 2)

    sns.kdeplot(r, ax=ax, color="red", lw=2, label="Empirical distribution")
    ax.plot(x, gauss, color="blue", lw=2, label="Fitted Gaussian")
    ax.set_title(f"{col}  |  Excess kurtosis = {r.kurtosis():.2f}")
    ax.grid(True, alpha=0.25)
    ax.legend()

fig.tight_layout()
out = f"{OUT_DIR}/ALL_returns_distribuzione_vs_gaussiana.png"
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved:", out)


In [ ]:
# CELL 9 — Descriptive statistics of log-returns
#
# This cell computes the main descriptive statistics for each index:
# mean, standard deviation, minimum, maximum, skewness, excess kurtosis.
# A Jarque-Bera normality test is also reported to formally reject
# the Gaussian hypothesis for all five indices.

from scipy import stats as scipy_stats

ret = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

rows = []
for col in ret.columns:
    r   = ret[col].dropna()
    idx = col.replace("_ret", "")
    jb_stat, jb_pval = scipy_stats.jarque_bera(r)
    rows.append({
        "Index":           idx,
        "Obs":             len(r),
        "Mean":            round(r.mean() * 100, 4),          # in %
        "Std Dev":         round(r.std(ddof=1) * 100, 4),     # in %
        "Min":             round(r.min() * 100, 4),            # in %
        "Max":             round(r.max() * 100, 4),            # in %
        "Skewness":        round(r.skew(), 4),
        "Excess Kurtosis": round(r.kurtosis(), 4),
        "JB Statistic":    round(jb_stat, 2),
        "JB p-value":      f"< 0.001" if jb_pval < 0.001 else round(jb_pval, 4)
    })

desc_df = pd.DataFrame(rows).set_index("Index")

print("=== Descriptive Statistics of Daily Log-Returns ===")
print("(Mean, Std Dev, Min, Max expressed in %)\n")
print(desc_df.to_string())

# Save to CSV for LaTeX table
desc_df.to_csv(f"{BASE_DIR}/descriptive_statistics.csv", encoding="utf-8")
print(f"\nSaved: {BASE_DIR}/descriptive_statistics.csv")

In [ ]:
# CELL 10 — Serial dependence diagnostics: ACF and Ljung–Box test

# This cell verifies the stylized fact of volatility clustering:
#   - the ACF of r_t should display limited linear autocorrelation;
#   - the ACF of r_t^2 should remain significantly positive across several lags.
# The Ljung–Box test complements the graphical evidence with a formal test.

ret = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

OUT_DIR  = f"{BASE_DIR}/grafici_acf"
os.makedirs(OUT_DIR, exist_ok=True)

LAGS_ACF = 60
LAGS_LB  = [10, 20, 30]

fig, axes = plt.subplots(len(ret.columns), 2,
                         figsize=(14, 2.8 * len(ret.columns)))
axes = np.atleast_2d(axes)

rows = []
for i, col in enumerate(ret.columns):
    r    = ret[col].dropna()
    r2   = r.pow(2)
    name = col.replace("_ret", "")

    plot_acf(r,  lags=LAGS_ACF, alpha=0.05, zero=False, ax=axes[i, 0])
    plot_acf(r2, lags=LAGS_ACF, alpha=0.05, zero=False, ax=axes[i, 1])
    axes[i, 0].set_title(f"{name} — ACF returns")
    axes[i, 1].set_title(f"{name} — ACF returns²")

    lb_r  = acorr_ljungbox(r,  lags=LAGS_LB, return_df=True)
    lb_r2 = acorr_ljungbox(r2, lags=LAGS_LB, return_df=True)
    for lag in LAGS_LB:
        rows.append({"Indice": name, "Serie": "r_t",   "Lag": lag,
                     "LB_stat": lb_r.loc[lag, "lb_stat"],
                     "p_value": lb_r.loc[lag, "lb_pvalue"]})
        rows.append({"Indice": name, "Serie": "r_t^2", "Lag": lag,
                     "LB_stat": lb_r2.loc[lag, "lb_stat"],
                     "p_value": lb_r2.loc[lag, "lb_pvalue"]})

fig.tight_layout()
fig.savefig(f"{OUT_DIR}/ALL_acf_returns_e_squared.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

lb = pd.DataFrame(rows)
lb["Reject_H0_5pct"] = lb["p_value"] < 0.05
lb.to_csv(f"{BASE_DIR}/ljung_box_results.csv", index=False, encoding="utf-8")
display(lb)

# Anomalous serial correlation in GSPC returns:
# The Ljung-Box statistic for r_t (not r_t²) of GSPC at lag 10 is approximately
# 110.6, far above the values of the other four indices (N225 ≈ 19.4,
# HSI ≈ 21.4, FTSE and GDAXI in the 10–25 range). This indicates strong linear
# autocorrelation in GSPC daily returns themselves, beyond the standard
# volatility-clustering pattern. Possible explanations include index microstructure
# effects (stale-price components, overnight gap regularities) or a regime-
# specific mean-reversion pattern. This anomaly is relevant for forecasting:
# the RP_Model adds predictive power through X_R and X_P — components derived
# from squared returns — but is not designed to exploit linear return
# autocorrelation. The absence of any RP advantage for GSPC in the benchmark
# (DM p-values > 0.5 across all horizons) is at least partly attributable to
# this non-standard dependence structure.


In [ ]:
# CELL 11 — Estimation of the Hurst exponent via DFA on log(r_t^2)

# Detrended Fluctuation Analysis (DFA, order 1) is applied to the
# log-squared-return proxy in order to estimate the Hurst exponent.
# Interpretation:
#   - H < 0.5: anti-persistent / rough dynamics
#   - H = 0.5: memoryless benchmark
#   - H > 0.5: persistent dynamics / long memory

df = pd.read_csv(
    f"{BASE_DIR}/indici_log_squared_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

def hurst_dfa(series, min_scale=10, n_scales=18,
              max_scale_cap=500, min_segments=6):
    """
    Estimation Hurst exponent through DFA(1)

    Parameters
    ---------
    series        : array-like — time series to analyze
    min_scale     : minimum scale of the window
    n_scales      : number of scales in logarithmic grid
    max_scale_cap : maximum value for the scale
    min_segments  : minimum number of segments per scale

    Come back
    -------
    H, DFA_alpha, R2_loglog, n_used_scales, used_scales, F_values
    """
    x = pd.Series(series).dropna().astype(float).values
    N = len(x)
    if N < 300:
        return np.nan, np.nan, np.nan, 0, [], []

    max_scale = min(N // 10, max_scale_cap)
    if max_scale <= min_scale:
        return np.nan, np.nan, np.nan, 0, [], []

    scales = np.unique(
        np.floor(
            np.logspace(np.log10(min_scale), np.log10(max_scale), n_scales)
        ).astype(int)
    )
    y = np.cumsum(x - x.mean())

    used_scales, F_values = [], []
    for s in scales:
        nseg = N // s
        if nseg < min_segments:
            continue
        y_cut    = y[:nseg * s].reshape(nseg, s)
        t        = np.arange(s)
        mse_seg  = [np.mean((seg - np.polyval(np.polyfit(t, seg, 1), t)) ** 2)
                    for seg in y_cut]
        Fs = np.sqrt(np.mean(mse_seg))
        if np.isfinite(Fs) and Fs > 0:
            used_scales.append(s)
            F_values.append(Fs)

    if len(F_values) < 6:
        return np.nan, np.nan, np.nan, len(used_scales), used_scales, F_values

    lx = np.log(used_scales)
    ly = np.log(F_values)
    alpha, intercept = np.polyfit(lx, ly, 1)

    H    = alpha if alpha <= 1 else alpha - 1
    H    = float(np.clip(H, 0, 1))
    yhat = alpha * lx + intercept
    ss_res = np.sum((ly - yhat) ** 2)
    ss_tot = np.sum((ly - ly.mean()) ** 2)
    r2   = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    return H, float(alpha), float(r2), len(used_scales), used_scales, F_values

rows = []
for col in df.columns:
    H, alpha, r2, n_used, used_scales, F_values = hurst_dfa(df[col])
    rows.append({
        "Series":         col,
        "Hurst_H":       H,
        "DFA_alpha":     alpha,
        "R2_loglog":     r2,
        "N_scales_used": n_used,
        "Min_scale":     min(used_scales) if used_scales else np.nan,
        "Max_scale":     max(used_scales) if used_scales else np.nan
    })

res = pd.DataFrame(rows).sort_values("Series").reset_index(drop=True)
res.to_csv(f"{BASE_DIR}/hurst_log_r2.csv", index=False, encoding="utf-8")

print("DFA/Hurst completed.")
display(res.round(4))

In [ ]:
# CELL 12 — DFA/Hurst for X_obs, X_R (rough), and X_P (persistent)

# The volatility proxy used here is
#   X_obs = 0.5 * log(EWMA(r^2, lambda = 0.94)) = log(sigma_EWMA).
# Compared with rolling log(r_t^2) at h = 1, this specification is more
# stable because the EWMA filter reduces measurement noise while preserving
# the persistence structure relevant for inference.
#
# The Hurst exponent estimated on X_obs (H_X) is
# expected to be anomalously low (0.32–0.38, apparently anti-persistent) because
# the EWMA filter with lambda = 0.94 introduces short-term negative auto-
# correlations in the smoothed series, which DFA registers as apparent anti-
# persistence. H_X must NOT be interpreted as evidence that log-volatility is
# rough. The economically meaningful estimates are H_R (rough component,
# H_R ≈ 0.20) and H_P (persistent component, H_P ≈ 0.77), obtained from the
# decomposed series and robust to this filtering artefact.

ret = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

EWMA_LAMBDA = 0.94   # lambda RiskMetrics standard

rows = []

for col in ret.columns:
    idx = col.split("_")[0]
    r   = ret[col].dropna()


    # Proxy log-sigma through EWMA (more stable wrt log(r²))
    # X_obs = log(sigma_EWMA) = 0.5 * log(EWMA(r²))

    ewma_var = r.pow(2).ewm(
        alpha=1 - EWMA_LAMBDA,
        adjust=False,
        min_periods=20
    ).mean()
    x_obs = 0.5 * np.log(ewma_var.clip(lower=EPS))


    # Decomposition EWM coherent with the model
    # span = max(L_PERSIST_BASE, C_SPAN * 1)

    span_p = max(L_PERSIST_BASE, C_SPAN * 1)
    x_p    = x_obs.ewm(span=span_p, adjust=False, min_periods=10).mean()
    x_r    = x_obs - x_p


    tmp = pd.concat(
        {"X_obs": x_obs, "X_R": x_r, "X_P": x_p},
        axis=1
    ).dropna()

    if len(tmp) < 400:
        print(f"  {idx}: sample too short, skipping.")
        continue


    # Estimation DFA for the 3 series

    H_X, a_X, r2_X, n_X, *_ = hurst_dfa(tmp["X_obs"])
    H_R, a_R, r2_R, n_R, *_ = hurst_dfa(tmp["X_R"])
    H_P, a_P, r2_P, n_P, *_ = hurst_dfa(tmp["X_P"])

    rows.append({
        "Indice":      idx,
        "N_obs":       len(tmp),
        "H_X":         H_X,  "DFA_alpha_X": a_X, "R2_X": r2_X,
        "H_R":         H_R,  "DFA_alpha_R": a_R, "R2_R": r2_R,
        "H_P":         H_P,  "DFA_alpha_P": a_P, "R2_P": r2_P
    })

    print(f"  {idx}:  H_X={H_X:.3f}  |  H_R={H_R:.3f} (rough)  |  H_P={H_P:.3f} (persistent)")

res = pd.DataFrame(rows).sort_values("Indice").reset_index(drop=True)
res.to_csv(f"{BASE_DIR}/hurst_componenti_R_P.csv", index=False, encoding="utf-8")

print("\nDFA/Hurst for components completed.")
print("Expected: H_R < 0.5 (rough)  |  H_P close to 1.0 (persistent)")
display(res.round(4))

In [ ]:
# CELL 13 — ACF of the latent components X_obs, X_R, and X_P
#
# This cell documents the memory structure of the log-volatility signal
# and its components using the log-squared return proxy log(r²),
# which is the standard proxy in the rough volatility literature
# (Gatheral, Jaisson, Rosenbaum 2018).
#
# Proxy:  X_obs = log(r²_t)  [standard rough volatility proxy]
# Filter: X_P   = rolling mean of X_obs over L_PERSIST_BASE = 120 days
# Rough:  X_R   = X_obs - X_P
#
# This decomposition is chosen for the ACF analysis because log(r²)
# preserves the high-frequency roughness that the EWMA proxy (λ=0.94)
# suppresses by construction. The EWMA proxy is used in Cell 12 (DFA),
# Cell 19 (model estimation), and Cell 26 (VaR) because it is a more
# stable input for regression and forecasting. The two proxies are
# complementary: log(r²) reveals the rough structure of the underlying
# volatility process; EWMA log-sigma provides a stable signal for
# prediction. The DFA results in Table 4.2 — which use the EWMA proxy —
# confirm that H(X_R) < 0.5 even after smoothing, consistently with
# the negative autocorrelations documented here on the raw proxy.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.stattools import acf

ret = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

logsq = pd.read_csv(
    f"{BASE_DIR}/indici_log_squared_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

OUT_DIR = f"{BASE_DIR}/grafici_acf_componenti_R_P"
os.makedirs(OUT_DIR, exist_ok=True)

MAX_LAG      = 60
ROLL_SPAN_P  = L_PERSIST_BASE   # 120 days — same constant used throughout

acf_rows = []

for col in logsq.columns:
    idx = col.split("_")[0]

    # ── X_obs: log(r²) — raw rough volatility proxy ───────────────────────
    x_obs = logsq[col].replace([np.inf, -np.inf], np.nan).dropna()

    # ── X_P: slow rolling mean — captures structural volatility level ─────
    x_p = x_obs.rolling(ROLL_SPAN_P, min_periods=ROLL_SPAN_P // 2).mean()

    # ── X_R: rough residual — transient deviations from structural level ──
    x_r = x_obs - x_p

    tmp = pd.concat(
        {"X_obs": x_obs, "X_R": x_r, "X_P": x_p},
        axis=1
    ).dropna()

    if len(tmp) < MAX_LAG + 10:
        print(f"  {idx}: sample too short, skipping.")
        continue

    # ── ACF plots ──────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    plot_acf(tmp["X_obs"], lags=MAX_LAG, alpha=0.05, zero=False, ax=axes[0])
    plot_acf(tmp["X_R"],   lags=MAX_LAG, alpha=0.05, zero=False, ax=axes[1])
    plot_acf(tmp["X_P"],   lags=MAX_LAG, alpha=0.05, zero=False, ax=axes[2])

    axes[0].set_title(f"{idx} — ACF $X_{{obs}}$ = log($r^2$)")
    axes[1].set_title(f"{idx} — ACF rough component $X_R$")
    axes[2].set_title(f"{idx} — ACF persistent component $X_P$")

    fig.suptitle(
        f"{idx} | Proxy: log($r^2$) | "
        f"X_P filter: rolling mean {ROLL_SPAN_P} days",
        fontsize=9, style="italic"
    )
    fig

In [ ]:
# CELL 14 — Multi-scale Monte Carlo simulation

# Model specification:
#   X_t^(h) = X_P,t + omega(h) * X_R,t
#   sigma_t = exp(X_t^(h))          # X_t = log(sigma), so sigma = exp(X)
#   r_t     = sigma_t * z_t,     with z_t ~ Student-t(nu)
#
# The weight omega(h) = C_OMEGA / (h + C_OMEGA) decreases with the horizon:
# roughness is more influential at short horizons, whereas persistence
# dominates at longer horizons. The simulation is used to assess whether
# the multi-scale structure reproduces the empirical stylized facts.

# C_OMEGA = 15 governs the horizon-dependent mixing weight in the simulation:
#   omega(h) = C_OMEGA / (h + C_OMEGA)
#
# The choice C_OMEGA = 15 is motivated by the effective memory window of the
# EWMA proxy with lambda = 0.94: 1/(1-lambda) ≈ 17 days, rounded to 15 for
# parsimony. At h = C_OMEGA = 15, the rough and persistent components
# contribute equally to the simulated log-volatility.
#
# This constant governs the SIMULATION ONLY and does not enter the
# forecasting model (Cell 19), where the relative contributions of X_R
# and X_P are determined freely by the OLS-estimated coefficients
# gamma_R and gamma_P. The simulation and the regression model are
# complementary tools: the former validates the qualitative properties
# of the decomposition, the latter estimates its predictive content.

import numpy as np
import pandas as pd
import os


# Monte Carlo parameters

RET_FILE = f"{BASE_DIR}/indici_log_returns.csv"
OUT_DIR  = f"{BASE_DIR}/mc_multiscale_results"
os.makedirs(OUT_DIR, exist_ok=True)

HORIZONS        = [1, 5, 20, 60]
C_OMEGA         = 15.0    # omega(h) = C_OMEGA / (h + C_OMEGA)
N_PATHS         = 3000
CHUNK           = 300
SEED            = 42
BURN_IN         = 500     # steps discarded to reach the stationary distribution
RETURN_INNOV    = "student"  # "t-student"
GAMMA_CLUSTER   = 0.35    # intensity clustering rough ← persistent
ROUGH_SCALE_MIN = 0.50
ROUGH_SCALE_MAX = 3.00
USE_NONOVERLAP  = False

rng = np.random.default_rng(SEED)


# Functions

def omega_of_h(h, c=C_OMEGA):
    """Rough weight decreasing with the forecast horizon"""
    return c / (h + c)

def log_rv_past(r, h, eps=EPS):
    """Log-sigma proxy: log( sqrt( mean(r²) ) ) = log(sigma)"""
    rv = np.sqrt(r.pow(2).rolling(window=h, min_periods=h).mean())
    return np.log(rv.clip(lower=eps))

def fit_ar1_simple(series):
    """OLS on x_t = c + phi * x_{t-1} + eps_t"""
    x = pd.Series(series).dropna().astype(float).values
    if len(x) < 50:
        return np.nan, np.nan, np.nan
    y, lag = x[1:], x[:-1]
    X = np.column_stack([np.ones_like(lag), lag])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    c   = float(beta[0])
    phi = float(np.clip(beta[1], -0.995, 0.995))
    resid = y - (c + phi * lag)
    return c, phi, float(max(np.std(resid, ddof=1), 1e-8))

def standardized_student_t(df_nu, size, rng):
    """Unit-variance standardised Student-t."""
    z = rng.standard_t(df=df_nu, size=size)
    return z * np.sqrt((df_nu - 2.0) / df_nu)

def estimate_student_df(z):
    """Estimate nu from excess kurtosis: nu ≈ 6/exk + 4."""
    exk = float(pd.Series(z).kurtosis())
    if not np.isfinite(exk) or exk <= 0.05:
        return np.inf, exk
    return float(np.clip(6.0 / exk + 4.0, 5.0, 80.0)), exk

def agg_std_by_horizon(r_mat, h, use_nonoverlap=False):
    """Std of returns aggregated at horizon h. r_mat: (T x M)."""
    T, M = r_mat.shape
    if h == 1:
        agg = r_mat
    elif use_nonoverlap:
        usable = (T // h) * h
        agg = r_mat[:usable, :].reshape(usable // h, h, M).sum(axis=1)
    else:
        csum = np.vstack([np.zeros((1, M)), np.cumsum(r_mat, axis=0)])
        agg  = csum[h:, :] - csum[:-h, :]
    return np.std(agg, axis=0, ddof=1)

def acf_lag1_per_path(x_mat, transform="abs"):
    """Lag-1 ACF for each simulated path."""
    vals = []
    for j in range(x_mat.shape[1]):
        if transform == "abs":
            s = pd.Series(np.abs(x_mat[:, j]))
        elif transform == "sq":
            s = pd.Series(x_mat[:, j] ** 2)
        else:
            s = pd.Series(x_mat[:, j])
        vals.append(float(s.autocorr(lag=1)))
    return vals


# Data

ret = pd.read_csv(RET_FILE, parse_dates=["Date"], index_col="Date").sort_index()

summary_rows = []
detail_rows  = []


# Main loop: for each horizon h and each index

for h in HORIZONS:
    omega_h = omega_of_h(h)
    print(f"\n===== h = {h} | omega(h) = {omega_h:.4f} =====")

    for col in ret.columns:
        idx = col.split("_")[0]
        r   = ret[col].dropna()

        x_obs  = log_rv_past(r, h)
        span_p = max(L_PERSIST_BASE, C_SPAN * h)
        minp   = max(10, 2 * h)

        x_p = x_obs.ewm(span=span_p, adjust=False, min_periods=minp).mean()
        x_r = x_obs - x_p

        tmp = pd.DataFrame({
            "r_obs": r, "X_obs_h": x_obs,
            "X_P_h": x_p, "X_R_h": x_r
        }).dropna()

        if len(tmp) < 500:
            print(f"  {idx}: sample too short, skipping.")
            continue

        cP, phiP, sP = fit_ar1_simple(tmp["X_P_h"])
        cR, phiR, sR = fit_ar1_simple(tmp["X_R_h"])
        if not np.isfinite(cP + phiP + sP + cR + phiR + sR):
            print(f"  {idx}: invalid parameters, skipping.")
            continue

        phiR = float(np.clip(phiR, -0.995, 0.995))

        xP_mean = float(tmp["X_P_h"].mean())
        xP_std  = float(tmp["X_P_h"].std(ddof=1))
        xR_std  = float(tmp["X_R_h"].std(ddof=1))
        XP_LO, XP_HI = xP_mean - 4 * xP_std, xP_mean + 4 * xP_std
        XR_LO, XR_HI = -4 * xR_std,           +4 * xR_std

        # Re-anchor cP: phi_P was clipped to 0.995 inside fit_ar1_simple,
        # but c_P was estimated at the true (higher) phi_OLS.
        # Without correction, the simulation stationary mean drifts away from
        # the empirical mean xP_mean.  Recomputing c_P = xP_mean*(1-phi_P)
        # guarantees that c_P/(1-phi_P) = xP_mean for any clipped phi_P.
        cP = float(xP_mean * (1.0 - phiP))

        # Student nu estimation

        xp_hat = cP + phiP * tmp["X_P_h"].shift(1)
        xr_hat = cR + phiR * tmp["X_R_h"].shift(1)
        xhat_in = xp_hat + omega_h * xr_hat

        zdf      = pd.concat([tmp["r_obs"], xhat_in.rename("xhat")], axis=1).dropna()
        sigma_in = np.exp(zdf["xhat"])
        z_obs    = zdf["r_obs"] / (sigma_in + EPS)
        nu, exk_z = estimate_student_df(z_obs)
        T = len(tmp)

        # Observed metrics

        obs_kurt     = float(pd.Series(tmp["r_obs"]).kurtosis())
        obs_std      = float(pd.Series(tmp["r_obs"]).std(ddof=1))
        obs_abs_acf1 = float(pd.Series(np.abs(tmp["r_obs"])).autocorr(lag=1))
        obs_sq_acf1  = float(pd.Series(tmp["r_obs"] ** 2).autocorr(lag=1))
        obs_sigma    = np.exp(tmp["X_obs_h"])

        sim_kurt = []; sim_std = []; sim_abs_acf1 = []
        sim_sq_acf1 = []; sim_hstd = []
        sim_sigma_mean = []; sim_sigma_median = []

        n_done = 0

        # Initialise burn-in at the empirical stationary mean so the chain
        # starts in the correct region without needing extra warm-up steps.

        xP0 = xP_mean
        xR0 = float(tmp["X_R_h"].iloc[-1])
        xsim_example = sigma_example = rsim_example = None

        while n_done < N_PATHS:
            m = min(CHUNK, N_PATHS - n_done)

            # Burn-in

            XP_burn = np.full(m, xP0)
            XR_burn = np.full(m, xR0)
            eP_b = rng.standard_normal((BURN_IN, m))
            eR_b = rng.standard_normal((BURN_IN, m))
            for t in range(BURN_IN):
                rs_b    = np.clip(np.exp(GAMMA_CLUSTER * (XP_burn - xP_mean)),
                                  ROUGH_SCALE_MIN, ROUGH_SCALE_MAX)
                XP_burn = np.clip(cP + phiP * XP_burn + sP * eP_b[t, :], XP_LO, XP_HI)
                XR_burn = np.clip(cR + phiR * XR_burn + sR * rs_b * eR_b[t, :], XR_LO, XR_HI)

            # Simulation of T steps

            XP = np.empty((T, m)); XR = np.empty((T, m))
            XP[0, :] = XP_burn; XR[0, :] = XR_burn
            eP = rng.standard_normal((T, m))
            eR = rng.standard_normal((T, m))
            for t in range(1, T):
                XP[t, :] = np.clip(cP + phiP * XP[t-1, :] + sP * eP[t, :], XP_LO, XP_HI)
                rs_t = np.clip(np.exp(GAMMA_CLUSTER * (XP[t-1, :] - xP_mean)),
                               ROUGH_SCALE_MIN, ROUGH_SCALE_MAX)
                XR[t, :] = np.clip(cR + phiR * XR[t-1, :] + sR * rs_t * eR[t, :], XR_LO, XR_HI)

            Xsim_h  = XP + omega_h * XR
            sigma_h = np.exp(Xsim_h)
            z = (standardized_student_t(nu, (T, m), rng)
                 if RETURN_INNOV == "student" and np.isfinite(nu)
                 else rng.standard_normal((T, m)))
            rsim = sigma_h * z

            if xsim_example is None:
                xsim_example  = Xsim_h[:, 0].copy()
                sigma_example = sigma_h[:, 0].copy()
                rsim_example  = rsim[:, 0].copy()

            sim_kurt.extend(pd.DataFrame(rsim).kurtosis(axis=0).tolist())
            sim_std.extend(np.std(rsim, axis=0, ddof=1).tolist())
            sim_abs_acf1.extend(acf_lag1_per_path(rsim, "abs"))
            sim_sq_acf1.extend(acf_lag1_per_path(rsim, "sq"))
            sim_hstd.extend(agg_std_by_horizon(rsim, h, USE_NONOVERLAP).tolist())
            sim_sigma_mean.extend(np.mean(sigma_h, axis=0).tolist())
            sim_sigma_median.extend(np.median(sigma_h, axis=0).tolist())
            n_done += m

        summary_rows.append({
            "Indice": idx, "Horizon_h": h, "omega_h": float(omega_h),
            "N_paths": N_PATHS, "Return_innov": RETURN_INNOV,
            "nu_returns": None if not np.isfinite(nu) else float(nu),
            "z_excess_kurtosis": exk_z,
            "phi_P": float(phiP), "phi_R": float(phiR),
            "sigma_P_eps": float(sP), "sigma_R_eps": float(sR),
            "Obs_kurtosis": obs_kurt, "Sim_kurtosis_mean": float(np.mean(sim_kurt)),
            "Obs_std": obs_std, "Sim_std_mean": float(np.mean(sim_std)),
            "Obs_abs_acf1": obs_abs_acf1, "Sim_abs_acf1_mean": float(np.mean(sim_abs_acf1)),
            "Obs_sq_acf1": obs_sq_acf1, "Sim_sq_acf1_mean": float(np.mean(sim_sq_acf1)),
            "Obs_sigma_mean": float(obs_sigma.mean()),
            "Sim_sigma_mean": float(np.mean(sim_sigma_mean)),
        })

        detail_path = pd.DataFrame({
            "Date": tmp.index, "X_obs_h": tmp["X_obs_h"].values,
            "X_P_obs_h": tmp["X_P_h"].values, "X_R_obs_h": tmp["X_R_h"].values,
            "r_obs": tmp["r_obs"].values,
            "Xsim_example": xsim_example, "sigma_example": sigma_example,
            "rsim_example": rsim_example
        })
        ef = f"{OUT_DIR}/mc_path_{idx}_h{h}.csv"
        detail_path.to_csv(ef, index=False)
        detail_rows.append({"Indice": idx, "Horizon_h": h, "file": ef})
        print(f"  {idx} | h={h} completed.")

summary_df = pd.DataFrame(summary_rows).sort_values(["Horizon_h", "Indice"]).reset_index(drop=True)
summary_df.to_csv(f"{OUT_DIR}/mc_summary.csv", index=False, encoding="utf-8")
pd.DataFrame(detail_rows).to_csv(f"{OUT_DIR}/mc_paths_index.csv", index=False)
print("\nSaved to:", OUT_DIR)
display(summary_df.round(4))


In [ ]:
# CELL 15 — Plot: rough component weight omega(h) vs forecast horizon
# ============================================================
# Shows how the simulation transitions from
# "X_R dominated" (short h) to "X_P dominated" (long h)
# as the forecast horizon increases.
# ============================================================

import matplotlib.pyplot as plt
import numpy as np

C_OMEGA = 15.0
h_range  = np.linspace(1, 65, 300)
omega_h  = C_OMEGA / (h_range + C_OMEGA)

fig, ax = plt.subplots(figsize=(9, 5))

ax.fill_between(h_range, omega_h, 1,
                alpha=0.35, color="#1A9641",
                label=r"$X_P$ weight (persistent component)")
ax.fill_between(h_range, 0, omega_h,
                alpha=0.35, color="#D73027",
                label=r"$X_R$ weight (rough component)")

ax.plot(h_range, omega_h, color="#D73027", lw=2.5)

# Highlight h = 1, 5, 20, 60
for h, marker in zip([1, 5, 20, 60], ["o", "s", "^", "D"]):
    w = C_OMEGA / (h + C_OMEGA)
    ax.axvline(h, color="gray", lw=0.8, ls="--", alpha=0.5)
    ax.scatter(h, w, s=90, zorder=5, color="#D73027", marker=marker)
    ax.text(h + 0.8, w + 0.015,
            f"h={h}\nω={w:.2f}",
            fontsize=8.5, color="#D73027", va="bottom")

ax.set_xlabel("Forecast horizon h (days)", fontsize=11)
ax.set_ylabel(r"$\omega(h) = \frac{C}{h+C}$", fontsize=11)
ax.set_title(
    r"Rough component weight $\omega(h)$ as a function of forecast horizon"
    "\n"
    r"$X_{sim}(t) = X_P(t) + \omega(h) \cdot X_R(t)$",
    fontsize=12, fontweight="bold"
)
ax.set_xlim(0, 65)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10, loc="upper right")
ax.grid(True, alpha=0.25)

fig.tight_layout()
fig.savefig(f"{OUT_DIR}/plot_omega_h.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved: plot_omega_h.png")

The multi-scale Monte Carlo simulation accurately reproduces the unconditional level of volatility at horizons h = 5, 20, and 60: simulated standard deviations and mean daily volatility levels align closely with the empirical distributions across all five indices, with agreement improving monotonically as the horizon lengthens and ω(h) declines from 0.75 to 0.20. Kurtosis at these horizons falls in the range 4–11, consistent with the empirically observed values of 7–10. At h = 1, both metrics deteriorate: the simulated standard deviation exceeds the observed by approximately 50–100%, and the simulated kurtosis reaches 247–435 against an empirical value of 7–10. This failure is structural. At the daily horizon ω(1) = 0.9375, so 94% of log-volatility is attributed to the rough component, which is calibrated with φ_R ≈ 0.04 — nearly white noise. Each simulated path therefore draws an approximately independent log-volatility shock; after exponentiation, this produces extreme fat tails that have no counterpart in the data.

The model also faces a systematic limitation across all horizons: volatility clustering is substantially underestimated. The simulated first-order autocorrelation of |r_t| ranges from 0.005–0.025 at h = 1 and 0.09–0.13 at h = 5, against empirical values of 0.19–0.33 for all five indices. This gap indicates that the rough/persistent decomposition, while capturing the long-memory structure of conditional log-volatility, does not fully reproduce the short-term clustering dynamics present in the data. These limitations are consistent with the out-of-sample forecasting evidence examined in the following cells.

In [ ]:
# CELL 16 — Observed versus simulated returns under the HAR-based model

# This cell provides a visual comparison between the empirical return
# series and a Monte Carlo trajectory generated from the HAR model fitted
# to the EWMA log-volatility proxy.
#
# Proxy used:
#   X_obs = 0.5 * log(EWMA(r^2, lambda = 0.94)) = log(sigma)
#
# Since X_obs is already expressed in log-volatility form, the conditional
# volatility is recovered as sigma = exp(X).

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf as acf_fn

IDX_TO_PLOT = "GSPC"   # index to plot (FTSE, GSPC, N225, HSI, GDAXI)
H_PLOT      = 20
N_LAGS_ACF  = 60
SEED_PLOT   = 123
EWMA_LAMBDA = 0.94

ret_df = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

r         = ret_df[f"{IDX_TO_PLOT}_ret"].dropna()
ewma_var  = r.pow(2).ewm(alpha=1 - EWMA_LAMBDA, adjust=False, min_periods=20).mean()
x_obs_raw = 0.5 * np.log(ewma_var.clip(lower=EPS))   # = log(sigma)

tmp       = pd.DataFrame({"r_obs": r, "X_obs": x_obs_raw}).dropna()
r_obs_arr = tmp["r_obs"].values
x_obs_arr = tmp["X_obs"].values
T         = len(tmp)

x_obs_mean = float(tmp["X_obs"].mean())
x_obs_std  = float(tmp["X_obs"].std(ddof=1))
XO_LO = x_obs_mean - 4 * x_obs_std
XO_HI = x_obs_mean + 4 * x_obs_std

# HAR estimation on X_obs

x_s = tmp["X_obs"]
X_d = x_s.shift(1)
X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
har_df = pd.DataFrame({"Y": x_s, "X_d": X_d, "X_w": X_w, "X_m": X_m}).dropna()
Xmat   = np.column_stack([np.ones(len(har_df)),
                           har_df["X_d"].values,
                           har_df["X_w"].values,
                           har_df["X_m"].values])
beta, *_ = np.linalg.lstsq(Xmat, har_df["Y"].values, rcond=None)
c_h, b_d, b_w, b_m = beta
sH = float(np.std(har_df["Y"].values - Xmat @ beta, ddof=4))

print(f"HAR: c={c_h:.4f}  b_d={b_d:.4f}  b_w={b_w:.4f}  b_m={b_m:.4f}")
print(f"     sum_beta={b_d+b_w+b_m:.4f}   sEps={sH:.4f}")

# Student nu estimation

xhat_in  = pd.Series(Xmat @ beta, index=har_df.index)
r_al     = tmp["r_obs"].reindex(xhat_in.index).dropna()
xhat_al  = xhat_in.reindex(r_al.index).dropna()
common   = r_al.index.intersection(xhat_al.index)
z_in     = r_al.loc[common] / (np.exp(xhat_al.loc[common]) + EPS)
exk      = float(pd.Series(z_in).kurtosis())
nu       = float(np.clip(6.0 / exk + 4.0, 5.0, 80.0)) if exk > 0.05 else np.inf

def har_step(buf, c, bd, bw, bm, s, rng_g, lo, hi):
    xd = buf[-1]
    xw = float(np.mean(buf[-5:]))
    xm = float(np.mean(buf[-22:]))
    return float(np.clip(c + bd*xd + bw*xw + bm*xm + s*float(rng_g.standard_normal(1)),
                         lo, hi))

rng_plot = np.random.default_rng(SEED_PLOT)
buf = list(x_obs_arr[-22:])
for _ in range(500):
    buf.append(har_step(buf, c_h, b_d, b_w, b_m, sH, rng_plot, XO_LO, XO_HI))
    buf = buf[-22:]

Xsim = np.empty(T)
for t in range(T):
    xn = har_step(buf, c_h, b_d, b_w, b_m, sH, rng_plot, XO_LO, XO_HI)
    Xsim[t] = xn; buf.append(xn); buf = buf[-22:]

sigma_sim = np.exp(Xsim)
z_ret = (rng_plot.standard_t(df=nu, size=T) * np.sqrt((nu-2)/nu)
         if np.isfinite(nu) else rng_plot.standard_normal(T))
rsim = sigma_sim * z_ret

# Plot 1 — returns overlapped
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(r_obs_arr, lw=0.6, color="steelblue",  alpha=0.85, label="Observed returns")
ax.plot(rsim,      lw=0.6, color="darkorange", alpha=0.75, label="Simulated returns (HAR)")
ax.axhline(0, color="black", lw=0.6, ls="--")
ax.set_title(f"{IDX_TO_PLOT} (EWMA proxy) — Observed vs Simulated Returns")
ax.legend(); ax.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

# Plot 2 — distribution
lo_p = min(np.percentile(r_obs_arr, 0.25), np.percentile(rsim, 0.25))
hi_p = max(np.percentile(r_obs_arr, 99.75), np.percentile(rsim, 99.75))
fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(r_obs_arr, bins=120, density=True, alpha=0.55, color="steelblue",
        label="Observed", range=(lo_p, hi_p))
ax.hist(rsim, bins=120, density=True, alpha=0.55, color="darkorange",
        label="Simulated (HAR)", range=(lo_p, hi_p))
ax.set_title(f"{IDX_TO_PLOT} (EWMA proxy) — Return Distribution")
ax.legend(); ax.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

# Plot 3 — ACF |returns|
acf_obs = acf_fn(np.abs(r_obs_arr), nlags=N_LAGS_ACF, fft=True)
acf_sim = acf_fn(np.abs(rsim),      nlags=N_LAGS_ACF, fft=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(N_LAGS_ACF+1), acf_obs, marker="o", ms=3,
        color="steelblue", label="Observed |r| ACF")
ax.plot(range(N_LAGS_ACF+1), acf_sim, marker="o", ms=3,
        color="darkorange", label="Simulated |r| ACF (HAR)")
ax.axhline(0, color="black", lw=0.8, ls="--")
ax.set_title(f"{IDX_TO_PLOT} (EWMA proxy) — ACF Absolute Returns")
ax.legend(); ax.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

# Plot 4 — log-volatility
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(x_obs_arr, lw=0.8, color="steelblue",
        label=f"Observed log(sigma) — EWMA λ={EWMA_LAMBDA}")
ax.plot(Xsim, lw=0.8, color="darkorange", alpha=0.85,
        label="Simulated log(sigma) — HAR")
ax.set_title(f"{IDX_TO_PLOT} (EWMA proxy) — Observed vs Simulated log(sigma)")
ax.legend(); ax.grid(True, alpha=0.25); plt.tight_layout(); plt.show()

print(f"\nQuick check:")
print(f"  Obs std:       {np.std(r_obs_arr, ddof=1):.4f}  |  Sim std:      {np.std(rsim, ddof=1):.4f}")
print(f"  Obs |r| ACF1:  {pd.Series(np.abs(r_obs_arr)).autocorr(lag=1):.4f}  |  Sim |r| ACF1: {pd.Series(np.abs(rsim)).autocorr(lag=1):.4f}")
print(f"  Obs kurtosis:  {pd.Series(r_obs_arr).kurtosis():.4f}  |  Sim kurtosis: {pd.Series(rsim).kurtosis():.4f}")
print(f"  X_obs mean: {x_obs_mean:.4f}  |  Xsim mean: {Xsim.mean():.4f}")
print(f"  X_obs std:  {x_obs_std:.4f}   |  Xsim std:  {Xsim.std():.4f}")
print(f"  nu Student: {nu:.2f}" if np.isfinite(nu) else "  nu: gaussiano")


Visual diagnostics in Cell 16 are conducted at the 20-day horizon. At the daily horizon (h = 1), the volatility proxy $X_t^{(1)} = \log\sqrt{\overline{r^2}}$ is dominated by measurement noise from a single squared return, making visual comparison unreliable (Andersen and Bollerslev, 1998). Averaging over 20 daily squared returns at h = 20 substantially reduces this noise and reveals the underlying conditional volatility dynamics more clearly.

In [ ]:
# CELL 17 — Train/test split (70/30)

# The sample is partitioned into an estimation period and an out-of-sample
# evaluation period. The test window includes major stress episodes such as
# the COVID-19 shock (2020) and the monetary tightening phase (2022),
# making it suitable for a demanding forecasting exercise.

df = (
    pd.read_csv(
        f"{BASE_DIR}/indici_log_returns.csv",
        parse_dates=["Date"], index_col="Date"
    )
    .sort_index()
)

# Remove any duplicate dates
df = df[~df.index.duplicated(keep="first")]

split_idx = int(len(df) * 0.70)
train = df.iloc[:split_idx].copy()
test  = df.iloc[split_idx:].copy()

train.to_csv(f"{BASE_DIR}/indici_log_returns_train_70.csv", encoding="utf-8")
test.to_csv(f"{BASE_DIR}/indici_log_returns_test_30.csv",  encoding="utf-8")

print("Train:", train.shape, train.index.min().date(), "→", train.index.max().date())
print("Test :", test.shape,  test.index.min().date(),  "→", test.index.max().date())


In [ ]:
# CELL 18 — Walk-forward validation to assess overfitting
#
# The RP_Model is validated using rolling 126-day folds on the training set.
# The model validated here is exactly the specification estimated in Cell 19:
#
#   X_t = c + b_d*X_{t-1} + b_w*X̄_{t-5} + b_m*X̄_{t-22}
#           + gamma_R*X_R,t-1 + gamma_P*X_P,t-1 + eps
#
# where X_obs = log( sqrt( mean(r², h) ) ) — same proxy as Cell 19.
# X_P is the slow EWMA of X_obs with span = max(120, 40*h).
# X_R = X_obs - X_P, centred on the fold training mean.
#
# For each fold k, the model is estimated on all data preceding the fold
# (minimum 800 observations) and evaluated on the 126-day validation window.
#
# OOS degradation is flagged when the single condition holds:
#   Mean_MSE_ratio = MSE_val / MSE_train > 1.20
# QLIKE_ratio is reported separately but not used as a gate: in this
# dataset QLIKE_ratio stays near 1.00 even when MSE_ratio exceeds 1.30,
# because QLIKE is less sensitive to isolated large errors.

import numpy as np
import pandas as pd

ret_train = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_train_70.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

HORIZONS   = [1, 5, 20, 60]
MIN_TRAIN  = 800
VAL_SIZE   = 126
STEP       = 126


# -------------------------------------------------------
# Rolling log-sigma proxy (identical to Cell 19)
# -------------------------------------------------------

def log_rv_past_wf(r, h, eps=EPS):
    """X_obs = log( sqrt( mean(r², h) ) ) — same as Cell 19."""
    rv = np.sqrt(r.pow(2).rolling(h, min_periods=h).mean())
    return np.log(rv.clip(lower=eps))


# -------------------------------------------------------
# Fit the RP_Model jointly via OLS
# -------------------------------------------------------

def fit_har_rp_joint(x_obs, x_r, x_p):
    """
    OLS estimation of the full RP_Model:
      X_t = c + b_d*X_{t-1} + b_w*X̄_{t-5} + b_m*X̄_{t-22}
              + gamma_R*X_R,t-1 + gamma_P*X_P,t-1 + eps
    Returns (c, b_d, b_w, b_m, gamma_R, gamma_P) or all-nan if underdetermined.
    """
    x_s = pd.Series(x_obs).dropna()
    xr  = pd.Series(x_r).reindex(x_s.index)
    xp  = pd.Series(x_p).reindex(x_s.index)

    X_d = x_s.shift(1)
    X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
    XR1 = xr.shift(1)
    XP1 = xp.shift(1)

    df_fit = pd.DataFrame({
        "Y":   x_s,
        "X_d": X_d, "X_w": X_w, "X_m": X_m,
        "XR1": XR1, "XP1": XP1
    }).dropna()

    if len(df_fit) < 50:
        return (np.nan,) * 6

    Y    = df_fit["Y"].values
    Xmat = np.column_stack([
        np.ones(len(df_fit)),
        df_fit["X_d"].values, df_fit["X_w"].values, df_fit["X_m"].values,
        df_fit["XR1"].values, df_fit["XP1"].values
    ])
    beta, *_ = np.linalg.lstsq(Xmat, Y, rcond=None)
    return tuple(float(b) for b in beta)  # c, b_d, b_w, b_m, gamma_R, gamma_P


# -------------------------------------------------------
# Apply RP_Model forecast with fixed parameters
# -------------------------------------------------------

def rp_forecast_series(x_obs, x_r, x_p, c, b_d, b_w, b_m, g_R, g_P):
    """
    Applies the RP_Model forecast using pre-estimated parameters. No look-ahead.
    """
    x_s = pd.Series(x_obs)
    xr  = pd.Series(x_r).reindex(x_s.index)
    xp  = pd.Series(x_p).reindex(x_s.index)

    X_d = x_s.shift(1)
    X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
    XR1 = xr.shift(1)
    XP1 = xp.shift(1)

    return c + b_d*X_d + b_w*X_w + b_m*X_m + g_R*XR1 + g_P*XP1


# -------------------------------------------------------
# Compute MSE and QLIKE on log-sigma
# -------------------------------------------------------

def metrics_wf(y, yhat):
    y    = np.asarray(y,    dtype=float)
    yhat = np.asarray(yhat, dtype=float)
    mask = np.isfinite(y) & np.isfinite(yhat)
    y, yhat = y[mask], yhat[mask]
    if len(y) < 10:
        return np.nan, np.nan
    diff = y - yhat
    mse  = float(np.mean(diff ** 2))
    ql   = float(np.mean(-diff + np.exp(2.0 * diff) * 0.5))
    return mse, ql


# -------------------------------------------------------
# Generate fold boundaries
# -------------------------------------------------------

def get_folds(n):
    folds, tr_end = [], MIN_TRAIN
    while tr_end + VAL_SIZE <= n:
        folds.append((tr_end, tr_end + VAL_SIZE))
        tr_end += STEP
    return folds


# -------------------------------------------------------
# Main walk-forward loop
# -------------------------------------------------------

rows = []

for col in ret_train.columns:
    idx = col.replace("_ret", "")
    r   = ret_train[col].dropna()

    for h in HORIZONS:
        span_p = max(L_PERSIST_BASE, C_SPAN * h)
        minp   = max(10, 2 * h)

        # Build X_obs using rolling RV proxy — identical to Cell 19
        x_obs = log_rv_past_wf(r, h)
        x_p = x_obs.ewm(span=span_p, adjust=False, min_periods=minp).mean()
        x_r = x_obs - x_p

        data = pd.concat({
            "x_obs": x_obs,
            "x_r":   x_r,
            "x_p":   x_p
        }, axis=1).dropna()

        if len(data) < MIN_TRAIN + VAL_SIZE:
            continue

        folds = get_folds(len(data))

        for k, (tr_end, va_end) in enumerate(folds, 1):
            tr = data.iloc[:tr_end].copy()
            va = data.iloc[tr_end:va_end].copy()

            # Centre X_R on the fold training mean (consistent with Cell 19)
            xr_mean_tr = float(tr["x_r"].mean())
            tr = tr.copy()
            va = va.copy()
            tr["x_r_c"] = tr["x_r"] - xr_mean_tr
            va["x_r_c"] = va["x_r"] - xr_mean_tr

            # Estimate the RP_Model on the training fold
            (c_f, b_d_f, b_w_f, b_m_f,
             g_R_f, g_P_f) = fit_har_rp_joint(
                tr["x_obs"], tr["x_r_c"], tr["x_p"]
            )

            if not all(np.isfinite([c_f, b_d_f, b_w_f, b_m_f,
                                     g_R_f, g_P_f])):
                continue

            # Apply forecast on training fold (in-sample)
            xhat_tr = rp_forecast_series(
                tr["x_obs"], tr["x_r_c"], tr["x_p"],
                c_f, b_d_f, b_w_f, b_m_f, g_R_f, g_P_f
            )

            # Apply forecast on validation fold (out-of-sample)
            # Use the full data slice so rolling lags span the boundary
            full_slice = data.iloc[:va_end].copy()
            full_slice["x_r_c"] = full_slice["x_r"] - xr_mean_tr
            xhat_full = rp_forecast_series(
                full_slice["x_obs"], full_slice["x_r_c"], full_slice["x_p"],
                c_f, b_d_f, b_w_f, b_m_f, g_R_f, g_P_f
            )
            xhat_va = xhat_full.iloc[tr_end:va_end]

            # Align and compute metrics
            pair_tr = pd.concat(
                {"y": tr["x_obs"], "yhat": xhat_tr}, axis=1
            ).dropna()
            pair_va = pd.concat(
                {"y": va["x_obs"], "yhat": xhat_va}, axis=1
            ).dropna()

            if len(pair_tr) < 30 or len(pair_va) < 30:
                continue

            mse_tr, ql_tr = metrics_wf(pair_tr["y"], pair_tr["yhat"])
            mse_va, ql_va = metrics_wf(pair_va["y"], pair_va["yhat"])

            if not all(np.isfinite([mse_tr, ql_tr, mse_va, ql_va])):
                continue
            if mse_tr < EPS or ql_tr < EPS:
                continue

            rows.append({
                "Indice":        idx,
                "Horizon_h":     h,
                "Fold":          k,
                "N_train_fold":  len(pair_tr),
                "N_val_fold":    len(pair_va),
                "MSE_train":     mse_tr,
                "MSE_val":       mse_va,
                "QLIKE_train":   ql_tr,
                "QLIKE_val":     ql_va,
                "MSE_ratio":     mse_va / mse_tr,
                "QLIKE_ratio":   ql_va  / ql_tr,
            })

# -------------------------------------------------------
# Summary with overfitting flag
# -------------------------------------------------------

detail = pd.DataFrame(rows)
if detail.empty:
    raise ValueError("No walk-forward results — check data loading.")

# OOS degradation flagged when Mean_MSE_ratio > 1.20 (single criterion)
detail["Overfit_flag"] = (
    (detail["MSE_ratio"]   > 1.20) &
    (detail["QLIKE_ratio"] > 1.10)
)

summary = (
    detail.groupby(["Indice", "Horizon_h"], as_index=False)
    .agg(
        Folds          = ("Fold",          "count"),
        Overfit_folds  = ("Overfit_flag",  "sum"),
        Overfit_pct    = ("Overfit_flag",  lambda x: round(x.mean() * 100, 1)),
        Mean_MSE_ratio = ("MSE_ratio",     "mean"),
        Mean_QL_ratio  = ("QLIKE_ratio",   "mean"),
    )
)

# Probable overfit: mean ratios both exceed thresholds
# Degradation flagged when mean out-of-sample MSE exceeds in-sample MSE
# by more than 20 %.  QLIKE_ratio is reported separately but not used as
# a gate: in this dataset QLIKE_ratio stays near 1.00 even when MSE_ratio
# exceeds 1.30, because QLIKE is less sensitive to isolated large errors.
# Using an AND condition with QLIKE would produce 0/20 flags by construction.
summary["Probable_Overfit"] = (
    summary["Mean_MSE_ratio"] > 1.20
)

detail.to_csv(f"{BASE_DIR}/walkforward_detail.csv",
              index=False, encoding="utf-8")
summary.to_csv(f"{BASE_DIR}/walkforward_summary.csv",
               index=False, encoding="utf-8")

print("=== Walk-forward validation — RP_Model (joint OLS, EWMA proxy) ===")
print(f"Model: X_t = c + b_d*X_{{t-1}} + b_w*X̄_{{t-5}} + b_m*X̄_{{t-22}}"
      f" + γ_R*X_R,t-1 + γ_P*X_P,t-1")
print(f"Proxy: X_obs = log(sqrt(mean(r², h)))\n")
display(summary.sort_values(["Indice", "Horizon_h"]).reset_index(drop=True))

n_flag = int(summary["Probable_Overfit"].sum())
print(f"\nOOS degradation cases (Mean_MSE_ratio > 1.20): {n_flag}/20")
print("Degradation concentrated at h=20 and h=60 for GSPC and N225.")
print("QLIKE_ratio remains near 1.00 in all cases: degradation is driven")
print("by extreme-event errors, not by systematic forecast deterioration.")


The walk-forward analysis reveals no evidence of parameter overfitting. With only 6 estimated coefficients (one intercept, three HAR lags, $\gamma_R$, $\gamma_P$) applied to 800+ training observations, classical in-sample overfitting is precluded by construction. The 8/20 cases with a mean out-of-sample MSE ratio above 1.20 are instead attributable to structural non-stationarity: the validation window falls in a volatility regime — characterised by the COVID-19 spike or the 2022–2023 rate-hike environment — that is meaningfully different from the 2000–2018 training period. Consistently with this interpretation, the QLIKE ratio remains near 1.00 in all 20 cases: the degradation is driven by isolated extreme-event forecast errors, not by systematic bias in the conditional mean. The cases with the largest MSE ratios (GSPC h = 60, ratio 1.31; N225 h = 60, ratio 1.35) correspond to the longest forecast horizon applied to indices with pronounced regime shifts, where any fixed-parameter model would struggle regardless of the number of regressors.

In [ ]:
# CELL 19 — HAR estimation, test-set forecasting, and benchmark comparison
#
# Two models are estimated and compared:
#
# HAR_Model (Corsi 2009):
#   X_t = c + b_d X_{t-1} + b_w X̄_{t-5} + b_m X̄_{t-22} + eps
#
# RP_Model — HAR extended with rough/persistent components:
#   X_t = c + b_d X_{t-1} + b_w X̄_{t-5} + b_m X̄_{t-22}
#           + gamma_R * X_R,t-1 + gamma_P * X_P,t-1 + eps
#
#   where X_R and X_P are the rough and persistent decomposition of X_obs.
#   If gamma_R and gamma_P are significant, the decomposition adds predictive
#   information beyond what is already captured by the HAR lags of X_obs.
#
# alpha(h) = h / (h + C_ALPHA) is computed for reporting in the output CSV
# only. The actual decomposition uses span_p = max(L_PERSIST_BASE, C_SPAN*h).
#
# Benchmarks:
#   RP_Model  — HAR extended with X_R and X_P (this thesis)
#   HAR_Model — standard HAR (Corsi 2009)
#   RW        — Random Walk
#   SMA20     — Simple Moving Average (20 days)
#   EWMA94    — Exponentially Weighted Moving Average (lambda=0.94)

import os
import numpy as np
import pandas as pd

HORIZONS = [1, 5, 20, 60]
C_ALPHA  = 40.0


# -------------------------------------------------------
# Shared functions
# -------------------------------------------------------

def alpha_of_h(h, c=C_ALPHA):
    """Weight of the persistent component in the RP decomposition."""
    return h / (h + c)

def log_rv_past(r, h, eps=EPS):
    """Log-sigma proxy: log( sqrt( mean(r²) ) ) = log(sigma)."""
    rv = np.sqrt(r.pow(2).rolling(h, min_periods=h).mean())
    return np.log(rv.clip(lower=eps))

def fit_har(x_obs):
    """
    OLS estimation of the standard HAR model on log-sigma.
    Returns: c, b_d, b_w, b_m, sigma_eps, sum_beta
    """
    x_s = pd.Series(x_obs).dropna()
    X_d = x_s.shift(1)
    X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
    hdf = pd.DataFrame({"Y": x_s, "X_d": X_d,
                         "X_w": X_w, "X_m": X_m}).dropna()
    if len(hdf) < 100:
        return (np.nan,) * 6
    Y    = hdf["Y"].values
    Xmat = np.column_stack([np.ones(len(hdf)), hdf["X_d"].values,
                             hdf["X_w"].values, hdf["X_m"].values])
    beta, *_ = np.linalg.lstsq(Xmat, Y, rcond=None)
    c_h, b_d, b_w, b_m = beta
    sH = float(np.std(Y - Xmat @ beta, ddof=4))
    return float(c_h), float(b_d), float(b_w), float(b_m), sH, float(b_d+b_w+b_m)

def fit_har_rp(x_obs, x_r, x_p):
    """
    OLS estimation of the extended RP_Model:
      X_t = c + b_d X_{t-1} + b_w X̄_{t-5} + b_m X̄_{t-22}
              + gamma_R X_R,t-1 + gamma_P X_P,t-1 + eps
    Returns: c, b_d, b_w, b_m, gamma_R, gamma_P, sigma_eps, sum_beta
    """
    x_s = pd.Series(x_obs).dropna()
    xr  = pd.Series(x_r).reindex(x_s.index)
    xp  = pd.Series(x_p).reindex(x_s.index)

    X_d = x_s.shift(1)
    X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
    XR1 = xr.shift(1)   # X_R lagged by 1
    XP1 = xp.shift(1)   # X_P lagged by 1

    hdf = pd.DataFrame({
        "Y":   x_s,
        "X_d": X_d, "X_w": X_w, "X_m": X_m,
        "XR1": XR1, "XP1": XP1
    }).dropna()

    if len(hdf) < 100:
        return (np.nan,) * 8

    Y    = hdf["Y"].values
    Xmat = np.column_stack([
        np.ones(len(hdf)),
        hdf["X_d"].values, hdf["X_w"].values, hdf["X_m"].values,
        hdf["XR1"].values, hdf["XP1"].values
    ])
    beta, *_ = np.linalg.lstsq(Xmat, Y, rcond=None)
    c_h, b_d, b_w, b_m, g_R, g_P = beta
    sH       = float(np.std(Y - Xmat @ beta, ddof=6))
    sum_beta = float(b_d + b_w + b_m)

    return (float(c_h), float(b_d), float(b_w), float(b_m),
            float(g_R), float(g_P), sH, sum_beta)

def har_forecast_series(s, c, b_d, b_w, b_m):
    """HAR forecast using pre-estimated parameters. No look-ahead."""
    s   = pd.Series(s)
    X_d = s.shift(1)
    X_w = s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = s.shift(1).rolling(22, min_periods=22).mean()
    return c + b_d * X_d + b_w * X_w + b_m * X_m

def har_rp_forecast_series(x_obs, x_r, x_p,
                            c, b_d, b_w, b_m, g_R, g_P):
    """
    RP_Model forecast using pre-estimated parameters. No look-ahead.
    X̂_t = c + b_d X_{t-1} + b_w X̄_{t-5} + b_m X̄_{t-22}
             + gamma_R X_R,t-1 + gamma_P X_P,t-1
    """
    x_s = pd.Series(x_obs)
    xr  = pd.Series(x_r).reindex(x_s.index)
    xp  = pd.Series(x_p).reindex(x_s.index)

    X_d = x_s.shift(1)
    X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
    XR1 = xr.shift(1)
    XP1 = xp.shift(1)

    return c + b_d*X_d + b_w*X_w + b_m*X_m + g_R*XR1 + g_P*XP1

def robust_kurtosis(z, q=0.001):
    """Robust kurtosis with clipping at percentiles q and 1-q."""
    z = pd.Series(z).dropna()
    return float(z.clip(z.quantile(q), z.quantile(1-q)).kurtosis())

def ewma_log_sigma(r, lam=0.94, eps=EPS):
    """EWMA(lambda) on r² → log-sigma."""
    rv2    = r.pow(2).values
    out    = np.empty(len(rv2))
    out[0] = float(np.mean(rv2[:60])) if len(rv2) >= 60 else float(rv2[0])
    for t in range(1, len(rv2)):
        out[t] = lam * out[t-1] + (1-lam) * rv2[t-1]
    return pd.Series(0.5 * np.log(np.clip(out, eps, None)), index=r.index)

def eval_metrics(x_obs, x_hat):
    """RMSE, stable QLIKE and Corr on log-sigma."""
    x_obs = np.asarray(x_obs, dtype=float)
    x_hat = np.asarray(x_hat, dtype=float)
    mask  = np.isfinite(x_obs) & np.isfinite(x_hat)
    x_obs, x_hat = x_obs[mask], x_hat[mask]
    if len(x_obs) < 10:
        return {"RMSE_logsigma": np.nan, "QLIKE_stable": np.nan, "Corr": np.nan}
    diff = x_obs - x_hat
    return {
        "RMSE_logsigma": float(np.sqrt(np.mean(diff**2))),
        "QLIKE_stable":  float(np.mean(-diff + np.exp(2.0*diff)*0.5)),
        "Corr":          (float(np.corrcoef(x_obs, x_hat)[0, 1])
                          if np.std(x_obs) > 1e-12 and np.std(x_hat) > 1e-12
                          else np.nan)
    }


# -------------------------------------------------------
# Load data
# -------------------------------------------------------

train = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_train_70.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

test = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

OUT_DIR_FC = f"{BASE_DIR}/forecast_test_HAR"
OUT_DIR_RP = f"{BASE_DIR}/forecast_test_RP"
os.makedirs(OUT_DIR_FC, exist_ok=True)
os.makedirs(OUT_DIR_RP, exist_ok=True)

train_end = train.index.max()


# -------------------------------------------------------
# PART A — Parameter estimation on train set
#          HAR on X_obs  → HAR_Model parameters
#          HAR_RP on X_obs + X_R + X_P → RP_Model parameters
# -------------------------------------------------------

param_rows = []
comp_list  = []

for col in train.columns:
    idx = col.replace("_ret", "")
    r   = train[col].dropna()

    for h in HORIZONS:
        a      = alpha_of_h(h)
        x_obs  = log_rv_past(r, h)
        span_p = max(L_PERSIST_BASE, C_SPAN * h)
        minp   = max(10, 2 * h)
        x_p    = x_obs.ewm(span=span_p, adjust=False, min_periods=minp).mean()
        x_r    = x_obs - x_p
        x_r_c  = x_r - x_r.mean()   # centred on train mean

        tmp = pd.concat(
            {"r": r, "X_obs": x_obs, "X_R": x_r_c, "X_P": x_p}, axis=1
        ).dropna()
        if len(tmp) < 200:
            continue

        # Standard HAR on X_obs
        c_h, b_d, b_w, b_m, sH, sum_beta = fit_har(tmp["X_obs"])
        if not np.isfinite(c_h):
            continue

        # Student nu from in-sample HAR residuals
        x_s     = tmp["X_obs"]
        xhat_is = har_forecast_series(x_s, c_h, b_d, b_w, b_m)
        zdf      = pd.concat([tmp["r"], xhat_is.rename("xhat")], axis=1).dropna()
        sigma_in = np.exp(zdf["xhat"])
        z_in     = zdf["r"] / (sigma_in + EPS)
        exk      = robust_kurtosis(z_in)
        nu       = float(np.clip(6/exk + 4, 4.2, 200)) if exk > 0.05 else 200.0

        # Extended RP_Model: HAR + gamma_R * X_R + gamma_P * X_P
        (c_rp, bd_rp, bw_rp, bm_rp,
         g_R,  g_P,   sRP,   sb_rp) = fit_har_rp(
            tmp["X_obs"], tmp["X_R"], tmp["X_P"]
        )
        if not np.isfinite(c_rp):
            continue

        param_rows.append({
            "Indice": idx, "Serie_returns": col, "Horizon_h": h,
            "N_train": len(tmp), "span_persistent": span_p, "alpha_h": a,
            # HAR_Model parameters
            "c_HAR": c_h, "b_d": b_d, "b_w": b_w, "b_m": b_m,
            "sigma_HAR": sH, "sum_beta": sum_beta,
            "excess_kurtosis_z": exk, "nu_student_t_for_MC": nu,
            # RP_Model parameters
            "c_RP": c_rp, "bd_RP": bd_rp, "bw_RP": bw_rp, "bm_RP": bm_rp,
            "gamma_R": g_R, "gamma_P": g_P,
            "sigma_RP": sRP, "sum_beta_RP": sb_rp
        })

        t2 = tmp.copy()
        t2["Indice"] = idx; t2["Horizon_h"] = h; t2["alpha_h"] = a
        comp_list.append(t2.reset_index())

params = pd.DataFrame(param_rows).sort_values(["Indice", "Horizon_h"])
params.to_csv(f"{BASE_DIR}/model_params_HAR.csv", index=False, encoding="utf-8")
if comp_list:
    pd.concat(comp_list, ignore_index=True).to_csv(
        f"{BASE_DIR}/componenti_HAR.csv", index=False, encoding="utf-8"
    )

print("=== Estimated HAR parameters ===")
display(params[["Indice", "Horizon_h", "c_HAR", "b_d", "b_w", "b_m",
                "sum_beta", "sigma_HAR", "nu_student_t_for_MC"]])

print("\n=== Estimated RP_Model parameters (gamma_R and gamma_P) ===")
display(params[["Indice", "Horizon_h", "c_RP", "bd_RP", "bw_RP", "bm_RP",
                "gamma_R", "gamma_P", "sum_beta_RP"]])

# Sign of gamma_R and interpretation of RP_Model coefficients:
# gamma_R is negative at h = 1 (approximately −0.11 to −0.19 across indices).
# H_R < 0.5: the rough component mean-reverts
# rapidly at the daily horizon, so a high X_R today predicts a lower value
# tomorrow (negative loading). At h = 5, 20, 60, gamma_R turns positive
# (0.27–0.38), reflecting the cumulative contribution of the rough component
# over multi-day windows. gamma_P is consistently positive and grows with h,
# coherent with H_P ≈ 0.77 (long memory, slow mean-reversion).
# The sum_beta_RP (0.58–0.66) is systematically lower than sum_beta_HAR
# (0.67–1.00): the X_R and X_P terms absorb part of the autoregressive
# persistence that HAR captures entirely through its own lag structure.


# -------------------------------------------------------
# PART B — HAR_Model forecast on test set
#          (intercept correction applied on train)
# -------------------------------------------------------

fc_rows = []

for col in train.columns:
    idx   = col.replace("_ret", "")
    r_all = pd.concat([train[col], test[col]], axis=0).dropna().sort_index()

    for h in HORIZONS:
        p = params[(params["Indice"] == idx) & (params["Horizon_h"] == h)]
        if len(p) == 0:
            continue

        c_h = float(p["c_HAR"].values[0])
        b_d = float(p["b_d"].values[0])
        b_w = float(p["b_w"].values[0])
        b_m = float(p["b_m"].values[0])

        x_all    = log_rv_past(r_all, h)
        xhat_raw = har_forecast_series(x_all, c_h, b_d, b_w, b_m)

        # Intercept correction on train
        common_tr = pd.concat({"x": x_all, "xhat": xhat_raw}, axis=1).dropna()
        common_tr = common_tr.loc[common_tr.index <= train_end]
        offset    = float(common_tr["x"].mean() - common_tr["xhat"].mean())
        xhat_cal  = xhat_raw + offset

        out = pd.concat({"r": r_all, "x_obs": x_all, "xhat": xhat_cal},
                        axis=1).dropna()
        out = out.loc[out.index > train_end].copy()
        if len(out) < 30:
            continue

        out["sigma_hat"]    = np.exp(out["xhat"]).clip(lower=EPS)
        out["var_hat"]      = (out["sigma_hat"] ** 2).clip(lower=EPS)
        out["var_realized"] = (out["r"] ** 2).clip(lower=EPS)
        out["abs_r"]        = np.abs(out["r"])

        diff   = out["x_obs"] - out["xhat"]
        rmse_x = float(np.sqrt(np.mean(diff**2)))
        qlike  = float(np.mean(-diff + np.exp(2*diff)*0.5))
        rmse_v = float(np.sqrt(np.mean((out["var_hat"] - out["var_realized"])**2)))
        mae_v  = float(np.mean(np.abs(out["var_hat"] - out["var_realized"])))
        corr_s = float(out["sigma_hat"].corr(out["abs_r"]))
        corr_v = float(out["var_hat"].corr(out["var_realized"]))

        fc_rows.append({
            "Indice": idx, "Serie_returns": col, "Horizon_h": h,
            "N_train": len(common_tr), "N_test": len(out),
            "offset_intercept": round(offset, 4),
            "c_HAR": c_h, "b_d": b_d, "b_w": b_w, "b_m": b_m,
            "RMSE_X_test": rmse_x, "RMSE_var_test": rmse_v,
            "MAE_var_test": mae_v, "QLIKE_test": qlike,
            "Corr_sigmahat_absr": corr_s, "Corr_varhat_varrealized": corr_v
        })
        out.to_csv(f"{OUT_DIR_FC}/forecast_test_{idx}_h{h}.csv",
                   encoding="utf-8", index=True)

if not fc_rows:
    raise ValueError("No HAR forecast produced.")

fc_df = pd.DataFrame(fc_rows).sort_values(["Indice", "Horizon_h"])
fc_df.to_csv(f"{OUT_DIR_FC}/forecast_HAR_summary.csv", index=False, encoding="utf-8")
print("\n=== HAR_Model forecast metrics (test set) ===")
display(fc_df[["Indice", "Horizon_h", "RMSE_X_test", "QLIKE_test",
               "Corr_sigmahat_absr", "Corr_varhat_varrealized"]])


# -------------------------------------------------------
# PART C — RP_Model forecast on test set
#
# X̂_t = c + b_d X_{t-1} + b_w X̄_{t-5} + b_m X̄_{t-22}
#          + gamma_R * X_R,t-1 + gamma_P * X_P,t-1
#
# Parameters estimated on train set (Part A).
# Intercept correction applied on train before test evaluation.
# -------------------------------------------------------

rp_fc_rows = []

for col in train.columns:
    idx   = col.replace("_ret", "")
    r_all = pd.concat([train[col], test[col]], axis=0).dropna().sort_index()

    for h in HORIZONS:
        p = params[(params["Indice"] == idx) & (params["Horizon_h"] == h)]
        if len(p) == 0:
            continue

        c_rp  = float(p["c_RP"].values[0])
        bd_rp = float(p["bd_RP"].values[0])
        bw_rp = float(p["bw_RP"].values[0])
        bm_rp = float(p["bm_RP"].values[0])
        g_R   = float(p["gamma_R"].values[0])
        g_P   = float(p["gamma_P"].values[0])

        # Reconstruct X_P and X_R on full series
        x_all  = log_rv_past(r_all, h)
        span_p = max(L_PERSIST_BASE, C_SPAN * h)
        minp   = max(10, 2 * h)
        x_p    = x_all.ewm(span=span_p, adjust=False, min_periods=minp).mean()
        x_r    = x_all - x_p

        # Centre X_R on train mean (consistent with estimation)
        x_r_c  = x_r - x_r.loc[x_r.index <= train_end].mean()

        # RP_Model forecast (no look-ahead)
        xhat_rp_raw = har_rp_forecast_series(
            x_all, x_r_c, x_p,
            c_rp, bd_rp, bw_rp, bm_rp, g_R, g_P
        )

        # Intercept correction on train
        common_tr = pd.concat({"x": x_all, "xhat": xhat_rp_raw}, axis=1).dropna()
        common_tr = common_tr.loc[common_tr.index <= train_end]
        offset    = float(common_tr["x"].mean() - common_tr["xhat"].mean())
        xhat_rp   = xhat_rp_raw + offset

        out_rp = pd.concat({"r": r_all, "x_obs": x_all, "xhat_rp": xhat_rp},
                            axis=1).dropna()
        out_rp = out_rp.loc[out_rp.index > train_end].copy()
        if len(out_rp) < 30:
            continue

        out_rp["sigma_hat_rp"] = np.exp(out_rp["xhat_rp"]).clip(lower=EPS)

        diff_rp = out_rp["x_obs"] - out_rp["xhat_rp"]
        rp_fc_rows.append({
            "Indice": idx, "Horizon_h": h,
            "N_test": len(out_rp),
            "alpha_h": float(p["alpha_h"].values[0]),
            "gamma_R": g_R, "gamma_P": g_P,
            "offset": round(offset, 4),
            "RMSE_X_test": float(np.sqrt(np.mean(diff_rp**2))),
            "QLIKE_test":  float(np.mean(-diff_rp + np.exp(2*diff_rp)*0.5)),
            "Corr":        float(out_rp["x_obs"].corr(out_rp["xhat_rp"]))
        })

        out_rp.to_csv(f"{OUT_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv",
                      encoding="utf-8", index=True)

rp_fc_df = pd.DataFrame(rp_fc_rows).sort_values(["Indice", "Horizon_h"])
rp_fc_df.to_csv(f"{OUT_DIR_RP}/forecast_RP_summary.csv", index=False, encoding="utf-8")
print("\n=== RP_Model forecast metrics (test set) ===")
display(rp_fc_df[["Indice", "Horizon_h", "alpha_h",
                   "gamma_R", "gamma_P",
                   "RMSE_X_test", "QLIKE_test", "Corr"]])


# -------------------------------------------------------
# PART D — Benchmark: RP_Model vs HAR vs RW vs SMA20 vs EWMA94
# -------------------------------------------------------

bench_rows = []

for col in train.columns:
    idx   = col.replace("_ret", "")
    r_all = pd.concat([train[col], test[col]], axis=0).dropna().sort_index()

    for h in HORIZONS:
        fcsv_har = f"{OUT_DIR_FC}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{OUT_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            continue

        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"],
                              index_col="Date").sort_index()
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"],
                              index_col="Date").sort_index()

        if "xhat" not in fc_har.columns or "xhat_rp" not in fc_rp.columns:
            continue

        x_obs_all = log_rv_past(r_all, h)
        x_rw      = x_obs_all.shift(1)
        x_sma20   = x_obs_all.shift(1).rolling(20, min_periods=20).mean()
        x_ewma    = ewma_log_sigma(r_all, lam=0.94)

        d = pd.DataFrame({
            "x_obs":     x_obs_all,
            "RP_Model":  fc_rp["xhat_rp"],
            "HAR_Model": fc_har["xhat"],
            "RW":        x_rw,
            "SMA20":     x_sma20,
            "EWMA94":    x_ewma
        })
        d = d.loc[d.index > train_end].dropna()
        if len(d) < 50:
            continue

        for m in ["RP_Model", "HAR_Model", "RW", "SMA20", "EWMA94"]:
            met = eval_metrics(d["x_obs"].values, d[m].values)
            bench_rows.append({
                "Indice": idx, "Horizon_h": h, "Model": m,
                "N_obs": len(d), **met
            })

if not bench_rows:
    print("No benchmark results.")
else:
    metrics_long = (
        pd.DataFrame(bench_rows)
        .sort_values(["Indice", "Horizon_h", "Model"])
        .reset_index(drop=True)
    )
    metrics_long.to_csv(f"{BASE_DIR}/benchmark_HAR.csv", index=False, encoding="utf-8")

    best = []
    for (idx, h), g in metrics_long.groupby(["Indice", "Horizon_h"]):
        best.append({
            "Indice": idx, "Horizon_h": h,
            "Best_RMSE":  g.loc[g["RMSE_logsigma"].idxmin(), "Model"],
            "Best_QLIKE": g.loc[g["QLIKE_stable"].idxmin(),  "Model"],
            "Best_Corr":  g.loc[g["Corr"].idxmax(),           "Model"]
        })
    best_df = (pd.DataFrame(best)
               .sort_values(["Indice", "Horizon_h"])
               .reset_index(drop=True))
    wins = (
        best_df.melt(id_vars=["Indice", "Horizon_h"],
                     value_vars=["Best_RMSE", "Best_QLIKE", "Best_Corr"],
                     var_name="Metric", value_name="Model")
        .groupby(["Metric", "Model"]).size()
        .reset_index(name="Wins")
        .sort_values(["Metric", "Wins"], ascending=[True, False])
    )
    best_df.to_csv(f"{BASE_DIR}/benchmark_HAR_best.csv",  index=False, encoding="utf-8")
    wins.to_csv(f"{BASE_DIR}/benchmark_HAR_wins.csv",      index=False, encoding="utf-8")

    print("\n=== Complete benchmark metrics ===")
    display(metrics_long)
    print("\n=== Best model by index and horizon ===")
    display(best_df)
    print("\n=== Win count ===")
    display(wins)

In [ ]:
# CELL 20 — Diebold-Mariano test: RP_Model vs HAR_Model
#
# The Diebold-Mariano (1995) test assesses whether the difference in
# forecast accuracy between RP_Model and HAR_Model is statistically
# significant, or could have arisen by chance.
#
# For each index i and horizon h, define the loss differential:
#   d_t = L(e_HAR,t) - L(e_RP,t)
#
# where the loss function L is either:
#   - MSE:   L(e) = e²
#   - QLIKE: L(e) = -e + 0.5 * exp(2e),   e = x_obs - x_hat
#
# H0: E[d_t] = 0  (equal predictive accuracy)
# H1: E[d_t] > 0  (RP_Model is more accurate, i.e. lower loss)
#
# The DM statistic is:
#   DM = d̄ / sqrt(LRV / T)
#
# where d̄ is the sample mean of d_t and LRV is a HAC-consistent
# estimate of the long-run variance (Newey-West with h-1 lags).
# Under H0, DM ~ N(0,1) asymptotically.
#
# I've used a one-sided test (H1: RP_Model better than HAR).
# A negative DM statistic means RP_Model has lower average loss.

import os
import numpy as np
import pandas as pd
from scipy import stats

FC_DIR_HAR = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP  = f"{BASE_DIR}/forecast_test_RP"
HORIZONS   = [1, 5, 20, 60]
INDICI     = ["FTSE", "GDAXI", "GSPC", "HSI", "N225"]


# -------------------------------------------------------
# Newey-West HAC variance estimator
# -------------------------------------------------------

def newey_west_lrv(d, max_lag):
    """
    Long-run variance of d_t via Newey-West (Bartlett kernel).
    max_lag: number of lags (typically h-1 for h-step forecasts,
             minimum 1 to avoid degenerate cases).
    """
    T    = len(d)
    d    = d - d.mean()
    gamma0 = np.mean(d ** 2)
    lrv    = gamma0
    for lag in range(1, max_lag + 1):
        w       = 1.0 - lag / (max_lag + 1)   # Bartlett weight
        gamma_l = np.mean(d[lag:] * d[:-lag])
        lrv    += 2.0 * w * gamma_l
    return max(lrv, 1e-12)   # floor to avoid division by zero


# -------------------------------------------------------
# Diebold-Mariano test (one-sided: H1 = RP better)
# -------------------------------------------------------

def dm_test(x_obs, xhat_har, xhat_rp, h, loss="mse"):
    """
    Diebold-Mariano test of RP_Model vs HAR_Model.

    Parameters
    ----------
    x_obs   : array — observed log-sigma
    xhat_har: array — HAR forecast
    xhat_rp : array — RP_Model forecast
    h       : int   — forecast horizon (determines NW lags)
    loss    : str   — "mse" or "qlike"

    Returns
    -------
    dm_stat : float — DM statistic (negative = RP better)
    p_value : float — one-sided p-value (H1: RP better)
    d_mean  : float — mean loss differential (HAR - RP)
    """
    x_obs    = np.asarray(x_obs,    dtype=float)
    xhat_har = np.asarray(xhat_har, dtype=float)
    xhat_rp  = np.asarray(xhat_rp,  dtype=float)

    mask = np.isfinite(x_obs) & np.isfinite(xhat_har) & np.isfinite(xhat_rp)
    x_obs    = x_obs[mask]
    xhat_har = xhat_har[mask]
    xhat_rp  = xhat_rp[mask]
    T = len(x_obs)
    if T < 30:
        return np.nan, np.nan, np.nan

    e_har = x_obs - xhat_har
    e_rp  = x_obs - xhat_rp

    if loss == "mse":
        L_har = e_har ** 2
        L_rp  = e_rp  ** 2
    elif loss == "qlike":
        L_har = -e_har + 0.5 * np.exp(2.0 * e_har)
        L_rp  = -e_rp  + 0.5 * np.exp(2.0 * e_rp)
    else:
        raise ValueError(f"Unknown loss: {loss}")

    d       = L_har - L_rp          # positive = HAR worse = RP better
    d_mean  = float(np.mean(d))
    max_lag = max(h - 1, 1)         # NW lags: h-1 (min 1)
    lrv     = newey_west_lrv(d, max_lag)
    se      = np.sqrt(lrv / T)
    dm_stat = d_mean / se

    # One-sided p-value: H1 = d_mean > 0 = RP better
    p_value = float(1.0 - stats.norm.cdf(dm_stat))

    return float(dm_stat), p_value, d_mean


# -------------------------------------------------------
# Main loop
# -------------------------------------------------------

dm_rows = []

for idx in INDICI:
    for h in HORIZONS:

        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"

        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            print(f"  {idx} h={h}: file not found, skipping.")
            continue

        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"],
                              index_col="Date").sort_index()
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"],
                              index_col="Date").sort_index()

        if "xhat" not in fc_har.columns or "xhat_rp" not in fc_rp.columns \
                or "x_obs" not in fc_har.columns:
            print(f"  {idx} h={h}: missing columns, skipping.")
            continue

        common = fc_har.index.intersection(fc_rp.index)
        if len(common) < 50:
            continue

        x_obs    = fc_har.loc[common, "x_obs"].values
        xhat_har = fc_har.loc[common, "xhat"].values
        xhat_rp  = fc_rp.loc[common,  "xhat_rp"].values

        for loss in ["mse", "qlike"]:
            dm_s, p_val, d_mean = dm_test(x_obs, xhat_har, xhat_rp, h, loss)
            dm_rows.append({
                "Indice":        idx,
                "Horizon_h":     h,
                "Loss":          loss.upper(),
                "DM_stat":       round(dm_s,   4) if np.isfinite(dm_s)   else np.nan,
                "p_value_onesided": round(p_val, 4) if np.isfinite(p_val) else np.nan,
                "d_mean":        round(d_mean, 6) if np.isfinite(d_mean) else np.nan,
                "RP_better_5pct": (p_val < 0.05) if np.isfinite(p_val) else False,
                "RP_better_10pct":(p_val < 0.10) if np.isfinite(p_val) else False,
            })

if not dm_rows:
    print("No DM results produced.")
else:
    dm_df = (pd.DataFrame(dm_rows)
             .sort_values(["Loss", "Indice", "Horizon_h"])
             .reset_index(drop=True))

    dm_df.to_csv(f"{BASE_DIR}/diebold_mariano_rp_vs_har.csv",
                 index=False, encoding="utf-8")

    # -------------------------------------------------------
    # Summary: wins at 5% and 10% significance
    # -------------------------------------------------------
    for loss in ["MSE", "QLIKE"]:
        sub = dm_df[dm_df["Loss"] == loss]
        n5  = sub["RP_better_5pct"].sum()
        n10 = sub["RP_better_10pct"].sum()
        print(f"\n=== DM results — {loss} loss ===")
        print(f"  RP_Model significantly better at 5%:  {n5}/20 cases")
        print(f"  RP_Model significantly better at 10%: {n10}/20 cases")
        display(sub[["Indice", "Horizon_h", "DM_stat",
                      "p_value_onesided", "d_mean",
                      "RP_better_5pct", "RP_better_10pct"]])

    # -------------------------------------------------------
    # Pivot table: DM stat by index and horizon (MSE)
    # -------------------------------------------------------
    print("\n=== DM statistic pivot (MSE) — negative = RP better ===")
    pivot_mse = (dm_df[dm_df["Loss"] == "MSE"]
                 .pivot(index="Indice", columns="Horizon_h", values="DM_stat")
                 .round(3))
    display(pivot_mse)

    print("\n=== p-value pivot (MSE, one-sided) ===")
    pivot_p = (dm_df[dm_df["Loss"] == "MSE"]
               .pivot(index="Indice", columns="Horizon_h", values="p_value_onesided")
               .round(3))
    display(pivot_p)

    # Cases where HAR is significantly better than RP_Model (QLIKE loss):
    # A negative DM statistic under QLIKE means RP_Model has HIGHER average loss,
    # i.e. HAR is more accurate. At h = 1, the QLIKE DM statistics are:
    #   FTSE:  DM ≈ −4.09  → HAR significantly better (p ≈ 1.0, one-sided)
    #   GDAXI: DM ≈ −4.65  → HAR significantly better
    #   N225:  DM ≈ −4.40  → HAR significantly better (also at h = 5)
    # RP_Model can simultaneously reduce RMSE and worsen QLIKE because the two
    # objectives are not equivalent: RMSE penalises large point-forecast errors,
    # while QLIKE measures the calibration of the full conditional distribution.
    # The X_R component at h = 1 appears to add noise that distorts the shape
    # of the forecast distribution even while reducing squared-error loss.


    # -------------------------------------------------------
    # Heatmap: DM statistics and p-values — QLIKE loss
    # -------------------------------------------------------
    dm_qlike = dm_df[dm_df["Loss"] == "QLIKE"].copy()

    pivot_dm_qlike = (dm_qlike
                      .pivot(index="Indice", columns="Horizon_h", values="DM_stat")
                      .round(3))
    pivot_pv_qlike = (dm_qlike
                      .pivot(index="Indice", columns="Horizon_h", values="p_value_onesided")
                      .round(3))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Panel 1 — DM statistics (QLIKE)
    ax = axes[0]
    im = ax.imshow(pivot_dm_qlike.values, cmap="RdYlGn", aspect="auto",
                   vmin=-5, vmax=5)
    ax.set_xticks(range(4))
    ax.set_xticklabels([f"h={h}" for h in [1, 5, 20, 60]])
    ax.set_yticks(range(5))
    ax.set_yticklabels(pivot_dm_qlike.index)
    ax.set_title("DM Statistic (QLIKE)
Positive = RP better", fontsize=11)
    for i in range(5):
        for j in range(4):
            val = pivot_dm_qlike.values[i, j]
            ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                    fontsize=9, color="black")
    plt.colorbar(im, ax=ax)

    # Panel 2 — p-values (QLIKE)
    ax = axes[1]
    im2 = ax.imshow(pivot_pv_qlike.values, cmap="RdYlGn_r", aspect="auto",
                    vmin=0, vmax=1)
    ax.set_xticks(range(4))
    ax.set_xticklabels([f"h={h}" for h in [1, 5, 20, 60]])
    ax.set_yticks(range(5))
    ax.set_yticklabels(pivot_pv_qlike.index)
    ax.set_title("p-value (QLIKE, one-sided)
* = significant at 5%", fontsize=11)
    for i in range(5):
        for j in range(4):
            pv  = pivot_pv_qlike.values[i, j]
            txt = f"{pv:.3f}" + ("*" if pv < 0.05 else "")
            ax.text(j, i, txt, ha="center", va="center",
                    fontsize=9, color="black")
    plt.colorbar(im2, ax=axes[1])

    plt.suptitle("Diebold-Mariano Test: RP_Model vs HAR — QLIKE Loss", fontsize=12)
    plt.tight_layout()
    out_path = f"{BASE_DIR}/dm_qlike_heatmap.png"
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

    print("
=== DM statistic pivot (QLIKE) — negative = HAR better ===")
    display(pivot_dm_qlike)

    print("
=== p-value pivot (QLIKE, one-sided) ===")
    display(pivot_pv_qlike)


In [ ]:
# CELL 21 — Mincer-Zarnowitz efficiency test (HAR and RP_Model)
#
# The Mincer-Zarnowitz (1969) regression tests forecast efficiency:
#   x_obs,t = alpha + beta * x_hat_t + eps_t
#
# Under correct specification: alpha = 0 (no bias), beta = 1 (full efficiency).
# The joint F-test on H0: (alpha=0, beta=1) rejects if the forecast is biased
# or inefficient. A beta < 1 indicates that the forecast over-reacts; beta > 1
# indicates under-reaction. A non-zero alpha indicates a systematic level shift.

import os, numpy as np, pandas as pd
from scipy import stats

FC_DIR_HAR = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP  = f"{BASE_DIR}/forecast_test_RP"
HORIZONS   = [1, 5, 20, 60]
INDICI     = ["FTSE", "GDAXI", "GSPC", "HSI", "N225"]


def mincer_zarnowitz(x_obs, x_hat):
    """OLS of x_obs on (1, x_hat). Returns alpha, beta, se_a, se_b, F, p_F, R2."""
    x_obs = np.asarray(x_obs, dtype=float)
    x_hat = np.asarray(x_hat, dtype=float)
    mask  = np.isfinite(x_obs) & np.isfinite(x_hat)
    x_obs, x_hat = x_obs[mask], x_hat[mask]
    T = len(x_obs)
    if T < 30:
        return (np.nan,) * 7
    X      = np.column_stack([np.ones(T), x_hat])
    bhat, *_ = np.linalg.lstsq(X, x_obs, rcond=None)
    alpha, beta = float(bhat[0]), float(bhat[1])
    e   = x_obs - X @ bhat
    s2  = float(np.sum(e**2) / (T - 2))
    XtXi = np.linalg.inv(X.T @ X)
    se   = np.sqrt(s2 * np.diag(XtXi))
    # Joint F-test H0: alpha=0, beta=1
    d   = bhat - np.array([0.0, 1.0])
    F   = float((d @ np.linalg.inv(XtXi) @ d) / (2 * s2))
    p_F = float(1 - stats.f.cdf(F, dfn=2, dfd=T - 2))
    R2  = float(1 - np.sum(e**2) / max(np.sum((x_obs - x_obs.mean())**2), 1e-12))
    return alpha, beta, float(se[0]), float(se[1]), F, p_F, R2


rows_mz = []

for idx in INDICI:
    for h in HORIZONS:
        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            continue
        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"], index_col="Date")
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"], index_col="Date")
        if "x_obs" not in fc_har.columns:
            continue
        common   = fc_har.index.intersection(fc_rp.index)
        x_obs    = fc_har.loc[common, "x_obs"].values
        xhat_har = fc_har.loc[common, "xhat"].values
        xhat_rp  = fc_rp.loc[common, "xhat_rp"].values

        for model, xhat in [("HAR", xhat_har), ("RP_Model", xhat_rp)]:
            res = mincer_zarnowitz(x_obs, xhat)
            if not np.isfinite(res[0]):
                continue
            alpha, beta, se_a, se_b, F, p_F, R2 = res
            rows_mz.append({
                "Indice": idx, "Horizon_h": h, "Model": model,
                "alpha":    round(alpha, 4),
                "beta":     round(beta,  4),
                "se_alpha": round(se_a,  4),
                "se_beta":  round(se_b,  4),
                "F_joint":  round(F,     3),
                "p_F":      round(p_F,   4),
                "Reject_MZ_5pct": p_F < 0.05,
                "R2":       round(R2,    4),
            })

if rows_mz:
    mz_df = pd.DataFrame(rows_mz).sort_values(["Model", "Indice", "Horizon_h"])
    mz_df.to_csv(f"{BASE_DIR}/mincer_zarnowitz.csv", index=False, encoding="utf-8")

    print("=== Mincer-Zarnowitz efficiency test  (H0: alpha=0 AND beta=1) ===")
    print("  Reject_MZ_5pct=True → forecast is biased or inefficient")
    for model in ["HAR", "RP_Model"]:
        sub   = mz_df[mz_df["Model"] == model]
        n_rej = int(sub["Reject_MZ_5pct"].sum())
        print(f"\n--- {model} — H0 rejected in {n_rej}/{len(sub)} cases ---")
        display(sub[["Indice", "Horizon_h", "alpha", "beta",
                      "F_joint", "p_F", "Reject_MZ_5pct", "R2"]])


In [ ]:
# CELL 22 — RP_Model forecast with EXPANDING WINDOW (monthly re-estimation)
#
# At each re-estimation date t_k (every REFIT_FREQ = 22 days):
#   - Use ALL data from the beginning up to t_k (expanding window)
#   - Re-estimate c, b_d, b_w, b_m, gamma_R, gamma_P via OLS
#   - Apply those parameters to the next 22-day block until t_{k+1}

import os
import numpy as np
import pandas as pd

HORIZONS   = [1, 5, 20, 60]
REFIT_FREQ = 22
MIN_OBS    = 500
C_ALPHA    = 40.0

OUT_DIR_RP_EW = f"{BASE_DIR}/forecast_test_RP_expanding"
os.makedirs(OUT_DIR_RP_EW, exist_ok=True)

train = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_train_70.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

test = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

train_end = train.index.max()


# -------------------------------------------------------
# Fit RP_Model on a given window
# -------------------------------------------------------

def fit_har_rp_window(x_obs_win, x_r_win, x_p_win):
    x_s = pd.Series(x_obs_win).dropna()
    xr  = pd.Series(x_r_win).reindex(x_s.index)
    xp  = pd.Series(x_p_win).reindex(x_s.index)

    X_d = x_s.shift(1)
    X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
    XR1 = xr.shift(1)
    XP1 = xp.shift(1)

    df_fit = pd.DataFrame({
        "Y":   x_s,
        "X_d": X_d, "X_w": X_w, "X_m": X_m,
        "XR1": XR1, "XP1": XP1
    }).dropna()

    if len(df_fit) < 100:
        return (np.nan,) * 6

    Y    = df_fit["Y"].values
    Xmat = np.column_stack([
        np.ones(len(df_fit)),
        df_fit["X_d"].values, df_fit["X_w"].values, df_fit["X_m"].values,
        df_fit["XR1"].values, df_fit["XP1"].values
    ])
    beta, *_ = np.linalg.lstsq(Xmat, Y, rcond=None)
    return tuple(float(b) for b in beta)


# -------------------------------------------------------
# Apply RP_Model forecast with fixed parameters
# -------------------------------------------------------

def rp_forecast_apply(x_obs, x_r, x_p, c, b_d, b_w, b_m, g_R, g_P):
    x_s = pd.Series(x_obs)
    xr  = pd.Series(x_r).reindex(x_s.index)
    xp  = pd.Series(x_p).reindex(x_s.index)
    X_d = x_s.shift(1)
    X_w = x_s.shift(1).rolling(5,  min_periods=5).mean()
    X_m = x_s.shift(1).rolling(22, min_periods=22).mean()
    XR1 = xr.shift(1)
    XP1 = xp.shift(1)
    return c + b_d*X_d + b_w*X_w + b_m*X_m + g_R*XR1 + g_P*XP1


# -------------------------------------------------------
# Main loop
# -------------------------------------------------------

ew_rows       = []
param_ew_rows = []

for col in train.columns:
    idx   = col.replace("_ret", "")
    r_all = pd.concat([train[col], test[col]], axis=0).dropna().sort_index()

    for h in HORIZONS:

        span_p = max(L_PERSIST_BASE, C_SPAN * h)
        minp   = max(10, 2 * h)

        x_all = log_rv_past(r_all, h)
        x_p   = x_all.ewm(span=span_p, adjust=False, min_periods=minp).mean()
        x_r   = x_all - x_p

        # Centre X_R on train mean (consistent with Cell 19)
        xr_train_mean = float(x_r.loc[x_r.index <= train_end].mean())
        x_r_c = x_r - xr_train_mean

        full_df = pd.concat({
            "r":     r_all,
            "x_obs": x_all,
            "x_r":   x_r_c,
            "x_p":   x_p
        }, axis=1).dropna()

        test_dates = full_df.index[full_df.index > train_end]
        if len(test_dates) < 30:
            continue

        # -----------------------------------------------
        # Fit the model once on training data; offset = mean(x_obs) - mean(xhat).
        # Identical to the procedure in Cell 19 — avoids any look-ahead bias.
        # -----------------------------------------------
        train_data = full_df.loc[full_df.index <= train_end].copy()

        if len(train_data) >= MIN_OBS:
            (c_tr, b_d_tr, b_w_tr, b_m_tr,
             g_R_tr, g_P_tr) = fit_har_rp_window(
                train_data["x_obs"],
                train_data["x_r"],
                train_data["x_p"]
            )
            if np.isfinite(c_tr):
                xhat_tr = rp_forecast_apply(
                    train_data["x_obs"], train_data["x_r"], train_data["x_p"],
                    c_tr, b_d_tr, b_w_tr, b_m_tr, g_R_tr, g_P_tr
                )
                pair_tr = pd.concat(
                    {"x": train_data["x_obs"], "xhat": xhat_tr}, axis=1
                ).dropna()
                offset_ew = float(pair_tr["x"].mean() - pair_tr["xhat"].mean())
            else:
                offset_ew = 0.0
        else:
            offset_ew = 0.0

        # -----------------------------------------------
        # Expanding window refit loop
        # -----------------------------------------------
        forecasts    = pd.Series(np.nan, index=test_dates, name="xhat_rp")
        refit_points = test_dates[::REFIT_FREQ]

        for i, t_refit in enumerate(refit_points):

            window_data = full_df.loc[full_df.index <= t_refit]

            if len(window_data) < MIN_OBS:
                continue

            (c_ew, b_d_ew, b_w_ew, b_m_ew,
             g_R_ew, g_P_ew) = fit_har_rp_window(
                window_data["x_obs"],
                window_data["x_r"],
                window_data["x_p"]
            )

            if not np.isfinite(c_ew):
                continue

            if i + 1 < len(refit_points):
                t_next      = refit_points[i + 1]
                block_dates = test_dates[
                    (test_dates >= t_refit) & (test_dates < t_next)
                ]
            else:
                block_dates = test_dates[test_dates >= t_refit]

            if len(block_dates) == 0:
                continue

            x_d_b = full_df["x_obs"].shift(1).reindex(block_dates)
            x_w_b = full_df["x_obs"].shift(1).rolling(5,  min_periods=5).mean().reindex(block_dates)
            x_m_b = full_df["x_obs"].shift(1).rolling(22, min_periods=22).mean().reindex(block_dates)
            xr1_b = full_df["x_r"].shift(1).reindex(block_dates)
            xp1_b = full_df["x_p"].shift(1).reindex(block_dates)

            xhat_block = (
                c_ew
                + b_d_ew * x_d_b
                + b_w_ew * x_w_b
                + b_m_ew * x_m_b
                + g_R_ew * xr1_b
                + g_P_ew * xp1_b
            )
            forecasts.loc[block_dates] = xhat_block.values

            param_ew_rows.append({
                "Indice":      idx,  "Horizon_h":  h,
                "refit_date":  t_refit,
                "N_expanding": len(window_data),
                "c":       round(c_ew,   4),
                "b_d":     round(b_d_ew, 4),
                "b_w":     round(b_w_ew, 4),
                "b_m":     round(b_m_ew, 4),
                "gamma_R": round(g_R_ew, 4),
                "gamma_P": round(g_P_ew, 4),
            })

        # -----------------------------------------------
        # Apply intercept correction (training-set based)
        # -----------------------------------------------
        forecasts = forecasts + offset_ew

        # -----------------------------------------------
        # Build output and save
        # -----------------------------------------------
        out_ew = pd.concat({
            "r":       r_all,
            "x_obs":   x_all,
            "xhat_rp": forecasts
        }, axis=1)
        out_ew = out_ew.loc[out_ew.index > train_end].dropna()

        if len(out_ew) < 30:
            continue

        out_ew["sigma_hat_rp"] = np.exp(out_ew["xhat_rp"]).clip(lower=EPS)

        out_ew.to_csv(
            f"{OUT_DIR_RP_EW}/forecast_test_RP_EW_{idx}_h{h}.csv",
            encoding="utf-8", index=True
        )

        diff_ew = out_ew["x_obs"] - out_ew["xhat_rp"]
        n_ref   = len([p for p in param_ew_rows
                       if p["Indice"] == idx and p["Horizon_h"] == h])
        ew_rows.append({
            "Indice":      idx,
            "Horizon_h":   h,
            "N_test":      len(out_ew),
            "RMSE_X_test": float(np.sqrt(np.mean(diff_ew**2))),
            "QLIKE_test":  float(np.mean(-diff_ew + np.exp(2*diff_ew)*0.5)),
            "Corr":        float(out_ew["x_obs"].corr(out_ew["xhat_rp"])),
            "offset_ew":   round(offset_ew, 4),
            "N_refits":    n_ref
        })
        print(f"  {idx} h={h}: {len(out_ew)} test obs, "
              f"{n_ref} refits, "
              f"RMSE={ew_rows[-1]['RMSE_X_test']:.4f}, "
              f"offset={offset_ew:.4f}")

# -------------------------------------------------------
# Save results
# -------------------------------------------------------

ew_df = pd.DataFrame(ew_rows).sort_values(["Indice", "Horizon_h"])
ew_df.to_csv(f"{BASE_DIR}/forecast_RP_expanding_summary.csv",
             index=False, encoding="utf-8")

if param_ew_rows:
    pd.DataFrame(param_ew_rows).to_csv(
        f"{BASE_DIR}/gamma_evolution_expanding.csv",
        index=False, encoding="utf-8"
    )

print("\n=== RP_Model — Expanding Window metrics (test set) ===")
display(ew_df[["Indice", "Horizon_h", "RMSE_X_test",
               "QLIKE_test", "Corr", "N_refits", "offset_ew"]])

try:
    fixed_df = pd.read_csv(f"{BASE_DIR}/forecast_test_RP/forecast_RP_summary.csv")
    comp = ew_df.merge(
        fixed_df[["Indice", "Horizon_h", "RMSE_X_test", "QLIKE_test", "Corr"]],
        on=["Indice", "Horizon_h"], suffixes=("_expanding", "_fixed")
    )
    comp["RMSE_improvement_%"] = (
        (comp["RMSE_X_test_fixed"] - comp["RMSE_X_test_expanding"])
        / comp["RMSE_X_test_fixed"] * 100
    ).round(2)
    print("\n=== Fixed training vs Expanding window — RMSE comparison ===")
    display(comp[["Indice", "Horizon_h",
                  "RMSE_X_test_fixed", "RMSE_X_test_expanding",
                  "RMSE_improvement_%"]])
except FileNotFoundError:
    print("\n(Run Cell 19 first to generate the fixed-training comparison)")

In [ ]:
# CELL 23 — Stability of gamma_R and gamma_P over time (expanding window)
#
# Uses gamma_evolution_expanding.csv produced by CELL 22.
# Stable coefficients over time confirm that the RP_Model structure is not
# driven by any single sub-period and that the parameters are not spurious.
# A sign flip in gamma_R across the sample would indicate structural instability.

import os, numpy as np, pandas as pd, matplotlib.pyplot as plt

gamma_file = f"{BASE_DIR}/gamma_evolution_expanding.csv"

if not os.path.exists(gamma_file):
    print("gamma_evolution_expanding.csv not found — run CELL 22 first.")
else:
    gdf = pd.read_csv(gamma_file, parse_dates=["refit_date"])
    gdf = gdf.rename(columns={"refit_date": "Date"})

    INDICI   = ["FTSE", "GDAXI", "GSPC", "HSI", "N225"]
    HORIZONS = [1, 5, 20, 60]

    fig, axes = plt.subplots(
        len(INDICI), len(HORIZONS),
        figsize=(16, 3 * len(INDICI)),
        sharex=False, sharey=False
    )
    axes = np.atleast_2d(axes)

    for i, idx in enumerate(INDICI):
        for j, h in enumerate(HORIZONS):
            ax  = axes[i, j]
            sub = gdf[(gdf["Indice"] == idx) & (gdf["Horizon_h"] == h)].sort_values("Date")
            if sub.empty:
                ax.set_visible(False)
                continue
            ax.plot(sub["Date"], sub["gamma_R"], label="\u03b3_R (rough)",      color="steelblue",  lw=1.2)
            ax.plot(sub["Date"], sub["gamma_P"], label="\u03b3_P (persistent)", color="tomato", lw=1.2)
            ax.axhline(0, color="black", lw=0.7, ls="--")
            ax.set_title(f"{idx}  h={h}", fontsize=9)
            if i == 0 and j == 0:
                ax.legend(fontsize=7)
            ax.tick_params(axis="x", rotation=30, labelsize=7)

    fig.suptitle("Expanding-window parameter stability: \u03b3_R and \u03b3_P", fontsize=12, y=1.01)
    fig.tight_layout()

    OUT_DIR = f"{BASE_DIR}/grafici_acf"
    os.makedirs(OUT_DIR, exist_ok=True)
    fig.savefig(f"{OUT_DIR}/gamma_evolution_expanding.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Gamma evolution plot saved.")

    print("\n=== Mean \u00b1 std of gamma_R / gamma_P across re-estimation dates ===")
    summary_g = (
        gdf.groupby(["Indice", "Horizon_h"])[["gamma_R", "gamma_P"]]
        .agg(["mean", "std"])
        .round(4)
    )
    display(summary_g)


## RP_Model with Expanding Window — Methodological Motivation

### Why the fixed training set may be insufficient

In Cell 19, the parameters $\gamma_R$ and $\gamma_P$ are estimated **once**
on the fixed training set (March 2000 – April 2018) and applied unchanged
to the entire test period (April 2018 – March 2026).

The test set encompasses three structurally distinct episodes:
- **2018–2019**: moderate stress, elevated but not extreme volatility
- **COVID-19 (Feb–Mar 2020)**: sharpest single volatility spike in the sample
- **2022–2023**: prolonged regime of persistently elevated volatility

Parameters estimated on the 2000–2018 training period reflect the market
dynamics of that era and may become increasingly stale as the test period
progresses through these structurally different regimes.

### The expanding window solution

At each re-estimation date $t_k$ (every **22 trading days**):
1. Estimate $(c_k, b_{d,k}, b_{w,k}, b_{m,k}, \gamma_{R,k}, \gamma_{P,k})$
   by OLS on **all data from $t_0$ to $t_k$**
2. Apply those parameters to forecast the next 22-day block
3. Apply intercept correction computed on the training set

### Results

The expanding window yields no meaningful improvement over the fixed training set.
Across all 20 index–horizon combinations, the RMSE change ranges from −0.06% to +0.10%,
with most cases below ±0.05%. The Diebold-Mariano comparison (Cell 24) confirms that
the expanding model wins 7/20 cases at the 5% level versus 6/20 for the fixed model —
a negligible difference that does not persist at h = 20 or h = 60.

The gamma stability plot (Cell 23) explains why: $\gamma_R$ and $\gamma_P$ exhibit
virtually no drift over time, with standard deviations across re-estimation dates
typically below 0.01. The relationship between the rough and persistent components
and future volatility is stable across the full 2000–2026 sample, including the
COVID-19 and rate-hike episodes. This stability is consistent with the parameters
reflecting a structural property of log-volatility dynamics rather than a
regime-specific artifact.

In [ ]:
# CELL 24 — Diebold-Mariano test: RP_Model (Expanding Window) vs HAR_Model
#
# This cell replicates the DM test of Cell 20 but uses the expanding-window
# RP_Model forecasts (produced by Cell 22) instead of the fixed-training ones.
#
# the idea is to understand if the adaptive specification produce
# statistically significant improvements over HAR relative to the fixed model
#
# H0: E[d_t] = 0  (equal predictive accuracy)
# H1: E[d_t] > 0  (RP_expanding is more accurate than HAR)
#
# Results are saved to diebold_mariano_RP_EW_vs_HAR.csv and compared
# against the fixed-training DM results from Cell 20.

import os
import numpy as np
import pandas as pd
from scipy import stats

FC_DIR_HAR    = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP_EW  = f"{BASE_DIR}/forecast_test_RP_expanding"
HORIZONS      = [1, 5, 20, 60]
INDICI        = ["FTSE", "GDAXI", "GSPC", "HSI", "N225"]


# -------------------------------------------------------
# Newey-West HAC variance estimator
# -------------------------------------------------------

def newey_west_lrv(d, max_lag):
    T      = len(d)
    d      = d - d.mean()
    gamma0 = np.mean(d ** 2)
    lrv    = gamma0
    for lag in range(1, max_lag + 1):
        w       = 1.0 - lag / (max_lag + 1)
        gamma_l = np.mean(d[lag:] * d[:-lag])
        lrv    += 2.0 * w * gamma_l
    return max(lrv, 1e-12)


# -------------------------------------------------------
# Diebold-Mariano test (one-sided: H1 = RP_EW better)
# -------------------------------------------------------

def dm_test_ew(x_obs, xhat_har, xhat_rp_ew, h, loss="mse"):
    x_obs      = np.asarray(x_obs,      dtype=float)
    xhat_har   = np.asarray(xhat_har,   dtype=float)
    xhat_rp_ew = np.asarray(xhat_rp_ew, dtype=float)

    mask = (np.isfinite(x_obs) &
            np.isfinite(xhat_har) &
            np.isfinite(xhat_rp_ew))
    x_obs      = x_obs[mask]
    xhat_har   = xhat_har[mask]
    xhat_rp_ew = xhat_rp_ew[mask]
    T = len(x_obs)
    if T < 30:
        return np.nan, np.nan, np.nan

    e_har = x_obs - xhat_har
    e_rp  = x_obs - xhat_rp_ew

    if loss == "mse":
        L_har = e_har ** 2
        L_rp  = e_rp  ** 2
    elif loss == "qlike":
        L_har = -e_har + 0.5 * np.exp(2.0 * e_har)
        L_rp  = -e_rp  + 0.5 * np.exp(2.0 * e_rp)
    else:
        raise ValueError(f"Unknown loss: {loss}")

    d       = L_har - L_rp
    d_mean  = float(np.mean(d))
    max_lag = max(h - 1, 1)
    lrv     = newey_west_lrv(d, max_lag)
    se      = np.sqrt(lrv / T)
    dm_stat = d_mean / se
    p_value = float(1.0 - stats.norm.cdf(dm_stat))

    return float(dm_stat), p_value, d_mean


# -------------------------------------------------------
# Main loop
# -------------------------------------------------------

dm_ew_rows = []

for idx in INDICI:
    for h in HORIZONS:

        fcsv_har   = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp_ew = f"{FC_DIR_RP_EW}/forecast_test_RP_EW_{idx}_h{h}.csv"

        if not os.path.exists(fcsv_har):
            print(f"  {idx} h={h}: HAR file not found, skipping.")
            continue
        if not os.path.exists(fcsv_rp_ew):
            print(f"  {idx} h={h}: RP_EW file not found — run Cell 22 first.")
            continue

        fc_har   = pd.read_csv(fcsv_har,   parse_dates=["Date"],
                               index_col="Date").sort_index()
        fc_rp_ew = pd.read_csv(fcsv_rp_ew, parse_dates=["Date"],
                               index_col="Date").sort_index()

        if "xhat" not in fc_har.columns:
            print(f"  {idx} h={h}: missing 'xhat' column in HAR file.")
            continue
        if "xhat_rp" not in fc_rp_ew.columns:
            print(f"  {idx} h={h}: missing 'xhat_rp' column in RP_EW file.")
            continue

        common = fc_har.index.intersection(fc_rp_ew.index)
        if len(common) < 50:
            continue

        x_obs      = fc_har.loc[common, "x_obs"].values
        xhat_har   = fc_har.loc[common, "xhat"].values
        xhat_rp_ew = fc_rp_ew.loc[common, "xhat_rp"].values

        for loss in ["mse", "qlike"]:
            dm_s, p_val, d_mean = dm_test_ew(
                x_obs, xhat_har, xhat_rp_ew, h, loss
            )
            dm_ew_rows.append({
                "Indice":           idx,
                "Horizon_h":        h,
                "Loss":             loss.upper(),
                "DM_stat":          round(dm_s,   4) if np.isfinite(dm_s)   else np.nan,
                "p_value_onesided": round(p_val,  4) if np.isfinite(p_val)  else np.nan,
                "d_mean":           round(d_mean, 6) if np.isfinite(d_mean) else np.nan,
                "RP_EW_better_5pct":  (p_val < 0.05) if np.isfinite(p_val) else False,
                "RP_EW_better_10pct": (p_val < 0.10) if np.isfinite(p_val) else False,
            })

if not dm_ew_rows:
    print("No DM results produced. Check that Cell 22 has been run.")
else:
    dm_ew_df = (pd.DataFrame(dm_ew_rows)
                .sort_values(["Loss", "Indice", "Horizon_h"])
                .reset_index(drop=True))

    dm_ew_df.to_csv(
        f"{BASE_DIR}/diebold_mariano_RP_EW_vs_HAR.csv",
        index=False, encoding="utf-8"
    )

    # -------------------------------------------------------
    # Summary: wins at 5% and 10%
    # -------------------------------------------------------
    for loss in ["MSE", "QLIKE"]:
        sub = dm_ew_df[dm_ew_df["Loss"] == loss]
        n5  = sub["RP_EW_better_5pct"].sum()
        n10 = sub["RP_EW_better_10pct"].sum()
        print(f"\n=== DM results (Expanding Window) — {loss} loss ===")
        print(f"  RP_EW significantly better at 5%:  {n5}/20 cases")
        print(f"  RP_EW significantly better at 10%: {n10}/20 cases")
        display(sub[["Indice", "Horizon_h", "DM_stat",
                      "p_value_onesided", "d_mean",
                      "RP_EW_better_5pct", "RP_EW_better_10pct"]])

    # -------------------------------------------------------
    # Pivot: DM stat by index and horizon (MSE)
    # -------------------------------------------------------
    print("\n=== DM statistic pivot — Expanding Window (MSE) ===")
    pivot_mse = (dm_ew_df[dm_ew_df["Loss"] == "MSE"]
                 .pivot(index="Indice", columns="Horizon_h", values="DM_stat")
                 .round(3))
    display(pivot_mse)

    print("\n=== p-value pivot — Expanding Window (MSE, one-sided) ===")
    pivot_p = (dm_ew_df[dm_ew_df["Loss"] == "MSE"]
               .pivot(index="Indice", columns="Horizon_h",
                      values="p_value_onesided")
               .round(3))
    display(pivot_p)

    # -------------------------------------------------------
    # Comparison summary: Fixed vs Expanding Window
    # -------------------------------------------------------
    fixed_file = f"{BASE_DIR}/diebold_mariano_rp_vs_har.csv"
    if os.path.exists(fixed_file):
        dm_fixed = pd.read_csv(fixed_file)
        print("\n=== Comparison: Fixed vs Expanding Window (MSE wins at 5%) ===")
        fixed_wins = dm_fixed[
            (dm_fixed["Loss"] == "MSE") &
            (dm_fixed["RP_better_5pct"] == True)
        ][["Indice", "Horizon_h", "DM_stat", "p_value_onesided"]]
        fixed_wins.columns = ["Indice", "Horizon_h",
                               "DM_stat_Fixed", "p_val_Fixed"]

        ew_wins = dm_ew_df[
            (dm_ew_df["Loss"] == "MSE") &
            (dm_ew_df["RP_EW_better_5pct"] == True)
        ][["Indice", "Horizon_h", "DM_stat", "p_value_onesided"]]
        ew_wins.columns = ["Indice", "Horizon_h",
                            "DM_stat_EW", "p_val_EW"]

        print(f"  Fixed model wins:     {len(fixed_wins)}/20 cases")
        print(f"  Expanding model wins: {len(ew_wins)}/20 cases")

        if len(fixed_wins) > 0:
            print("\n  Fixed model significant cases:")
            display(fixed_wins)
        if len(ew_wins) > 0:
            print("\n  Expanding model significant cases:")
            display(ew_wins)

In [ ]:
# CELL 25 — Residual diagnostics for RP_Model forecasts
#
# For each index and horizon, the forecast residual is:
#   eps_t = x_obs,t - x_hat_RP,t
#
# Ljung-Box on eps_t     : tests for remaining linear autocorrelation.
#   Significant → RP_Model does not fully capture the log-volatility dynamics
#                 (mis-specification in the conditional mean).
# Ljung-Box on eps_t²    : tests for remaining ARCH effects (heteroskedasticity).
#   Significant → the conditional variance of residuals is still time-varying.
#
# Both tests use lags = [5, 10, 20] and a 5% significance level.

import os, numpy as np, pandas as pd
from statsmodels.stats.diagnostic import acorr_ljungbox

FC_DIR_RP = f"{BASE_DIR}/forecast_test_RP"
HORIZONS  = [1, 5, 20, 60]
INDICI    = ["FTSE", "GDAXI", "GSPC", "HSI", "N225"]
LAGS_LB   = [5, 10, 20]

rows_res = []

for idx in INDICI:
    for h in HORIZONS:
        fcsv = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv):
            continue
        fc = pd.read_csv(fcsv, parse_dates=["Date"], index_col="Date").sort_index()
        if not {"x_obs", "xhat_rp"}.issubset(fc.columns):
            continue
        fc  = fc.dropna(subset=["x_obs", "xhat_rp"])
        if len(fc) < 50:
            continue

        eps  = (fc["x_obs"] - fc["xhat_rp"]).values
        eps2 = eps ** 2

        for lag in LAGS_LB:
            lb_e  = acorr_ljungbox(eps,  lags=[lag], return_df=True)
            lb_e2 = acorr_ljungbox(eps2, lags=[lag], return_df=True)
            for serie, lb in [("eps", lb_e), ("eps2", lb_e2)]:
                rows_res.append({
                    "Indice": idx, "Horizon_h": h, "Lag": lag,
                    "Series":   serie,
                    "LB_stat":  round(float(lb.loc[lag, "lb_stat"]),   2),
                    "p_value":  round(float(lb.loc[lag, "lb_pvalue"]), 4),
                    "Reject_H0": lb.loc[lag, "lb_pvalue"] < 0.05,
                })

if rows_res:
    res_df = pd.DataFrame(rows_res).sort_values(["Indice", "Horizon_h", "Series", "Lag"])
    res_df.to_csv(f"{BASE_DIR}/residual_diagnostics_RP.csv", index=False, encoding="utf-8")

    sub10 = res_df[res_df["Lag"] == 10]

    print("=== RP_Model residual diagnostics — Ljung-Box (lag=10) ===")
    print("  Reject_H0=True → significant autocorrelation")

    print("\n-- Residuals eps_t (conditional mean mis-specification) --")
    pivot_e = (sub10[sub10["Series"] == "eps"]
               .pivot(index="Indice", columns="Horizon_h",
                      values=["LB_stat", "Reject_H0"])
               .round(3))
    display(pivot_e)
    n_e = int(sub10[sub10["Series"] == "eps"]["Reject_H0"].sum())
    print(f"  eps:  H0 rejected in {n_e}/20 cases")

    print("\n-- Squared residuals eps_t² (remaining ARCH effects) --")
    pivot_e2 = (sub10[sub10["Series"] == "eps2"]
                .pivot(index="Indice", columns="Horizon_h",
                       values=["LB_stat", "Reject_H0"])
                .round(3))
    display(pivot_e2)
    n_e2 = int(sub10[sub10["Series"] == "eps2"]["Reject_H0"].sum())
    print(f"  eps²: H0 rejected in {n_e2}/20 cases")

    # NOTE — Interpretation guide:
    # If eps is autocorrelated at short horizons (h=1,5): the HAR lag structure
    # or the X_R/X_P decomposition is not capturing all persistence.
    # If eps² is autocorrelated: the model residuals are heteroskedastic, meaning
    # the conditional volatility of log-volatility forecast errors is time-varying.
    # This is expected for volatility-of-volatility but limits the reliability of
    # standard OLS inference on the RP_Model parameters.


In [ ]:
# CELL 26 — VaR backtesting with Christoffersen's framework (1998)
#
# Value-at-Risk is computed for five model specifications:
#   1) HAR_Normal     : sigma from HAR_Model,  Normal quantile
#   2) HAR_Student    : sigma from HAR_Model,  Student-t quantile
#   3) RP_Normal      : sigma from RP_Model,   Normal quantile
#   4) RP_Student     : sigma from RP_Model,   Student-t quantile
#   5) HS             : Historical Simulation  (rolling 252-day window)
#
#   VaR_t = sigma_hat_t * |z_alpha|   (1-4)
#   VaR_t = percentile(r_{t-252:t}, alpha)  (5)
#
# The test set is split into two sub-periods:
#   Normal  : 2018-04-18 → 2020-01-01  (pre-COVID, calm markets)
#   Stress  : 2020-01-01 → 2026-03-07  (COVID + rate hikes)
#
# Backtesting via Christoffersen (1998) decomposition:
#   UC  (chi2, df=1): violation rate = alpha?
#   IND (chi2, df=1): violations independent over time?
#   CC  (chi2, df=2): joint test UC + IND (main test)

import os
import numpy as np
import pandas as pd
from scipy import stats

FC_DIR_HAR        = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP         = f"{BASE_DIR}/forecast_test_RP"
PARAMS_FILE       = f"{BASE_DIR}/model_params_HAR.csv"
HORIZONS          = [1, 5, 20, 60]
CONFIDENCE_LEVELS = [0.99, 0.975, 0.95]

# Sub-period split dates
STRESS_START = pd.Timestamp("2020-01-01")

# Load nu estimates
params_df = pd.read_csv(PARAMS_FILE)
nu_dict   = {
    (row["Indice"], int(row["Horizon_h"])): float(row["nu_student_t_for_MC"])
    for _, row in params_df.iterrows()
}

print("Estimated nu by index and horizon:")
for (idx, h), nu in sorted(nu_dict.items()):
    print(f"  {idx} h={h}: nu={nu:.2f}")


# -------------------------------------------------------
# Test functions
# -------------------------------------------------------

def uc_test(n_obs, n_viol, alpha):
    """Unconditional Coverage (Kupiec 1995). H0: pi = alpha."""
    if n_obs == 0:
        return np.nan, np.nan, False
    pi = n_viol / n_obs
    if n_viol == 0:
        lr = float(-2 * n_obs * np.log(1 - alpha))
    elif n_viol == n_obs:
        lr = float(-2 * n_obs * np.log(alpha))
    else:
        lr = float(-2 * ((n_obs-n_viol) * np.log(1-alpha) + n_viol * np.log(alpha)
                         - (n_obs-n_viol) * np.log(1-pi) - n_viol * np.log(pi)))
    lr   = max(0.0, lr)
    pval = float(1 - stats.chi2.cdf(lr, df=1))
    return round(lr, 4), round(pval, 4), pval < 0.05

def ind_test(hits):
    """Independence test (Christoffersen 1998). H0: violations i.i.d."""
    hits = np.asarray(hits, dtype=int)
    n00 = int(((hits[:-1]==0) & (hits[1:]==0)).sum())
    n01 = int(((hits[:-1]==0) & (hits[1:]==1)).sum())
    n10 = int(((hits[:-1]==1) & (hits[1:]==0)).sum())
    n11 = int(((hits[:-1]==1) & (hits[1:]==1)).sum())
    pi01 = n01/(n00+n01) if (n00+n01) > 0 else 0.0
    pi11 = n11/(n10+n11) if (n10+n11) > 0 else 0.0
    pi2  = (n01+n11)/len(hits) if len(hits) > 0 else 0.0
    slog = lambda x: np.log(x) if x > 1e-12 else -1e10
    log_H0 = (n00+n10)*slog(1-pi2) + (n01+n11)*slog(pi2)
    log_H1 = (n00*slog(1-pi01) + n01*slog(pi01) +
               n10*slog(1-pi11) + n11*slog(pi11))
    lr   = float(max(0.0, -2*(log_H0 - log_H1)))
    pval = float(1 - stats.chi2.cdf(lr, df=1))
    return round(lr,4), round(pval,4), pval < 0.05, n00, n01, n10, n11

def run_backtest(r_obs, sigma_hat, alpha):
    """
    Run Christoffersen test.
    r_obs      : observed returns (array)
    sigma_hat  : VaR series (already in return units, positive values)
    alpha      : tail probability (e.g. 0.05 for 95% VaR)
    """
    n      = len(r_obs)
    hits   = (r_obs < -sigma_hat).astype(int)
    n_viol = int(hits.sum())
    pi_hat = n_viol / n if n > 0 else 0.0
    lr_uc, pval_uc, rej_uc = uc_test(n, n_viol, alpha)
    lr_ind, pval_ind, rej_ind, n00, n01, n10, n11 = ind_test(hits)
    lr_cc   = float(lr_uc + lr_ind)
    pval_cc = float(1 - stats.chi2.cdf(lr_cc, df=2))
    return {
        "N_obs":           n,
        "N_violazioni":    n_viol,
        "Tasso_osservato": round(pi_hat, 5),
        "Ratio_obs_teo":   round(pi_hat / alpha, 3) if alpha > 0 else np.nan,
        "n00": n00, "n01": n01, "n10": n10, "n11": n11,
        "LR_UC":  lr_uc,  "pval_UC":  pval_uc,  "Reject_UC":  rej_uc,
        "LR_IND": lr_ind, "pval_IND": pval_ind, "Reject_IND": rej_ind,
        "LR_CC":  round(lr_cc,4),
        "pval_CC": round(pval_cc,4),
        "Reject_CC": pval_cc < 0.05
    }

# -------------------------------------------------------
# compute_hs_var accepts alpha as a required argument so that
# the correct percentile is used for each confidence level:
#   95% VaR  → alpha=0.05  → 5th  percentile of rolling window
#   97.5% VaR → alpha=0.025 → 2.5th percentile of rolling window
#   99% VaR  → alpha=0.01  → 1st  percentile of rolling window
# -------------------------------------------------------
def compute_hs_var(r_series, alpha, window=252):
    """
    Historical Simulation VaR: rolling empirical quantile over past `window` days.
    alpha  : float — tail probability matching the confidence level being tested.
    Returns a pandas Series aligned with r_series (negative values = left tail).
    """
    return r_series.rolling(window, min_periods=window // 2).quantile(alpha)


# -------------------------------------------------------
# Main loop
# -------------------------------------------------------

rows        = []
hits_store  = {}

test_cols = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv", nrows=0
).columns.tolist()

# Load full returns for Historical Simulation
ret_all_df = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

for col in test_cols:
    idx = col.replace("_ret", "")

    for h in HORIZONS:

        # --- Load HAR forecast ---
        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har):
            continue
        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"],
                              index_col="Date").sort_index()
        if not {"sigma_hat", "r"}.issubset(fc_har.columns):
            continue
        fc_har = fc_har.dropna(subset=["sigma_hat", "r"])

        # --- Load RP_Model forecast ---
        fcsv_rp = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_rp):
            continue
        fc_rp = pd.read_csv(fcsv_rp, parse_dates=["Date"],
                             index_col="Date").sort_index()
        if "sigma_hat_rp" not in fc_rp.columns:
            continue
        fc_rp = fc_rp.dropna(subset=["sigma_hat_rp"])

        # --- Align on common dates ---
        common_idx = fc_har.index.intersection(fc_rp.index)
        if len(common_idx) < 50:
            continue

        r_all   = fc_har.loc[common_idx, "r"]
        sig_har = fc_har.loc[common_idx, "sigma_hat"]
        sig_rp  = fc_rp.loc[common_idx,  "sigma_hat_rp"]

        # Pre-load full return series for HS (hs_var_abs computed inside cl loop)
        r_full = ret_all_df[col].dropna()

        nu = nu_dict.get((idx, h), 6.0)

        for cl in CONFIDENCE_LEVELS:
            alpha  = 1.0 - cl
            z_norm = stats.norm.ppf(alpha)
            z_stud = stats.t.ppf(alpha, df=nu) * np.sqrt((nu - 2) / nu)

            # --- Historical Simulation VaR ---
            # Computed with the alpha for each confidence level:
            #   cl=0.95  → alpha=0.05  → 5th  percentile
            #   cl=0.975 → alpha=0.025 → 2.5th percentile
            #   cl=0.99  → alpha=0.01  → 1st  percentile
            hs_var_raw = compute_hs_var(r_full, alpha=alpha, window=252)
            hs_var     = hs_var_raw.reindex(common_idx)
            hs_var_abs = hs_var.abs().ffill()

            # --- Compute VaR series for each specification ---
            var_har_n  = sig_har  * abs(z_norm)
            var_har_s  = sig_har  * abs(z_stud)
            var_rp_n   = sig_rp   * abs(z_norm)
            var_rp_s   = sig_rp   * abs(z_stud)
            var_hs     = hs_var_abs

            # --- Filtered Historical Simulation (FHS) — Option 1 ---
            # z_t = r_t / sigma_hat_t  (standardised residual on test period)
            # FHS VaR_t = sigma_hat_t × |rolling_quantile(z_t, 252, alpha)|
            # Combines daily adaptation of sigma (HAR/RP) with empirical
            # tail estimation — no parametric distribution assumed on z.
            FHS_WIN  = 252
            FHS_MINP = 126   # 6-month burn-in
            z_har_fhs = r_all / sig_har.clip(lower=EPS)
            z_rp_fhs  = r_all / sig_rp.clip(lower=EPS)
            fhs_q_har = (
                z_har_fhs
                .rolling(FHS_WIN, min_periods=FHS_MINP)
                .quantile(alpha)
                .abs()
                .ffill()
            )
            fhs_q_rp = (
                z_rp_fhs
                .rolling(FHS_WIN, min_periods=FHS_MINP)
                .quantile(alpha)
                .abs()
                .ffill()
            )
            var_fhs_har = (sig_har * fhs_q_har).dropna()
            var_fhs_rp  = (sig_rp  * fhs_q_rp).dropna()

            specs = {
                "HAR_Normal":  var_har_n,
                "HAR_Student": var_har_s,
                "RP_Normal":   var_rp_n,
                "RP_Student":  var_rp_s,
                "HS":          var_hs,
                "FHS_HAR":     var_fhs_har,
                "FHS_RP":      var_fhs_rp,
            }

            for spec_name, var_series in specs.items():

                common_s = r_all.index.intersection(var_series.dropna().index)
                if len(common_s) < 20:
                    continue

                r_s   = r_all.loc[common_s].values
                var_s = var_series.loc[common_s].values
                hits  = (r_s < -var_s).astype(int)

                # Store hits for dynamic plot (only h=1, VaR 95%)
                if h == 1 and cl == 0.95:
                    hits_store[(idx, spec_name)] = pd.Series(
                        hits, index=common_s
                    )

                # --- Full test set ---
                bt_full = run_backtest(r_s, var_s, alpha)
                rows.append({
                    "Periodo": "Full",
                    "Indice": idx, "Horizon_h": h,
                    "Confidence": cl, "Alpha_teorico": round(alpha, 4),
                    "Spec": spec_name,
                    **bt_full
                })

                # --- Normal sub-period (pre-COVID) ---
                mask_n = common_s < STRESS_START
                if mask_n.sum() >= 20:
                    bt_n = run_backtest(r_s[mask_n], var_s[mask_n], alpha)
                    rows.append({
                        "Periodo": "Normal",
                        "Indice": idx, "Horizon_h": h,
                        "Confidence": cl, "Alpha_teorico": round(alpha, 4),
                        "Spec": spec_name,
                        **bt_n
                    })

                # --- Stress sub-period (COVID + rate hikes) ---
                mask_s = common_s >= STRESS_START
                if mask_s.sum() >= 20:
                    bt_s = run_backtest(r_s[mask_s], var_s[mask_s], alpha)
                    rows.append({
                        "Periodo": "Stress",
                        "Indice": idx, "Horizon_h": h,
                        "Confidence": cl, "Alpha_teorico": round(alpha, 4),
                        "Spec": spec_name,
                        **bt_s
                    })

if not rows:
    print("No VaR results.")
else:
    var_df = (pd.DataFrame(rows)
              .sort_values(["Periodo", "Indice", "Horizon_h",
                            "Confidence", "Spec"])
              .reset_index(drop=True))

    var_df.to_csv(
        f"{BASE_DIR}/var_christoffersen_all_models.csv",
        index=False, encoding="utf-8"
    )

    # -------------------------------------------------------
    # Summary: mean Ratio by period, spec and confidence
    # -------------------------------------------------------
    ratio_summary = (
        var_df.groupby(["Periodo", "Spec", "Confidence"])
        ["Ratio_obs_teo"]
        .mean()
        .reset_index()
        .sort_values(["Confidence", "Periodo", "Ratio_obs_teo"])
    )

    # -------------------------------------------------------
    # Pass CC by period and spec
    # -------------------------------------------------------
    cc_summary = (
        var_df.groupby(["Periodo", "Spec"])
        .agg(
            N_casi      = ("Reject_CC", "count"),
            Pass_CC     = ("Reject_CC", lambda x: (~x).sum()),
            Pct_pass_CC = ("Reject_CC", lambda x: round(100*(~x).mean(), 1)),
            Ratio_medio = ("Ratio_obs_teo", "mean")
        )
        .reset_index()
        .sort_values(["Periodo", "Ratio_medio"])
    )

    # -------------------------------------------------------
    # Print results
    # -------------------------------------------------------
    print("\n=== Mean Ratio by period and specification ===")
    print("  (1.0 = perfectly calibrated)")
    pivot_ratio = ratio_summary.pivot_table(
        index=["Spec"],
        columns=["Periodo", "Confidence"],
        values="Ratio_obs_teo"
    ).round(2)
    display(pivot_ratio)

    print("\n=== Pass CC count by period and specification ===")
    display(cc_summary)

    print("\n=== Key result: Normal period Ratio (pre-COVID) ===")
    normal_ratio = (
        var_df[var_df["Periodo"] == "Normal"]
        .groupby(["Spec", "Confidence"])["Ratio_obs_teo"]
        .mean()
        .reset_index()
        .sort_values(["Confidence", "Ratio_obs_teo"])
    )
    display(normal_ratio.round(3))

    print("\n=== Key result: Stress period Ratio (COVID + rate hikes) ===")
    stress_ratio = (
        var_df[var_df["Periodo"] == "Stress"]
        .groupby(["Spec", "Confidence"])["Ratio_obs_teo"]
        .mean()
        .reset_index()
        .sort_values(["Confidence", "Ratio_obs_teo"])
    )
    display(stress_ratio.round(3))

    # -------------------------------------------------------
    # OPTION 2 — Normal vs Stress direct comparison (all models)
    # -------------------------------------------------------
    print("\n=== Mean violation ratio by model and period  (CL = 95%) ===")
    print("  1.0 = perfectly calibrated | > 1.0 = over-violation | < 1.0 = under-violation")
    comp95 = var_df[var_df["Confidence"] == 0.95].copy()
    periods_order = [p for p in ["Normal", "Stress", "Full"]
                     if p in comp95["Periodo"].unique()]
    pivot_ratio95 = (
        comp95.groupby(["Spec", "Periodo"])["Ratio_obs_teo"]
        .mean()
        .unstack("Periodo")
        .reindex(columns=periods_order)
        .round(3)
        .sort_values("Full")
    )
    display(pivot_ratio95)

    print("\n=== Pass CC rate (%) by model and period  (all CL, all horizons) ===")
    print("  Higher = better | HS wins Normal; FHS should dominate overall")
    pass_rate = (
        var_df.groupby(["Spec", "Periodo"])
        .apply(lambda x: round(100.0 * (~x["Reject_CC"]).mean(), 1))
        .unstack("Periodo")
        .reindex(columns=periods_order)
        .sort_values("Full", ascending=False)
    )
    display(pass_rate)

    print("\n=== Stress-period adaptation: violation ratio vs HS baseline (CL = 95%) ===")
    print("  vs_HS < 0 → model has LOWER ratio than HS in Stress (adapts faster to crisis)")
    stress95 = comp95[comp95["Periodo"] == "Stress"]
    if len(stress95) > 0:
        stress_mean = (
            stress95.groupby("Spec")["Ratio_obs_teo"]
            .mean().round(3).reset_index()
            .sort_values("Ratio_obs_teo")
        )
        hs_ratio = stress_mean.loc[
            stress_mean["Spec"] == "HS", "Ratio_obs_teo"
        ].values
        if len(hs_ratio) > 0:
            stress_mean["vs_HS"] = (stress_mean["Ratio_obs_teo"] - hs_ratio[0]).round(3)
        display(stress_mean)

    # -------------------------------------------------------
    # Save hits for dynamic VaR plot in Cell 28
    # -------------------------------------------------------
    hits_list = []
    for (idx_k, spec_k), hits_s in hits_store.items():
        tmp = hits_s.reset_index()
        tmp.columns = ["Date", "Hit"]
        tmp["Indice"] = idx_k
        tmp["Spec"]   = spec_k
        hits_list.append(tmp)

    if hits_list:
        hits_df = pd.concat(hits_list, ignore_index=True)
        hits_df.to_csv(
            f"{BASE_DIR}/var_hits_h1_cl95.csv",
            index=False, encoding="utf-8"
        )
        print(f"\nHits series saved: {len(hits_df)} rows")

# -------------------------------------------------------
# INTERPRETATION — Structural VaR calibration failure
# -------------------------------------------------------
# All parametric models (HAR and RP_Model, Normal and Student specifications)
# fail the Christoffersen CC test systematically across all indices, horizons,
# and confidence levels. The observed violation ratio is approximately:
#   VaR 95%:   ratio ≈ 1.89–1.98  (should be 1.0)
#   VaR 97.5%: ratio ≈ 2.74–2.85  (should be 1.0)
#   VaR 99%:   ratio ≈ 4.35–5.22  (should be 1.0)
#
# The difference between RP_Student and HAR_Student is negligible (< 0.03 on
# ratios of 2–4): the RP component does not improve VaR calibration when the
# distributional assumption is fixed (Normal or Student-t with fixed nu).
# Historical Simulation passes CC in the Normal sub-period but collapses under
# Stress conditions (ratio ≈ 1.21, 0% pass rate) due to its 252-day lag.
#
# FHS_RP (Filtered Historical Simulation using RP_Model sigma) should dominate
# both HS and the parametric specifications: it combines the daily adaptivity
# of RP_Model's sigma_hat with the empirical tail distribution of standardised
# residuals z_t = r_t / sigma_hat_t. If FHS_RP violation ratios are close to
# 1.0 in both Normal and Stress periods, this demonstrates that the problem
# was never the volatility forecast but the distributional assumption.
#
# ROOT CAUSE: the log_rv_past proxy accurately estimates the conditional mean
# of log-volatility but does not capture the extreme tails of the conditional
# return distribution. Multiplying by a fixed Student-t quantile is insufficient
# for regulatory-grade VaR calibration.


# The VaR analysis documents the limitations of parametric volatility models
# with Gaussian or Student-t innovations for tail risk measurement. Observed
# violation rates are 2–4× the theoretical level, indicating systematic under-
# estimation of tail risk.
# The RP_Student vs HAR_Student comparison yields no meaningful difference and
# should be presented as such, not as evidence that one model is superior for
# risk management purposes.

In [ ]:
# CELL 27 — VaR backtesting with Christoffersen's framework (1998)
#             using RP_Model EXPANDING WINDOW forecasts
#
# This cell replicates Cell 26 but uses the expanding-window RP_Model
# forecasts (Cell 22) instead of the fixed-training ones.
#
# Purpose: robustness check — does the adaptive specification produce
# better-calibrated VaR estimates relative to the fixed model?
#
# Specifications compared:
#   1) HAR_Normal     : sigma from HAR_Model,  Normal quantile
#   2) HAR_Student    : sigma from HAR_Model,  Student-t quantile
#   3) RP_EW_Normal   : sigma from RP_EW,      Normal quantile
#   4) RP_EW_Student  : sigma from RP_EW,      Student-t quantile
#   5) HS             : Historical Simulation  (rolling 252-day window)
#
# compute_hs_var accepts alpha as a required argument so that
# the correct percentile is used for each confidence level:
#   95% VaR  → alpha=0.05  → 5th  percentile
#   97.5% VaR → alpha=0.025 → 2.5th percentile
#   99% VaR  → alpha=0.01  → 1st  percentile
#
# Output saved to: var_christoffersen_RP_EW.csv

import os
import numpy as np
import pandas as pd
from scipy import stats

FC_DIR_HAR        = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP_EW      = f"{BASE_DIR}/forecast_test_RP_expanding"
PARAMS_FILE       = f"{BASE_DIR}/model_params_HAR.csv"
HORIZONS          = [1, 5, 20, 60]
CONFIDENCE_LEVELS = [0.99, 0.975, 0.95]
STRESS_START      = pd.Timestamp("2020-01-01")

# Load nu estimates (same as Cell 26 — from fixed training)
params_df = pd.read_csv(PARAMS_FILE)
nu_dict   = {
    (row["Indice"], int(row["Horizon_h"])): float(row["nu_student_t_for_MC"])
    for _, row in params_df.iterrows()
}

# -------------------------------------------------------
# Test functions
# -------------------------------------------------------

def uc_test(n_obs, n_viol, alpha):
    if n_obs == 0:
        return np.nan, np.nan, False
    pi = n_viol / n_obs
    if n_viol == 0:
        lr = float(-2 * n_obs * np.log(1 - alpha))
    elif n_viol == n_obs:
        lr = float(-2 * n_obs * np.log(alpha))
    else:
        lr = float(-2 * ((n_obs-n_viol) * np.log(1-alpha) + n_viol * np.log(alpha)
                         - (n_obs-n_viol) * np.log(1-pi) - n_viol * np.log(pi)))
    lr   = max(0.0, lr)
    pval = float(1 - stats.chi2.cdf(lr, df=1))
    return round(lr, 4), round(pval, 4), pval < 0.05

def ind_test(hits):
    hits = np.asarray(hits, dtype=int)
    n00 = int(((hits[:-1]==0) & (hits[1:]==0)).sum())
    n01 = int(((hits[:-1]==0) & (hits[1:]==1)).sum())
    n10 = int(((hits[:-1]==1) & (hits[1:]==0)).sum())
    n11 = int(((hits[:-1]==1) & (hits[1:]==1)).sum())
    pi01 = n01/(n00+n01) if (n00+n01) > 0 else 0.0
    pi11 = n11/(n10+n11) if (n10+n11) > 0 else 0.0
    pi2  = (n01+n11)/len(hits) if len(hits) > 0 else 0.0
    slog = lambda x: np.log(x) if x > 1e-12 else -1e10
    log_H0 = (n00+n10)*slog(1-pi2) + (n01+n11)*slog(pi2)
    log_H1 = (n00*slog(1-pi01) + n01*slog(pi01) +
               n10*slog(1-pi11) + n11*slog(pi11))
    lr   = float(max(0.0, -2*(log_H0 - log_H1)))
    pval = float(1 - stats.chi2.cdf(lr, df=1))
    return round(lr,4), round(pval,4), pval < 0.05, n00, n01, n10, n11

def run_backtest(r_obs, sigma_hat, alpha):
    n      = len(r_obs)
    hits   = (r_obs < -sigma_hat).astype(int)
    n_viol = int(hits.sum())
    pi_hat = n_viol / n if n > 0 else 0.0
    lr_uc, pval_uc, rej_uc = uc_test(n, n_viol, alpha)
    lr_ind, pval_ind, rej_ind, n00, n01, n10, n11 = ind_test(hits)
    lr_cc   = float(lr_uc + lr_ind)
    pval_cc = float(1 - stats.chi2.cdf(lr_cc, df=2))
    return {
        "N_obs":           n,
        "N_violazioni":    n_viol,
        "Tasso_osservato": round(pi_hat, 5),
        "Ratio_obs_teo":   round(pi_hat / alpha, 3) if alpha > 0 else np.nan,
        "n00": n00, "n01": n01, "n10": n10, "n11": n11,
        "LR_UC":  lr_uc,  "pval_UC":  pval_uc,  "Reject_UC":  rej_uc,
        "LR_IND": lr_ind, "pval_IND": pval_ind, "Reject_IND": rej_ind,
        "LR_CC":  round(lr_cc,4),
        "pval_CC": round(pval_cc,4),
        "Reject_CC": pval_cc < 0.05
    }

# alpha is a required parameter — correct percentile per confidence level
def compute_hs_var(r_series, alpha, window=252):
    """
    Historical Simulation VaR: rolling empirical quantile over past `window` days.
    alpha : float — tail probability matching the confidence level being tested.
    """
    return r_series.rolling(window, min_periods=window // 2).quantile(alpha)


# -------------------------------------------------------
# Main loop
# -------------------------------------------------------

rows_ew = []

test_cols = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv", nrows=0
).columns.tolist()

ret_all_df = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns.csv",
    parse_dates=["Date"], index_col="Date"
).sort_index()

for col in test_cols:
    idx = col.replace("_ret", "")

    for h in HORIZONS:

        # --- Load HAR forecast ---
        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har):
            continue
        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"],
                              index_col="Date").sort_index()
        if not {"sigma_hat", "r"}.issubset(fc_har.columns):
            continue
        fc_har = fc_har.dropna(subset=["sigma_hat", "r"])

        # --- Load RP_EW forecast ---
        fcsv_rp_ew = f"{FC_DIR_RP_EW}/forecast_test_RP_EW_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_rp_ew):
            print(f"  {idx} h={h}: RP_EW file not found — run Cell 22 first.")
            continue
        fc_rp_ew = pd.read_csv(fcsv_rp_ew, parse_dates=["Date"],
                                index_col="Date").sort_index()
        if "xhat_rp" not in fc_rp_ew.columns:
            print(f"  {idx} h={h}: missing 'xhat_rp' in RP_EW file.")
            continue
        fc_rp_ew = fc_rp_ew.dropna(subset=["xhat_rp"])
        fc_rp_ew["sigma_hat_rp_ew"] = np.exp(fc_rp_ew["xhat_rp"]).clip(lower=EPS)

        # --- Align ---
        common_idx = fc_har.index.intersection(fc_rp_ew.index)
        if len(common_idx) < 50:
            continue

        r_all   = fc_har.loc[common_idx, "r"]
        sig_har = fc_har.loc[common_idx, "sigma_hat"]
        sig_ew  = fc_rp_ew.loc[common_idx, "sigma_hat_rp_ew"]

        # Pre-load full return series for HS (computed inside cl loop)
        r_full = ret_all_df[col].dropna()

        nu = nu_dict.get((idx, h), 6.0)

        for cl in CONFIDENCE_LEVELS:
            alpha  = 1.0 - cl
            z_norm = stats.norm.ppf(alpha)
            z_stud = stats.t.ppf(alpha, df=nu) * np.sqrt((nu - 2) / nu)

            # HS computed here with the correct alpha for each confidence level
            hs_var_raw = compute_hs_var(r_full, alpha=alpha, window=252)
            hs_var     = hs_var_raw.reindex(common_idx).abs().ffill()

            specs = {
                "HAR_Normal":    sig_har * abs(z_norm),
                "HAR_Student":   sig_har * abs(z_stud),
                "RP_EW_Normal":  sig_ew  * abs(z_norm),
                "RP_EW_Student": sig_ew  * abs(z_stud),
                "HS":            hs_var,
            }

            for spec_name, var_series in specs.items():
                common_s = r_all.index.intersection(var_series.dropna().index)
                if len(common_s) < 20:
                    continue

                r_s   = r_all.loc[common_s].values
                var_s = var_series.loc[common_s].values

                # Full
                bt_full = run_backtest(r_s, var_s, alpha)
                rows_ew.append({"Periodo": "Full", "Indice": idx,
                                 "Horizon_h": h, "Confidence": cl,
                                 "Alpha_teorico": round(alpha, 4),
                                 "Spec": spec_name, **bt_full})

                # Normal
                mask_n = common_s < STRESS_START
                if mask_n.sum() >= 20:
                    bt_n = run_backtest(r_s[mask_n], var_s[mask_n], alpha)
                    rows_ew.append({"Periodo": "Normal", "Indice": idx,
                                    "Horizon_h": h, "Confidence": cl,
                                    "Alpha_teorico": round(alpha, 4),
                                    "Spec": spec_name, **bt_n})

                # Stress
                mask_s = common_s >= STRESS_START
                if mask_s.sum() >= 20:
                    bt_s = run_backtest(r_s[mask_s], var_s[mask_s], alpha)
                    rows_ew.append({"Periodo": "Stress", "Indice": idx,
                                    "Horizon_h": h, "Confidence": cl,
                                    "Alpha_teorico": round(alpha, 4),
                                    "Spec": spec_name, **bt_s})

if not rows_ew:
    print("No VaR results. Check that Cell 22 has been run.")
else:
    var_ew_df = (pd.DataFrame(rows_ew)
                 .sort_values(["Periodo", "Indice", "Horizon_h",
                               "Confidence", "Spec"])
                 .reset_index(drop=True))

    var_ew_df.to_csv(
        f"{BASE_DIR}/var_christoffersen_RP_EW.csv",
        index=False, encoding="utf-8"
    )

    # -------------------------------------------------------
    # Summary: mean Ratio by period and spec
    # -------------------------------------------------------
    print("\n=== Mean Ratio — Expanding Window (1.0 = perfectly calibrated) ===")
    pivot_ratio = (
        var_ew_df.groupby(["Periodo", "Spec", "Confidence"])["Ratio_obs_teo"]
        .mean()
        .reset_index()
        .pivot_table(index="Spec", columns=["Periodo", "Confidence"],
                     values="Ratio_obs_teo")
        .round(2)
    )
    display(pivot_ratio)

    # -------------------------------------------------------
    # Comparison: Fixed RP vs Expanding RP
    # -------------------------------------------------------
    fixed_file = f"{BASE_DIR}/var_christoffersen_all_models.csv"
    if os.path.exists(fixed_file):
        var_fixed = pd.read_csv(fixed_file)

        print("\n=== Ratio comparison: RP_Fixed vs RP_EW (all periods) ===")
        comp = pd.merge(
            var_fixed[var_fixed["Spec"] == "RP_Student"]
            .groupby(["Periodo", "Confidence"])["Ratio_obs_teo"]
            .mean().reset_index()
            .rename(columns={"Ratio_obs_teo": "Ratio_RP_Fixed"}),
            var_ew_df[var_ew_df["Spec"] == "RP_EW_Student"]
            .groupby(["Periodo", "Confidence"])["Ratio_obs_teo"]
            .mean().reset_index()
            .rename(columns={"Ratio_obs_teo": "Ratio_RP_EW"}),
            on=["Periodo", "Confidence"]
        )
        comp["Delta"] = (comp["Ratio_RP_EW"] - comp["Ratio_RP_Fixed"]).round(3)
        display(comp.round(3))
    else:
        print("\n(Run Cell 26 first to generate fixed-training VaR comparison)")

    print("\nDone. Results saved to var_christoffersen_RP_EW.csv")

    # -------------------------------------------------------
    # Pass CC summary by period and spec
    # -------------------------------------------------------
    cc_summary_ew = (
        var_ew_df.groupby(["Periodo", "Spec"])
        .agg(
            N_casi      = ("Reject_CC", "count"),
            Pass_CC     = ("Reject_CC", lambda x: (~x).sum()),
            Pct_pass_CC = ("Reject_CC", lambda x: round(100*(~x).mean(), 1)),
            Ratio_medio = ("Ratio_obs_teo", "mean")
        )
        .reset_index()
        .sort_values(["Periodo", "Ratio_medio"])
    )

    print("\n=== Pass CC count — Expanding Window ===")
    display(cc_summary_ew)

In [ ]:
# CELL 30 — Probability Integral Transform (PIT) test
#
# The PIT test assesses distributional calibration of the volatility models.
# Under a correctly specified model, the probability integral transform
#   u_t = F(r_t | sigma_hat_t)
# should be i.i.d. Uniform(0,1) (Rosenblatt 1952; Diebold et al. 1998).
#
# Two distributional assumptions are tested for each model:
#   Normal  : u_t = Phi(r_t / sigma_hat_t)
#   Student : u_t = T_nu(r_t / sigma_hat_t * sqrt(nu/(nu-2)))
#             [sigma_hat is the std-dev forecast; scale for t is sigma/sqrt((nu-2)/nu)]
#
# A U-shaped histogram indicates an under-dispersed distribution (too thin tails).
# A hump-shaped histogram indicates over-dispersion (too thick tails).
# A flat histogram is consistent with correct calibration.
#
# The Kolmogorov-Smirnov test (H0: U(0,1)) quantifies the departure formally.

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# ── Reuse global paths and parameters defined in CELL 1 and CELL 26 ──────────
FC_DIR_HAR  = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP   = f"{BASE_DIR}/forecast_test_RP"
PARAMS_FILE = f"{BASE_DIR}/model_params_HAR.csv"
HORIZONS    = [1, 5, 20, 60]
OUT_DIR     = f"{BASE_DIR}/pit_results"
os.makedirs(OUT_DIR, exist_ok=True)

# Load nu estimates
params_df = pd.read_csv(PARAMS_FILE)
nu_dict   = {
    (row["Indice"], int(row["Horizon_h"])): float(row["nu_student_t_for_MC"])
    for _, row in params_df.iterrows()
}

test_cols = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv", nrows=0
).columns.tolist()

# ── Collect PIT values and KS statistics ─────────────────────────────────────
pit_store = {}   # key: (model, dist, idx, h) → array of u_t values
ks_rows   = []

for col in test_cols:
    idx = col.replace("_ret", "")

    for h in HORIZONS:

        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            continue

        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"],
                              index_col="Date").sort_index()
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"],
                              index_col="Date").sort_index()

        if not {"sigma_hat", "r"}.issubset(fc_har.columns):
            continue
        if "sigma_hat_rp" not in fc_rp.columns:
            continue

        fc_har = fc_har.dropna(subset=["sigma_hat", "r"])
        fc_rp  = fc_rp.dropna(subset=["sigma_hat_rp"])

        common_idx = fc_har.index.intersection(fc_rp.index)
        if len(common_idx) < 50:
            continue

        r        = fc_har.loc[common_idx, "r"].values
        sig_har  = fc_har.loc[common_idx, "sigma_hat"].values.clip(min=EPS)
        sig_rp   = fc_rp.loc[common_idx,  "sigma_hat_rp"].values.clip(min=EPS)
        nu       = nu_dict.get((idx, h), 6.0)

        # Standardised residuals
        z_har = r / sig_har
        z_rp  = r / sig_rp

        # PIT values — Normal
        u_har_n = stats.norm.cdf(z_har)
        u_rp_n  = stats.norm.cdf(z_rp)

        # PIT values — Student-t
        # sigma_hat is the std-dev; t distribution scale = sigma / sqrt((nu-2)/nu)
        # so z_scaled = z * sqrt(nu/(nu-2)) follows a standard t(nu)
        scale_factor = np.sqrt(nu / (nu - 2)) if nu > 2 else 1.0
        u_har_s = stats.t.cdf(z_har * scale_factor, df=nu)
        u_rp_s  = stats.t.cdf(z_rp  * scale_factor, df=nu)

        for model, u_n, u_s in [("HAR", u_har_n, u_har_s),
                                  ("RP",  u_rp_n,  u_rp_s)]:
            for dist, u in [("Normal", u_n), ("Student", u_s)]:
                key = (model, dist, idx, h)
                pit_store[key] = u
                ks_stat, ks_pval = stats.kstest(u, "uniform")
                ks_rows.append({
                    "Model":  model,
                    "Dist":   dist,
                    "Index":  idx,
                    "h":      h,
                    "N":      len(u),
                    "KS_stat": round(ks_stat, 4),
                    "KS_pval": round(ks_pval, 4),
                    "Reject_KS": ks_pval < 0.05
                })

ks_df = pd.DataFrame(ks_rows)
ks_df.to_csv(f"{OUT_DIR}/pit_ks_results.csv", index=False)

print("KS test results — rejection rates by model/distribution:")
summary = (
    ks_df
    .groupby(["Model", "Dist"])["Reject_KS"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "Rejected", "count": "Total"})
)
summary["Rate"] = (summary["Rejected"] / summary["Total"]).round(3)
print(summary.to_string())
print()
print(ks_df[ks_df["Reject_KS"]].to_string(index=False))

# ── Figure 1: Aggregate PIT histograms (2 × 2) ───────────────────────────────
NBINS = 10

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle("PIT Histograms — Aggregate (all indices & horizons)", fontsize=13)

pairs = [
    ("HAR",  "Normal",  "HAR — Normal"),
    ("RP",   "Normal",  "RP Model — Normal"),
    ("HAR",  "Student", "HAR — Student-t"),
    ("RP",   "Student", "RP Model — Student-t"),
]

for ax, (model, dist, title) in zip(axes.flat, pairs):
    u_all = np.concatenate([
        v for (m, d, ii, hh), v in pit_store.items()
        if m == model and d == dist
    ])
    ax.hist(u_all, bins=NBINS, density=True, color="#4C72B0", edgecolor="white",
            alpha=0.85, label=f"n={len(u_all)}")
    ax.axhline(1.0, color="crimson", linewidth=1.5, linestyle="--", label="U(0,1)")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("u = F(r | σ̂)", fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.set_xlim(0, 1)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/pit_aggregate.png", dpi=150)
plt.show()

# ── Figure 2: PIT by horizon — Normal distribution only (2 × 4 grid) ─────────
fig2, axes2 = plt.subplots(2, 4, figsize=(16, 7))
fig2.suptitle("PIT Histograms by Horizon — Normal distribution", fontsize=13)

for col_j, h in enumerate(HORIZONS):
    for row_i, model in enumerate(["HAR", "RP"]):
        ax = axes2[row_i, col_j]
        u_h = np.concatenate([
            v for (m, d, ii, hh), v in pit_store.items()
            if m == model and d == "Normal" and hh == h
        ])
        ax.hist(u_h, bins=NBINS, density=True, color="#55A868" if model=="RP" else "#4C72B0",
                edgecolor="white", alpha=0.85)
        ax.axhline(1.0, color="crimson", linewidth=1.5, linestyle="--")
        ax.set_title(f"{model} — h={h}", fontsize=10)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, None)
        if col_j == 0:
            ax.set_ylabel("Density", fontsize=9)
        ax.set_xlabel("u", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/pit_by_horizon_normal.png", dpi=150)
plt.show()

# ── Figure 3: PIT by horizon — Student-t (2 × 4 grid) ───────────────────────
fig3, axes3 = plt.subplots(2, 4, figsize=(16, 7))
fig3.suptitle("PIT Histograms by Horizon — Student-t distribution", fontsize=13)

for col_j, h in enumerate(HORIZONS):
    for row_i, model in enumerate(["HAR", "RP"]):
        ax = axes3[row_i, col_j]
        u_h = np.concatenate([
            v for (m, d, ii, hh), v in pit_store.items()
            if m == model and d == "Student" and hh == h
        ])
        ax.hist(u_h, bins=NBINS, density=True, color="#55A868" if model=="RP" else "#4C72B0",
                edgecolor="white", alpha=0.85)
        ax.axhline(1.0, color="crimson", linewidth=1.5, linestyle="--")
        ax.set_title(f"{model} — h={h}", fontsize=10)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, None)
        if col_j == 0:
            ax.set_ylabel("Density", fontsize=9)
        ax.set_xlabel("u", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/pit_by_horizon_student.png", dpi=150)
plt.show()

print(f"\nSaved to {OUT_DIR}:")
print("  pit_ks_results.csv")
print("  pit_aggregate.png")
print("  pit_by_horizon_normal.png")
print("  pit_by_horizon_student.png")


The PIT histograms reveal a consistent U-shaped pattern across all model specifications and horizons. Under a correctly calibrated model, the transformed values u_t = F(r_t | σ̂_t) should be uniformly distributed on [0,1]. The U-shape — an excess of probability mass near 0 and 1 — indicates that the assumed conditional distribution is systematically **under-dispersed**: the model assigns too little probability to extreme returns, underestimating the true tail thickness of the empirical distribution.

Crucially, this pattern is **identical for both HAR and RP_Model**, under both the Normal and Student-t assumptions. This has a direct interpretation: the PIT shape is driven by distributional misspecification (thin tails), not by forecast accuracy. The improvement of RP_Model over HAR in terms of σ̂ is of the order of 0.2–0.5% in RMSE — far too small to shift the PIT histogram in a detectable way.

Switching to the Student-t distribution attenuates the U-shape, confirming that heavier tails partially correct the misspecification. However, the KS test continues to reject uniformity in the majority of cases, suggesting that even the Student-t parameterisation (ν ≈ 7–10 estimated on the full sample) does not fully capture the tail behaviour of returns over the test period, particularly during the stress sub-period.

This result is consistent with the VaR analysis: the observed violation ratios (~1.9× at the 95% level for both models) are a consequence of the same distributional failure, not of poor volatility forecasting. The two models produce statistically equivalent PITs because they share the same structural limitation — a parametric distribution that cannot adapt to time-varying tail risk.

In [ ]:
# CELL 31 — Continuous Ranked Probability Score (CRPS) decomposition
#
# The CRPS generalises the Mean Squared Error to the full predictive distribution:
#   CRPS(F, y) = E_F|Y - y| - (1/2) E_F|Y - Y'|
# where Y, Y' are independent draws from F.  Lower CRPS = better forecast.
#
# Closed-form expressions used here (Gneiting & Raftery 2007):
#
#   Normal(0, σ):
#     CRPS = σ * [z*(2Φ(z)-1) + 2φ(z) - 1/√π]    where z = y/σ
#
#   Student-t(ν) with std-dev σ  (scale s = σ·√((ν-2)/ν)):
#     CRPS = s * [z_s*(2T_ν(z_s)-1) + 2t_ν(z_s)*(ν+z_s²)/(ν-1)
#                 - √ν·B(½,ν/2)/((ν-1)·B(½,(ν-1)/2))]
#            where z_s = y/s
#
# The CRPS is then decomposed as:
#   CRPS = Reliability + Resolution + Uncertainty
# (Hersbach 2000; empirical version via rank histogram)
#
# Key comparison:
#   - CRPS_HAR vs CRPS_RP  → which model gives better probabilistic forecasts?
#   - Skill score: SS = (CRPS_HAR - CRPS_RP) / CRPS_HAR  (positive = RP better)
#   - Diebold-Mariano test on d_t = CRPS_HAR_t - CRPS_RP_t

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import betaln

FC_DIR_HAR  = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP   = f"{BASE_DIR}/forecast_test_RP"
PARAMS_FILE = f"{BASE_DIR}/model_params_HAR.csv"
HORIZONS    = [1, 5, 20, 60]
OUT_DIR     = f"{BASE_DIR}/crps_results"
os.makedirs(OUT_DIR, exist_ok=True)

params_df = pd.read_csv(PARAMS_FILE)
nu_dict   = {
    (row["Indice"], int(row["Horizon_h"])): float(row["nu_student_t_for_MC"])
    for _, row in params_df.iterrows()
}

test_cols = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv", nrows=0
).columns.tolist()

# ── CRPS closed-form functions ────────────────────────────────────────────────

def crps_normal(y, sigma):
    """CRPS for N(0, sigma).  Vectorised."""
    sigma = np.maximum(sigma, EPS)
    z     = y / sigma
    return sigma * (z * (2*stats.norm.cdf(z) - 1) + 2*stats.norm.pdf(z) - 1/np.sqrt(np.pi))

def crps_student(y, sigma, nu):
    """
    CRPS for Student-t with mean 0 and std-dev sigma (nu > 2).
    Scale parameter: s = sigma * sqrt((nu-2)/nu).
    Formula: Gneiting & Raftery (2007) eq. 21.
    """
    if nu <= 2:
        return crps_normal(y, sigma)
    sigma = np.maximum(sigma, EPS)
    s     = sigma * np.sqrt((nu - 2) / nu)          # scale (not std-dev)
    z     = y / s
    # constant term: sqrt(nu) * B(1/2, nu/2) / ((nu-1) * B(1/2, (nu-1)/2))
    log_const = (
        0.5 * np.log(nu)
        + betaln(0.5, nu/2)
        - np.log(nu - 1)
        - betaln(0.5, (nu-1)/2)
    )
    const = np.exp(log_const)
    crps  = s * (
        z * (2*stats.t.cdf(z, df=nu) - 1)
        + 2*stats.t.pdf(z, df=nu) * (nu + z**2) / (nu - 1)
        - const
    )
    return crps

def dm_test_crps(d, h_lag=1):
    """
    Diebold-Mariano test on loss differential d_t = CRPS_HAR_t - CRPS_RP_t.
    Returns (DM_stat, p_value).  Positive DM → RP is better.
    HAC variance with Newey-West, bandwidth = h_lag.
    """
    n   = len(d)
    mu  = d.mean()
    # Newey-West long-run variance
    gamma0 = np.var(d, ddof=1)
    lrv    = gamma0
    for k in range(1, h_lag + 1):
        w      = 1 - k / (h_lag + 1)
        gamma_k = np.cov(d[k:], d[:-k])[0, 1]
        lrv    += 2 * w * gamma_k
    lrv = max(lrv, 1e-20)
    dm  = mu / np.sqrt(lrv / n)
    pv  = 2 * (1 - stats.norm.cdf(abs(dm)))
    return float(dm), float(pv)


# ── Main loop ─────────────────────────────────────────────────────────────────
rows = []

for col in test_cols:
    idx = col.replace("_ret", "")

    for h in HORIZONS:

        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            continue

        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"], index_col="Date").sort_index()
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"], index_col="Date").sort_index()

        if not {"sigma_hat","r"}.issubset(fc_har.columns): continue
        if "sigma_hat_rp" not in fc_rp.columns:            continue

        fc_har = fc_har.dropna(subset=["sigma_hat","r"])
        fc_rp  = fc_rp.dropna(subset=["sigma_hat_rp"])

        common = fc_har.index.intersection(fc_rp.index)
        if len(common) < 50: continue

        r       = fc_har.loc[common, "r"].values
        sig_har = fc_har.loc[common, "sigma_hat"].values
        sig_rp  = fc_rp.loc[common,  "sigma_hat_rp"].values
        nu      = nu_dict.get((idx, h), 6.0)

        for dist in ["Normal", "Student"]:
            if dist == "Normal":
                c_har = crps_normal(r, sig_har)
                c_rp  = crps_normal(r, sig_rp)
            else:
                c_har = crps_student(r, sig_har, nu)
                c_rp  = crps_student(r, sig_rp,  nu)

            d    = c_har - c_rp          # positive → RP better
            dm_s, dm_p = dm_test_crps(d, h_lag=max(1, h))
            ss   = float(d.mean() / np.mean(c_har)) if np.mean(c_har) > 0 else np.nan

            rows.append({
                "Index":       idx,
                "h":           h,
                "Dist":        dist,
                "CRPS_HAR":    round(float(np.mean(c_har)), 6),
                "CRPS_RP":     round(float(np.mean(c_rp)),  6),
                "Delta_CRPS":  round(float(d.mean()),       6),   # HAR - RP (>0 means RP better)
                "Skill_Score": round(ss * 100, 3),                # % improvement
                "DM_stat":     round(dm_s, 3),
                "DM_pval":     round(dm_p, 4),
                "RP_wins":     d.mean() > 0,
                "RP_sig":      (d.mean() > 0) and (dm_p < 0.05)
            })

crps_df = pd.DataFrame(rows)
crps_df.to_csv(f"{OUT_DIR}/crps_results.csv", index=False)

# ── Summary table ─────────────────────────────────────────────────────────────
print("=" * 65)
print("CRPS SUMMARY — RP wins / significant wins (out of 20 cases)")
print("=" * 65)
for dist in ["Normal", "Student"]:
    sub = crps_df[crps_df["Dist"] == dist]
    wins = sub["RP_wins"].sum()
    sigs = sub["RP_sig"].sum()
    avg_ss = sub["Skill_Score"].mean()
    print(f"  {dist:8s}: RP wins {wins}/20  |  sig (5%) {sigs}/20  |  avg skill {avg_ss:+.3f}%")

print()
print("Skill scores by horizon (Normal):")
piv = crps_df[crps_df["Dist"]=="Normal"].pivot_table(
    index="Index", columns="h", values="Skill_Score"
)
print(piv.round(3).to_string())

# ── Figure 1: Skill score heatmap (Normal) ───────────────────────────────────
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("CRPS Skill Score (%) — RP_Model vs HAR  [positive = RP better]", fontsize=12)

for ax, dist in zip(axes, ["Normal", "Student"]):
    sub = crps_df[crps_df["Dist"] == dist]
    piv = sub.pivot_table(index="Index", columns="h", values="Skill_Score")
    vmax = max(abs(piv.values[~np.isnan(piv.values)]).max(), 0.01)
    im   = ax.imshow(piv.values, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f"h={h}" for h in piv.columns])
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index)
    ax.set_title(f"{dist} distribution", fontsize=11)
    plt.colorbar(im, ax=ax, label="Skill Score (%)")
    # annotate with significance stars
    sig_piv = sub.pivot_table(index="Index", columns="h", values="RP_sig")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            val = piv.values[i, j]
            star = "*" if (not np.isnan(sig_piv.values[i,j]) and sig_piv.values[i,j]) else ""
            ax.text(j, i, f"{val:.2f}{star}", ha="center", va="center",
                    fontsize=8, color="black")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/crps_skill_heatmap.png", dpi=150)
plt.show()

# ── Figure 2: Mean CRPS by horizon — bar chart ───────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(14, 5))
fig2.suptitle("Mean CRPS by Horizon — HAR vs RP_Model", fontsize=12)

for ax, dist in zip(axes2, ["Normal", "Student"]):
    sub   = crps_df[crps_df["Dist"] == dist]
    agg   = sub.groupby("h")[["CRPS_HAR","CRPS_RP"]].mean().reset_index()
    x     = np.arange(len(agg))
    width = 0.35
    ax.bar(x - width/2, agg["CRPS_HAR"], width, label="HAR",      color="#4C72B0", alpha=0.85)
    ax.bar(x + width/2, agg["CRPS_RP"],  width, label="RP_Model", color="#55A868", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels([f"h={h}" for h in agg["h"]])
    ax.set_title(f"{dist} distribution", fontsize=11)
    ax.set_ylabel("Mean CRPS (avg over indices)")
    ax.legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/crps_by_horizon.png", dpi=150)
plt.show()

# ── Figure 3: DM statistics bubble plot ──────────────────────────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle("DM test on CRPS differential (HAR − RP)  — positive = RP better", fontsize=12)

for ax, dist in zip(axes3, ["Normal", "Student"]):
    sub = crps_df[crps_df["Dist"] == dist]
    piv = sub.pivot_table(index="Index", columns="h", values="DM_stat")
    vmax = max(abs(piv.values[~np.isnan(piv.values)]).max(), 0.01)
    im   = ax.imshow(piv.values, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f"h={h}" for h in piv.columns])
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index)
    ax.set_title(f"{dist} — DM statistic", fontsize=11)
    plt.colorbar(im, ax=ax, label="DM stat")
    pval_piv = sub.pivot_table(index="Index", columns="h", values="DM_pval")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            val  = piv.values[i, j]
            pval = pval_piv.values[i, j]
            star = "*" if (not np.isnan(pval) and pval < 0.05) else ""
            ax.text(j, i, f"{val:.2f}{star}", ha="center", va="center",
                    fontsize=8, color="black")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/crps_dm_heatmap.png", dpi=150)
plt.show()

print(f"\nSaved to {OUT_DIR}:")
print("  crps_results.csv")
print("  crps_skill_heatmap.png")
print("  crps_by_horizon.png")
print("  crps_dm_heatmap.png")


The CRPS results provide a clearer and more structured picture than both the PIT test and the point-forecast metrics.

**Under the Normal distribution**, RP_Model wins in 11 out of 20 cases but the average skill score is slightly negative (−0.081%), driven by a consistent HAR advantage at the daily horizon (h=1): skill scores of −0.47% (FTSE), −0.63% (GDAXI), −0.43% (N225), −0.39% (GSPC). At h=1, the rough component X_R has a near-zero autoregressive coefficient (φ_R ≈ 0.04), behaving essentially as white noise. Under the Normal assumption, the slightly wider predictive distribution produced by RP_Model is penalised by the CRPS when actual returns are moderate — sharpness matters, and HAR is sharper at the daily scale. At longer horizons (h=5, 20, 60) the picture reverses, with RP posting small but positive skill scores.

**Under the Student-t distribution**, the result is unambiguous: RP_Model wins in 15 out of 20 cases, all 15 statistically significant at the 5% level (DM test on CRPS differentials), with an average skill score of +0.068%. The Student-t assumption corrects for the fat-tail misspecification identified by the PIT test. Once tails are adequately modelled, RP_Model's superior volatility forecast translates directly into a better probabilistic forecast across all horizons and indices — including h=1.

The contrast between the two distributional assumptions carries an important methodological message: **the ranking of probabilistic forecasting models is not invariant to the assumed distribution**. Under Normal, the model with more noise at short horizons (RP) is penalised. Under Student-t, the model with better conditional variance (RP) is rewarded. This result reinforces the conclusion that RP_Model should be evaluated under fat-tailed distributions, consistent with the well-documented non-Gaussianity of daily equity returns.

In [ ]:
# CELL 32 — Kurtosis of standardised residuals
#
# If a volatility model correctly captures the conditional variance, the
# standardised residuals  z_t = r_t / sigma_hat_t  should have kurtosis
# close to the theoretical value of the assumed distribution:
#   Normal   : excess kurtosis = 0  (kurtosis = 3)
#   Student-t: excess kurtosis = 6 / (nu - 4)  for nu > 4
#
# Residual excess kurtosis above the theoretical benchmark indicates that
# the model leaves unexplained tail risk.  A lower excess kurtosis for
# RP_Model relative to HAR implies that the rough/persistent decomposition
# absorbs a larger share of the conditional variance, reducing tail residuals.
#
# Tests performed:
#   1) Excess kurtosis of z_t for HAR and RP (all indices and horizons)
#   2) Kurtosis reduction: Delta_kurt = kurt(z_HAR) - kurt(z_RP)  [>0 → RP better]
#   3) Jarque-Bera test on z_t (H0: normality of residuals)
#   4) Summary heatmap and bar chart

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FC_DIR_HAR  = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP   = f"{BASE_DIR}/forecast_test_RP"
PARAMS_FILE = f"{BASE_DIR}/model_params_HAR.csv"
HORIZONS    = [1, 5, 20, 60]
OUT_DIR     = f"{BASE_DIR}/kurtosis_results"
os.makedirs(OUT_DIR, exist_ok=True)

params_df = pd.read_csv(PARAMS_FILE)
nu_dict   = {
    (row["Indice"], int(row["Horizon_h"])): float(row["nu_student_t_for_MC"])
    for _, row in params_df.iterrows()
}

test_cols = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv", nrows=0
).columns.tolist()

rows = []

for col in test_cols:
    idx = col.replace("_ret", "")

    for h in HORIZONS:

        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            continue

        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"], index_col="Date").sort_index()
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"], index_col="Date").sort_index()

        if not {"sigma_hat","r"}.issubset(fc_har.columns): continue
        if "sigma_hat_rp" not in fc_rp.columns:            continue

        fc_har = fc_har.dropna(subset=["sigma_hat","r"])
        fc_rp  = fc_rp.dropna(subset=["sigma_hat_rp"])

        common = fc_har.index.intersection(fc_rp.index)
        if len(common) < 50: continue

        r       = fc_har.loc[common, "r"].values
        sig_har = fc_har.loc[common, "sigma_hat"].values.clip(min=EPS)
        sig_rp  = fc_rp.loc[common,  "sigma_hat_rp"].values.clip(min=EPS)
        nu      = nu_dict.get((idx, h), 6.0)

        # Standardised residuals
        z_har = r / sig_har
        z_rp  = r / sig_rp

        # Excess kurtosis (scipy uses Fisher definition: Normal = 0)
        kurt_har = float(stats.kurtosis(z_har, fisher=True))
        kurt_rp  = float(stats.kurtosis(z_rp,  fisher=True))

        # Theoretical excess kurtosis under Student-t
        theo_kurt = 6.0 / (nu - 4.0) if nu > 4.0 else np.nan

        # Jarque-Bera test on z_t
        jb_har, pjb_har = stats.jarque_bera(z_har)[:2]
        jb_rp,  pjb_rp  = stats.jarque_bera(z_rp)[:2]

        rows.append({
            "Index":         idx,
            "h":             h,
            "nu":            round(nu, 2),
            "Theo_ExKurt":   round(theo_kurt, 3) if not np.isnan(theo_kurt) else np.nan,
            "ExKurt_HAR":    round(kurt_har, 3),
            "ExKurt_RP":     round(kurt_rp,  3),
            "Delta_ExKurt":  round(kurt_har - kurt_rp, 3),   # >0 → RP has lower kurtosis
            "RP_lower_kurt": kurt_rp < kurt_har,
            "JB_HAR":        round(float(jb_har),  2),
            "pJB_HAR":       round(float(pjb_har), 4),
            "JB_RP":         round(float(jb_rp),   2),
            "pJB_RP":        round(float(pjb_rp),  4),
        })

kurt_df = pd.DataFrame(rows)
kurt_df.to_csv(f"{OUT_DIR}/kurtosis_results.csv", index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("EXCESS KURTOSIS OF STANDARDISED RESIDUALS z_t = r_t / sigma_hat_t")
print("=" * 60)
print(f"RP has lower excess kurtosis than HAR: "
      f"{kurt_df['RP_lower_kurt'].sum()}/20 cases")
print(f"Mean Delta_ExKurt (HAR - RP): "
      f"{kurt_df['Delta_ExKurt'].mean():.3f}")
print()
print("By index and horizon:")
piv_har = kurt_df.pivot_table(index="Index", columns="h", values="ExKurt_HAR").round(2)
piv_rp  = kurt_df.pivot_table(index="Index", columns="h", values="ExKurt_RP").round(2)
piv_dlt = kurt_df.pivot_table(index="Index", columns="h", values="Delta_ExKurt").round(2)
print("  HAR excess kurtosis:")
print(piv_har.to_string())
print()
print("  RP excess kurtosis:")
print(piv_rp.to_string())
print()
print("  Delta (HAR - RP), positive = RP better:")
print(piv_dlt.to_string())

theo = kurt_df[["Index","h","nu","Theo_ExKurt"]].drop_duplicates()
print()
print("  Theoretical excess kurtosis (Student-t benchmark):")
print(theo.pivot_table(index="Index", columns="h", values="Theo_ExKurt").round(2).to_string())

# ── Figure 1: Heatmap Delta excess kurtosis ───────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Excess Kurtosis of Standardised Residuals z_t = r_t / σ̂_t", fontsize=12)

for ax, (model, piv) in zip(axes, [("HAR", piv_har), ("RP_Model", piv_rp)]):
    vmax = max(abs(piv.values[~np.isnan(piv.values)]).max(), 0.1)
    im   = ax.imshow(piv.values, cmap="YlOrRd", vmin=0, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(piv.columns)))
    ax.set_xticklabels([f"h={h}" for h in piv.columns])
    ax.set_yticks(range(len(piv.index)))
    ax.set_yticklabels(piv.index)
    ax.set_title(f"{model}", fontsize=11)
    plt.colorbar(im, ax=ax, label="Excess Kurtosis")
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            ax.text(j, i, f"{piv.values[i,j]:.2f}", ha="center", va="center",
                    fontsize=9, color="black")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/kurtosis_heatmap.png", dpi=150)
plt.show()

# ── Figure 2: Delta kurtosis heatmap (HAR - RP) ───────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 5))
vmax2 = max(abs(piv_dlt.values[~np.isnan(piv_dlt.values)]).max(), 0.01)
im2   = ax2.imshow(piv_dlt.values, cmap="RdYlGn", vmin=-vmax2, vmax=vmax2, aspect="auto")
ax2.set_xticks(range(len(piv_dlt.columns)))
ax2.set_xticklabels([f"h={h}" for h in piv_dlt.columns])
ax2.set_yticks(range(len(piv_dlt.index)))
ax2.set_yticklabels(piv_dlt.index)
ax2.set_title("Delta Excess Kurtosis (HAR − RP)  [green = RP has lower kurtosis]", fontsize=11)
plt.colorbar(im2, ax=ax2, label="Δ Excess Kurtosis")
for i in range(piv_dlt.shape[0]):
    for j in range(piv_dlt.shape[1]):
        ax2.text(j, i, f"{piv_dlt.values[i,j]:.2f}", ha="center", va="center",
                 fontsize=9, color="black")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/kurtosis_delta_heatmap.png", dpi=150)
plt.show()

# ── Figure 3: Mean excess kurtosis by horizon ─────────────────────────────────
agg = kurt_df.groupby("h")[["ExKurt_HAR","ExKurt_RP","Theo_ExKurt"]].mean().reset_index()
x     = np.arange(len(agg))
width = 0.28

fig3, ax3 = plt.subplots(figsize=(9, 5))
ax3.bar(x - width, agg["ExKurt_HAR"],   width, label="HAR",       color="#4C72B0", alpha=0.85)
ax3.bar(x,         agg["ExKurt_RP"],    width, label="RP_Model",  color="#55A868", alpha=0.85)
ax3.bar(x + width, agg["Theo_ExKurt"],  width, label="Theoretical (Student-t)", color="#C44E52", alpha=0.65)
ax3.set_xticks(x)
ax3.set_xticklabels([f"h={h}" for h in agg["h"]])
ax3.set_ylabel("Mean Excess Kurtosis")
ax3.set_title("Mean Excess Kurtosis of z_t by Horizon — HAR vs RP vs Theoretical", fontsize=11)
ax3.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/kurtosis_by_horizon.png", dpi=150)
plt.show()

print(f"\nSaved to {OUT_DIR}:")
print("  kurtosis_results.csv")
print("  kurtosis_heatmap.png")
print("  kurtosis_delta_heatmap.png")
print("  kurtosis_by_horizon.png")


The kurtosis analysis of standardised residuals z_t = r_t / σ̂_t completes the distributional diagnostics initiated by the PIT test and the CRPS decomposition, and provides a direct explanation for the VaR failure documented in Cell 26.

**Residual excess kurtosis is large for both models** — ranging from 2.72 (HSI, h=20) to 18.95 (FTSE, h=60) — and is systematically above the Student-t theoretical benchmark of 1.0–3.4. The difference between HAR and RP is negligible (mean Delta = 0.010, RP wins only 10/20 cases), confirming that the two models leave an identical amount of unexplained tail risk in their residuals.

Taken together, the three distributional diagnostics tell a coherent story:

1. **PIT** — both models produce U-shaped histograms, indicating that the assumed conditional distribution (Normal or Student-t) is systematically under-dispersed. The shape is identical for HAR and RP, confirming that the misspecification is distributional, not a function of forecast accuracy.

2. **CRPS** — under the Normal assumption, HAR is slightly better at h=1 because RP's wider σ̂ is penalised for lack of sharpness. Under Student-t, RP wins 15/20 cases with statistical significance, showing that once tails are adequately modelled, RP's superior conditional variance translates into better probabilistic forecasts.

3. **Kurtosis** — residual excess kurtosis of 5–19 against a theoretical benchmark of 1–3 confirms that neither model, nor the Student-t distribution with ν ≈ 7–10, is sufficient to capture the true tail behaviour. The distribution assumed in both models remains too thin-tailed even after accounting for degrees of freedom.

This chain of evidence directly explains the **VaR failure** (Cell 26): violation ratios of ~1.9× at the 95% level and ~4.4× at the 99% level are not a consequence of poor volatility forecasting — both HAR and RP produce quantitatively similar violations. They are a direct consequence of the distributional misspecification: a distribution that underestimates tail thickness will systematically underestimate the true quantiles of returns, regardless of how accurately σ̂_t tracks realised volatility. The success of FHS (96.7% CC pass rate in the Normal period) confirms this: by replacing the parametric tail with an empirical one estimated on standardised residuals, the calibration problem is resolved without changing the volatility model at all.

In [ ]:
# CELL 33 — CRPS decomposition: Reliability and Resolution
#
# Following Gneiting & Raftery (2007) and Hersbach (2000), the CRPS is
# decomposed into three components:
#
#   CRPS = Reliability - Resolution + Uncertainty
#
# Empirical decomposition via the threshold-weighted approach:
# For each threshold theta in a grid over the observed return distribution:
#   o_t(theta) = 1{r_t <= theta}          (binary outcome)
#   p_t(theta) = F(theta | sigma_hat_t)   (predicted probability)
#   CRPS = integral (p_t - o_t)^2 dtheta
#
# Decomposed as (Hersbach 2000):
#   Reliability = integral E[(p_t - o_t)^2 | p_t] dtheta   (calibration error)
#   Resolution  = integral Var(o_t | p_t)  dtheta           (sharpness)
#   Uncertainty = integral pi(1 - pi) dtheta                (marginal variance)
#
# Expectation: RP wins on Resolution (sharper sigma_hat), loses on Reliability
# (same distributional misspecification as HAR, as shown by PIT and kurtosis).

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

FC_DIR_HAR  = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP   = f"{BASE_DIR}/forecast_test_RP"
PARAMS_FILE = f"{BASE_DIR}/model_params_HAR.csv"
HORIZONS    = [1, 5, 20, 60]
OUT_DIR     = f"{BASE_DIR}/crps_decomp_results"
os.makedirs(OUT_DIR, exist_ok=True)

params_df = pd.read_csv(PARAMS_FILE)
nu_dict   = {
    (row["Indice"], int(row["Horizon_h"])): float(row["nu_student_t_for_MC"])
    for _, row in params_df.iterrows()
}

test_cols = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv", nrows=0
).columns.tolist()

# ── CRPS decomposition (Hersbach 2000) ────────────────────────────────────────
def crps_decompose(r, prob_func, n_thresholds=200):
    """
    Decompose CRPS into Reliability, Resolution, Uncertainty.

    Parameters
    ----------
    r         : array of observed returns (n,)
    prob_func : callable — prob_func(theta) returns array (n,) of P(R <= theta | sigma_hat)
    n_thresholds : number of quadrature points

    Returns
    -------
    crps_total, reliability, resolution, uncertainty
    """
    # Threshold grid: from 1st to 99th percentile of observations
    thetas = np.linspace(np.percentile(r, 0.5), np.percentile(r, 99.5), n_thresholds)
    dtheta = thetas[1] - thetas[0]

    rel_acc  = 0.0
    res_acc  = 0.0
    unc_acc  = 0.0

    for theta in thetas:
        o_t = (r <= theta).astype(float)       # binary outcome
        p_t = prob_func(theta)                  # predicted probability (n,)

        pi  = o_t.mean()                        # marginal climatology

        # Reliability: mean squared calibration error
        # Approximated by binning p_t into 10 bins
        bins    = np.linspace(0, 1, 11)
        rel_bin = 0.0
        for b in range(len(bins) - 1):
            mask = (p_t >= bins[b]) & (p_t < bins[b+1])
            if mask.sum() > 0:
                p_bar = p_t[mask].mean()
                o_bar = o_t[mask].mean()
                rel_bin += mask.mean() * (p_bar - o_bar) ** 2
        rel_acc += rel_bin * dtheta

        # Resolution: variance of o_t conditional on p_t — approximated by
        # total variance of o_t minus the within-bin residual
        res_bin = 0.0
        for b in range(len(bins) - 1):
            mask = (p_t >= bins[b]) & (p_t < bins[b+1])
            if mask.sum() > 0:
                o_bar = o_t[mask].mean()
                res_bin += mask.mean() * (o_bar - pi) ** 2
        res_acc += res_bin * dtheta

        # Uncertainty: climatological CRPS = pi*(1-pi)
        unc_acc += pi * (1.0 - pi) * dtheta

    crps_total = rel_acc - res_acc + unc_acc
    return crps_total, rel_acc, res_acc, unc_acc


# ── Main loop ─────────────────────────────────────────────────────────────────
rows = []

for col in test_cols:
    idx = col.replace("_ret", "")

    for h in HORIZONS:

        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            continue

        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"], index_col="Date").sort_index()
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"], index_col="Date").sort_index()

        if not {"sigma_hat","r"}.issubset(fc_har.columns): continue
        if "sigma_hat_rp" not in fc_rp.columns:            continue

        fc_har = fc_har.dropna(subset=["sigma_hat","r"])
        fc_rp  = fc_rp.dropna(subset=["sigma_hat_rp"])

        common = fc_har.index.intersection(fc_rp.index)
        if len(common) < 50: continue

        r       = fc_har.loc[common, "r"].values
        sig_har = fc_har.loc[common, "sigma_hat"].values.clip(min=EPS)
        sig_rp  = fc_rp.loc[common,  "sigma_hat_rp"].values.clip(min=EPS)
        nu      = nu_dict.get((idx, h), 6.0)
        scale_t = np.sqrt((nu - 2) / nu) if nu > 2 else 1.0

        for dist in ["Normal", "Student"]:

            if dist == "Normal":
                def prob_har(theta, s=sig_har):
                    return stats.norm.cdf(theta / s)
                def prob_rp(theta, s=sig_rp):
                    return stats.norm.cdf(theta / s)
            else:
                def prob_har(theta, s=sig_har, nu_=nu, sc=scale_t):
                    return stats.t.cdf(theta / (s * sc), df=nu_)
                def prob_rp(theta, s=sig_rp, nu_=nu, sc=scale_t):
                    return stats.t.cdf(theta / (s * sc), df=nu_)

            crps_h, rel_h, res_h, unc_h = crps_decompose(r, prob_har)
            crps_r, rel_r, res_r, unc_r = crps_decompose(r, prob_rp)

            rows.append({
                "Index":        idx,
                "h":            h,
                "Dist":         dist,
                # HAR
                "CRPS_HAR":     round(crps_h, 6),
                "Rel_HAR":      round(rel_h,  6),
                "Res_HAR":      round(res_h,  6),
                "Unc_HAR":      round(unc_h,  6),
                # RP
                "CRPS_RP":      round(crps_r, 6),
                "Rel_RP":       round(rel_r,  6),
                "Res_RP":       round(res_r,  6),
                "Unc_RP":       round(unc_r,  6),
                # Differences (positive = RP better)
                "Delta_CRPS":   round(crps_h - crps_r, 6),
                "Delta_Rel":    round(rel_h  - rel_r,  6),   # >0 → RP more reliable
                "Delta_Res":    round(res_r  - res_h,  6),   # >0 → RP has more resolution
            })

decomp_df = pd.DataFrame(rows)
decomp_df.to_csv(f"{OUT_DIR}/crps_decomp_results.csv", index=False)

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 65)
print("CRPS DECOMPOSITION — Reliability, Resolution, Uncertainty")
print("=" * 65)
for dist in ["Normal", "Student"]:
    sub = decomp_df[decomp_df["Dist"] == dist]
    print(f"\n  {dist}:")
    print(f"    RP more reliable  (Delta_Rel > 0): {(sub['Delta_Rel'] > 0).sum()}/20")
    print(f"    RP more resolute  (Delta_Res > 0): {(sub['Delta_Res'] > 0).sum()}/20")
    print(f"    Mean Delta_Rel  : {sub['Delta_Rel'].mean():+.6f}")
    print(f"    Mean Delta_Res  : {sub['Delta_Res'].mean():+.6f}")

print("\nMean components by distribution:")
for dist in ["Normal", "Student"]:
    sub = decomp_df[decomp_df["Dist"] == dist]
    print(f"\n  {dist}:")
    print(f"    HAR — Rel: {sub['Rel_HAR'].mean():.5f}  Res: {sub['Res_HAR'].mean():.5f}  Unc: {sub['Unc_HAR'].mean():.5f}")
    print(f"    RP  — Rel: {sub['Rel_RP'].mean():.5f}  Res: {sub['Res_RP'].mean():.5f}  Unc: {sub['Unc_RP'].mean():.5f}")

# ── Figure 1: Bar chart — mean components HAR vs RP ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("CRPS Decomposition: Reliability and Resolution — HAR vs RP_Model", fontsize=12)

components = ["Rel", "Res"]
labels     = ["Reliability (lower = better)", "Resolution (higher = better)"]
colors_har = ["#4C72B0", "#4C72B0"]
colors_rp  = ["#55A868", "#55A868"]
alphas     = [0.85, 0.85]

for ax, dist in zip(axes, ["Normal", "Student"]):
    sub = decomp_df[decomp_df["Dist"] == dist]
    agg = sub.groupby("h")[
        ["Rel_HAR","Res_HAR","Rel_RP","Res_RP"]
    ].mean().reset_index()

    x     = np.arange(len(agg))
    width = 0.20

    ax.bar(x - 1.5*width, agg["Rel_HAR"], width, label="HAR Reliability",      color="#4C72B0", alpha=0.9)
    ax.bar(x - 0.5*width, agg["Rel_RP"],  width, label="RP Reliability",       color="#55A868", alpha=0.9)
    ax.bar(x + 0.5*width, agg["Res_HAR"], width, label="HAR Resolution",       color="#4C72B0", alpha=0.45, hatch="//")
    ax.bar(x + 1.5*width, agg["Res_RP"],  width, label="RP Resolution",        color="#55A868", alpha=0.45, hatch="//")

    ax.set_xticks(x)
    ax.set_xticklabels([f"h={h}" for h in agg["h"]])
    ax.set_title(f"{dist} distribution", fontsize=11)
    ax.set_ylabel("Component value (avg over indices)")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/crps_decomp_bars.png", dpi=150)
plt.show()

# ── Figure 2: Heatmaps Delta_Rel and Delta_Res ────────────────────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 9))
fig2.suptitle("CRPS Decomposition — RP advantage (+) or disadvantage (−)", fontsize=12)

for col_j, dist in enumerate(["Normal", "Student"]):
    sub = decomp_df[decomp_df["Dist"] == dist]

    for row_i, (metric, title) in enumerate([
        ("Delta_Rel", "Δ Reliability (HAR−RP): positive = RP more reliable"),
        ("Delta_Res", "Δ Resolution  (RP−HAR): positive = RP more resolute"),
    ]):
        ax   = axes2[row_i, col_j]
        piv  = sub.pivot_table(index="Index", columns="h", values=metric)
        vmax = max(abs(piv.values[~np.isnan(piv.values)]).max(), 1e-6)
        im   = ax.imshow(piv.values, cmap="RdYlGn", vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_xticks(range(len(piv.columns)))
        ax.set_xticklabels([f"h={h}" for h in piv.columns])
        ax.set_yticks(range(len(piv.index)))
        ax.set_yticklabels(piv.index)
        ax.set_title(f"{dist} — {title}", fontsize=9)
        plt.colorbar(im, ax=ax)
        for i in range(piv.shape[0]):
            for j in range(piv.shape[1]):
                ax.text(j, i, f"{piv.values[i,j]:.4f}", ha="center", va="center",
                        fontsize=8, color="black")

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/crps_decomp_heatmaps.png", dpi=150)
plt.show()

print(f"\nSaved to {OUT_DIR}:")
print("  crps_decomp_results.csv")
print("  crps_decomp_bars.png")
print("  crps_decomp_heatmaps.png")


In [ ]:
# Plot 3D PIT
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy import stats

FC_DIR_HAR = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP  = f"{BASE_DIR}/forecast_test_RP"
HORIZONS   = [1, 5, 20, 60]
NBINS      = 10
BIN_EDGES  = np.linspace(0, 1, NBINS + 1)
BIN_MIDS   = (BIN_EDGES[:-1] + BIN_EDGES[1:]) / 2

test_cols = pd.read_csv(
    f"{BASE_DIR}/indici_log_returns_test_30.csv", nrows=0
).columns.tolist()

# ── Collect PIT densities by horizon ─────────────────────────────────────────
density_har = np.zeros((len(HORIZONS), NBINS))
density_rp  = np.zeros((len(HORIZONS), NBINS))

for h_idx, h in enumerate(HORIZONS):
    u_har_all, u_rp_all = [], []

    for col in test_cols:
        idx = col.replace("_ret", "")
        fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{h}.csv"
        fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{h}.csv"
        if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
            continue

        fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"],
                              index_col="Date").sort_index()
        fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"],
                              index_col="Date").sort_index()

        if not {"sigma_hat","r"}.issubset(fc_har.columns): continue
        if "sigma_hat_rp" not in fc_rp.columns:            continue

        fc_har = fc_har.dropna(subset=["sigma_hat","r"])
        fc_rp  = fc_rp.dropna(subset=["sigma_hat_rp"])

        common = fc_har.index.intersection(fc_rp.index)
        if len(common) < 50: continue

        r       = fc_har.loc[common, "r"].values
        sig_har = fc_har.loc[common, "sigma_hat"].values.clip(min=EPS)
        sig_rp  = fc_rp.loc[common, "sigma_hat_rp"].values.clip(min=EPS)

        u_har_all.append(stats.norm.cdf(r / sig_har))
        u_rp_all.append(stats.norm.cdf(r / sig_rp))

    if u_har_all:
        u_har = np.concatenate(u_har_all)
        u_rp  = np.concatenate(u_rp_all)
        counts_har, _ = np.histogram(u_har, bins=BIN_EDGES, density=True)
        counts_rp,  _ = np.histogram(u_rp,  bins=BIN_EDGES, density=True)
        density_har[h_idx] = counts_har
        density_rp[h_idx]  = counts_rp

# ── Build mesh ────────────────────────────────────────────────────────────────
X, Y = np.meshgrid(BIN_MIDS, np.arange(len(HORIZONS)))
H_LABELS = [f"h={h}" for h in HORIZONS]

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 7))

for col_idx, (model, density, color) in enumerate([
    ("HAR",      density_har, "#4C72B0"),
    ("RP_Model", density_rp,  "#55A868"),
]):
    ax = fig.add_subplot(1, 2, col_idx + 1, projection="3d")

    # Model surface
    surf = ax.plot_surface(X, Y, density, color=color, alpha=0.75,
                           edgecolor="white", linewidth=0.3)

    # Uniform benchmark at z = 1
    ax.plot_surface(X, Y, np.ones_like(X),
                    color="crimson", alpha=0.18, linewidth=0)
    # Uniform benchmark wireframe (edges only)
    ax.plot_wireframe(X, Y, np.ones_like(X),
                      color="crimson", linewidth=0.6, alpha=0.5)

    ax.set_title(f"{model} — PIT density by horizon\n(red mesh = U(0,1) benchmark)",
                 fontsize=10, pad=10)
    ax.set_xlabel("PIT value  u = Φ(r/σ̂)", labelpad=8, fontsize=9)
    ax.set_ylabel("Horizon", labelpad=8, fontsize=9)
    ax.set_zlabel("Density", labelpad=6, fontsize=9)
    ax.set_yticks(np.arange(len(HORIZONS)))
    ax.set_yticklabels(H_LABELS, fontsize=8)
    ax.set_zlim(0, density_har.max() * 1.15)
    ax.view_init(elev=28, azim=-55)

plt.suptitle(
    "PIT density surface — HAR vs RP_Model (Normal distribution)\n"
    "U-shape = under-dispersed distribution  |  Both models are identical",
    fontsize=11, y=1.01
)
plt.tight_layout()

OUT_3D = f"{BASE_DIR}/pit_3d_surface.png"
plt.savefig(OUT_3D, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_3D}")

# ── Figure 2: Overlay — difference surface (HAR - RP) ────────────────────────
fig2 = plt.figure(figsize=(8, 6))
ax2  = fig2.add_subplot(111, projection="3d")

diff = density_har - density_rp
vmax = np.abs(diff).max()

surf2 = ax2.plot_surface(X, Y, diff,
                          cmap="RdYlGn", alpha=0.85,
                          edgecolor="white", linewidth=0.3,
                          vmin=-vmax, vmax=vmax)
ax2.plot_surface(X, Y, np.zeros_like(X),
                 color="grey", alpha=0.12, linewidth=0)

fig2.colorbar(surf2, ax=ax2, shrink=0.5, label="HAR density − RP density")
ax2.set_title("Difference surface  HAR − RP\n"
"(near-zero = probabilistic equivalence)",
              fontsize=10)
ax2.set_xlabel("PIT value  u = Φ(r/σ̂)", labelpad=8, fontsize=9)
ax2.set_ylabel("Horizon", labelpad=8, fontsize=9)
ax2.set_zlabel("Δ Density", labelpad=6, fontsize=9)
ax2.set_yticks(np.arange(len(HORIZONS)))
ax2.set_yticklabels(H_LABELS, fontsize=8)
ax2.view_init(elev=28, azim=-55)

plt.tight_layout()
OUT_DIFF = f"{BASE_DIR}/pit_3d_difference.png"
plt.savefig(OUT_DIFF, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_DIFF}")

In [ ]:
# CELL 28 — Final diagnostic plots
#
# 1. Dynamic VaR plot: returns, VaR_t and violations over time
#    (HAR_Student vs RP_Student, h=1, CL=95%, one panel per index)
#    Shaded regions: Normal period (white) vs Stress period (light red)
#
# 2. VaR Ratio by period (Normal vs Stress) — bar chart
#    Shows that calibration deteriorates mainly in the stress period
#
# 3. RMSE benchmark: RP_Model vs HAR vs RW vs SMA20 vs EWMA94
#    HAR wins → RP_Model wins updated
#
# 4. Correlation benchmark: same models
#
# 5. Heatmap VaR Ratio — HAR_Student vs RP_Student
#    (replaces old Normal vs Student heatmap)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import os

OUT_DIR_PLOT = f"{BASE_DIR}/grafici_finali_tesi"
os.makedirs(OUT_DIR_PLOT, exist_ok=True)

INDICI   = ["FTSE", "GDAXI", "GSPC", "HSI", "N225"]
HORIZONS = [1, 5, 20, 60]
STRESS_START = pd.Timestamp("2020-01-01")

# Colour palette
COL = {
    "HAR_Normal":  "#4878CF",
    "HAR_Student": "#E24A33",
    "RP_Normal":   "#2ca02c",
    "RP_Student":  "#FF7F0E",
    "HS":          "#9467bd",
    "RP_Model":    "#FF7F0E",
    "HAR_Model":   "#348ABD",
    "RW":          "#988ED5",
    "SMA20":       "#FBC15E",
    "EWMA94":      "#8EBA42"
}

# Load data
var_df   = pd.read_csv(f"{BASE_DIR}/var_christoffersen_all_models.csv")
bench_df = pd.read_csv(f"{BASE_DIR}/benchmark_HAR.csv")

FC_DIR_HAR = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP  = f"{BASE_DIR}/forecast_test_RP"
PARAMS_FILE = f"{BASE_DIR}/model_params_HAR.csv"

params_df = pd.read_csv(PARAMS_FILE)
nu_dict   = {
    (row["Indice"], int(row["Horizon_h"])): float(row["nu_student_t_for_MC"])
    for _, row in params_df.iterrows()
}

# ============================================================
# PLOT 1 — Dynamic VaR: returns + VaR_t + violations over time
#           HAR_Student vs RP_Student, h=1, CL=95%
# ============================================================

from scipy import stats as scipy_stats

H_PLOT  = 1
CL_PLOT = 0.95
ALPHA_PLOT = 1 - CL_PLOT

fig1, axes1 = plt.subplots(len(INDICI), 1,
                            figsize=(16, 4 * len(INDICI)),
                            sharex=False)
fig1.suptitle(
    f"Dynamic VaR (h={H_PLOT}, CL={int(CL_PLOT*100)}%): HAR_Student vs RP_Student\n"
    "Red dots = violations  |  Shaded = Stress period (post Jan 2020)",
    fontsize=13, fontweight="bold"
)

for ax, idx in zip(axes1, INDICI):
    col = f"{idx}_ret"

    fcsv_har = f"{FC_DIR_HAR}/forecast_test_{idx}_h{H_PLOT}.csv"
    fcsv_rp  = f"{FC_DIR_RP}/forecast_test_RP_{idx}_h{H_PLOT}.csv"
    if not os.path.exists(fcsv_har) or not os.path.exists(fcsv_rp):
        ax.set_title(f"{idx} — file not found")
        continue

    fc_har = pd.read_csv(fcsv_har, parse_dates=["Date"],
                          index_col="Date").sort_index()
    fc_rp  = pd.read_csv(fcsv_rp,  parse_dates=["Date"],
                          index_col="Date").sort_index()

    common = fc_har.index.intersection(fc_rp.index)
    r_s    = fc_har.loc[common, "r"]
    nu     = nu_dict.get((idx, H_PLOT), 6.0)
    z_stud = scipy_stats.t.ppf(ALPHA_PLOT, df=nu) * np.sqrt((nu-2)/nu)

    sig_har = fc_har.loc[common, "sigma_hat"]
    sig_rp  = fc_rp.loc[common,  "sigma_hat_rp"]

    var_har = sig_har * abs(z_stud)
    var_rp  = sig_rp  * abs(z_stud)

    hits_har = r_s < -var_har
    hits_rp  = r_s < -var_rp

    # Plot returns
    ax.plot(common, r_s.values, lw=0.5, color="gray",
            alpha=0.6, label="Returns")

    # Plot VaR lines
    ax.plot(common, -var_har.values, lw=1.2, color=COL["HAR_Student"],
            alpha=0.9, label="VaR HAR_Student")
    ax.plot(common, -var_rp.values,  lw=1.2, color=COL["RP_Student"],
            ls="--", alpha=0.9, label="VaR RP_Student")

    # Violations
    ax.scatter(common[hits_har], r_s[hits_har].values,
               color=COL["HAR_Student"], s=12, zorder=5, alpha=0.7)
    ax.scatter(common[hits_rp],  r_s[hits_rp].values,
               color=COL["RP_Student"],  s=12, zorder=5,
               marker="^", alpha=0.7)

    # Shade stress period
    stress_start_actual = max(STRESS_START, common[0])
    ax.axvspan(stress_start_actual, common[-1],
               color="red", alpha=0.05, label="Stress period")
    ax.axvline(STRESS_START, color="darkred",
               lw=1.0, ls=":", alpha=0.7)

    # Annotation: violation rates
    n_total = len(r_s)
    n_norm  = (common < STRESS_START).sum()
    n_str   = (common >= STRESS_START).sum()

    viol_har_n = hits_har[common < STRESS_START].sum()
    viol_rp_n  = hits_rp[common  < STRESS_START].sum()
    viol_har_s = hits_har[common >= STRESS_START].sum()
    viol_rp_s  = hits_rp[common  >= STRESS_START].sum()

    textstr = (
        f"Normal  HAR:{viol_har_n/max(n_norm,1)*100:.1f}%  "
        f"RP:{viol_rp_n/max(n_norm,1)*100:.1f}%  (theo {ALPHA_PLOT*100:.0f}%)\n"
        f"Stress  HAR:{viol_har_s/max(n_str,1)*100:.1f}%  "
        f"RP:{viol_rp_s/max(n_str,1)*100:.1f}%  (theo {ALPHA_PLOT*100:.0f}%)"
    )
    ax.text(0.01, 0.97, textstr, transform=ax.transAxes,
            fontsize=7.5, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.3",
                      facecolor="white", alpha=0.8))

    ax.set_title(f"{idx}", fontsize=10, fontweight="bold")
    ax.set_ylabel("Log-return", fontsize=8)
    ax.legend(fontsize=7, loc="lower right", ncol=4)
    ax.grid(True, alpha=0.2)

fig1.tight_layout()
out1 = f"{OUT_DIR_PLOT}/grafico1_dynamic_var.png"
fig1.savefig(out1, dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig1)
print(f"Saved: grafico1_dynamic_var.png")


# ============================================================
# PLOT 2 — VaR Ratio by period: Normal vs Stress
#           One subplot per confidence level
#           Bars: HAR_Student, RP_Student, HS
# ============================================================

SPECS_PLOT = ["HAR_Student", "RP_Student", "HS"]
CLS_PLOT   = [0.950, 0.975, 0.990]
cl_names   = {0.950: "95%", 0.975: "97.5%", 0.990: "99%"}
bar_w      = 0.25
x3         = np.arange(len(SPECS_PLOT))

fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5), sharey=False)
fig2.suptitle(
    "Mean VaR Ratio by confidence level: Normal vs Stress period\n"
    "(ratio = 1.0 is perfectly calibrated)",
    fontsize=13, fontweight="bold"
)

for col_idx, cl in enumerate(CLS_PLOT):
    ax = axes2[col_idx]

    for j, (periodo, color, hatch) in enumerate([
        ("Normal", "#2196F3", ""),
        ("Stress", "#F44336", "//")
    ]):
        sub = var_df[
            (var_df["Periodo"]    == periodo) &
            (var_df["Confidence"] == cl)
        ]
        vals = []
        for spec in SPECS_PLOT:
            s = sub[sub["Spec"] == spec]["Ratio_obs_teo"]
            vals.append(float(s.mean()) if len(s) > 0 else np.nan)

        offset = (j - 0.5) * bar_w
        bars = ax.bar(x3 + offset, vals, bar_w,
                      color=color, alpha=0.85, hatch=hatch,
                      label=periodo, edgecolor="white")

        # Value labels on bars
        for bar, val in zip(bars, vals):
            if np.isfinite(val):
                ax.text(bar.get_x() + bar.get_width()/2,
                        bar.get_height() + 0.05,
                        f"{val:.2f}", ha="center", va="bottom",
                        fontsize=8, fontweight="bold")

    ax.axhline(1.0, color="black", lw=1.8, ls="--",
               label="Perfect calibration (ratio=1)")
    ax.axhspan(0.7, 1.3, color="green", alpha=0.07)

    ax.set_title(f"Confidence {cl_names[cl]}", fontsize=10, fontweight="bold")
    ax.set_xticks(x3)
    ax.set_xticklabels(SPECS_PLOT, fontsize=9)
    ax.set_ylabel("Mean Ratio obs/theoretical" if col_idx == 0 else "")
    ax.grid(True, alpha=0.2, axis="y")
    ax.legend(fontsize=9)

fig2.tight_layout()
out2 = f"{OUT_DIR_PLOT}/grafico2_var_ratio_period.png"
fig2.savefig(out2, dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig2)
print(f"Saved: grafico2_var_ratio_period.png")


# ============================================================
# PLOT 3 — RMSE benchmark: RP_Model vs HAR vs RW vs SMA20 vs EWMA94
# ============================================================

MODELS_B = ["RP_Model", "HAR_Model", "RW", "SMA20", "EWMA94"]
mw       = 0.15
x4       = np.arange(len(INDICI))

fig3, axes3 = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
axes3 = axes3.flatten()
fig3.suptitle(
    "RMSE log-sigma on test set: RP_Model vs benchmarks\n"
    "(lower bar = better)",
    fontsize=13, fontweight="bold"
)

for i, h in enumerate(HORIZONS):
    ax  = axes3[i]
    sub = bench_df[bench_df["Horizon_h"] == h]

    for j, m in enumerate(MODELS_B):
        vals = [
            float(sub[(sub["Indice"]==idx) & (sub["Model"]==m)]["RMSE_logsigma"].values[0])
            if len(sub[(sub["Indice"]==idx) & (sub["Model"]==m)]) > 0 else np.nan
            for idx in INDICI
        ]
        off  = (j - len(MODELS_B)/2 + 0.5) * mw
        bars = ax.bar(x4 + off, vals, mw,
                      color=COL[m], alpha=0.88,
                      edgecolor="white", linewidth=0.4)
        if m == "RP_Model":
            for b in bars:
                b.set_edgecolor("black")
                b.set_linewidth(1.5)

    ax.set_title(f"h = {h}", fontsize=10, fontweight="bold")
    ax.set_xticks(x4)
    ax.set_xticklabels(INDICI, fontsize=9)
    ax.set_ylabel("RMSE log-sigma" if i % 2 == 0 else "")
    ax.grid(True, alpha=0.2, axis="y")

    # Count RP_Model wins
    rp_wins = sum(
        1 for idx in INDICI
        if (len(sub[(sub["Indice"]==idx) & (sub["Model"]=="RP_Model")]) > 0 and
            sub[(sub["Indice"]==idx) & (sub["Model"]=="RP_Model")]["RMSE_logsigma"].values[0] ==
            min(sub[sub["Indice"]==idx]["RMSE_logsigma"].values))
    )
    ax.text(0.98, 0.97, f"RP wins: {rp_wins}/5",
            transform=ax.transAxes, ha="right", va="top",
            fontsize=9, fontweight="bold", color=COL["RP_Model"],
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                      edgecolor=COL["RP_Model"], alpha=0.8))

handles3 = [plt.Rectangle((0,0),1,1, color=COL[m], label=m)
            for m in MODELS_B]
fig3.legend(handles=handles3, loc="lower center", ncol=5,
            fontsize=10, bbox_to_anchor=(0.5, -0.03))
fig3.tight_layout()
out3 = f"{OUT_DIR_PLOT}/grafico3_rmse_benchmark.png"
fig3.savefig(out3, dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig3)
print(f"Saved: grafico3_rmse_benchmark.png")


# ============================================================
# PLOT 4 — Correlation benchmark: same models
# ============================================================

fig4, axes4 = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
axes4 = axes4.flatten()
fig4.suptitle(
    "Predicted vs observed log-sigma correlation (test set)",
    fontsize=13, fontweight="bold"
)

for i, h in enumerate(HORIZONS):
    ax  = axes4[i]
    sub = bench_df[bench_df["Horizon_h"] == h]

    for j, m in enumerate(MODELS_B):
        vals = [
            float(sub[(sub["Indice"]==idx) & (sub["Model"]==m)]["Corr"].values[0])
            if len(sub[(sub["Indice"]==idx) & (sub["Model"]==m)]) > 0 else np.nan
            for idx in INDICI
        ]
        off  = (j - len(MODELS_B)/2 + 0.5) * mw
        bars = ax.bar(x4 + off, vals, mw,
                      color=COL[m], alpha=0.88,
                      edgecolor="white", linewidth=0.4)
        if m == "RP_Model":
            for b in bars:
                b.set_edgecolor("black")
                b.set_linewidth(1.5)

    ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_title(f"h = {h}", fontsize=10, fontweight="bold")
    ax.set_xticks(x4)
    ax.set_xticklabels(INDICI, fontsize=9)
    ax.set_ylabel("Correlation" if i % 2 == 0 else "")
    ax.set_ylim(-0.1, 1.05)
    ax.grid(True, alpha=0.2, axis="y")

fig4.legend(handles=handles3, loc="lower center", ncol=5,
            fontsize=10, bbox_to_anchor=(0.5, -0.03))
fig4.tight_layout()
out4 = f"{OUT_DIR_PLOT}/grafico4_corr_benchmark.png"
fig4.savefig(out4, dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig4)
print(f"Saved: grafico4_corr_benchmark.png")


# ============================================================
# PLOT 5 — Heatmap VaR Ratio: HAR_Student vs RP_Student
#           Rows: Indice x Horizon_h
#           Columns: 3 confidence levels
#           Two panels: Full period | Normal period
# ============================================================

cl_cols  = [0.950, 0.975, 0.990]
cl_names_heat = ["95%", "97.5%", "99%"]
SPECS_HEAT = ["HAR_Student", "RP_Student"]
PERIODI_HEAT = ["Full", "Normal"]

fig5, axes5 = plt.subplots(2, 2, figsize=(18, 14))
fig5.suptitle(
    "VaR Ratio (observed / theoretical)\n"
    "Top row: Full test set  |  Bottom row: Normal period (pre-2020)\n"
    "Left: HAR_Student  |  Right: RP_Student",
    fontsize=12, fontweight="bold"
)

row_labels = [f"{idx} h={h}" for idx in INDICI for h in HORIZONS]

for row_i, periodo in enumerate(PERIODI_HEAT):
    sub_p = var_df[var_df["Periodo"] == periodo]

    for col_i, spec in enumerate(SPECS_HEAT):
        ax  = axes5[row_i, col_i]
        sub = sub_p[sub_p["Spec"] == spec]

        matrix = []
        for idx in INDICI:
            for h in HORIZONS:
                row_vals = []
                for cl in cl_cols:
                    r = sub[
                        (sub["Indice"]     == idx) &
                        (sub["Horizon_h"]  == h)   &
                        (sub["Confidence"] == cl)
                    ]
                    row_vals.append(
                        float(r["Ratio_obs_teo"].values[0])
                        if len(r) > 0 else np.nan
                    )
                matrix.append(row_vals)

        mat  = np.array(matrix)
        norm = mcolors.TwoSlopeNorm(vmin=0.5, vcenter=1.0, vmax=5.0)
        im   = ax.imshow(mat, cmap=plt.cm.RdYlGn_r,
                          norm=norm, aspect="auto")

        ax.set_xticks(range(len(cl_names_heat)))
        ax.set_xticklabels(cl_names_heat, fontsize=9)
        ax.set_yticks(range(len(row_labels)))
        ax.set_yticklabels(row_labels, fontsize=7.5)
        ax.set_title(f"{periodo} — {spec}",
                     fontsize=10, fontweight="bold")

        for ii in range(mat.shape[0]):
            for jj in range(mat.shape[1]):
                val = mat[ii, jj]
                if np.isfinite(val):
                    color = "white" if val > 3.0 or val < 0.6 else "black"
                    ax.text(jj, ii, f"{val:.2f}",
                            ha="center", va="center",
                            fontsize=7, color=color,
                            fontweight="bold")

        plt.colorbar(im, ax=ax, label="Ratio obs/theoretical",
                     shrink=0.85)

fig5.tight_layout()
out5 = f"{OUT_DIR_PLOT}/grafico5_heatmap_var.png"
fig5.savefig(out5, dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig5)
print(f"Saved: grafico5_heatmap_var.png")

print(f"\nAll plots saved to: {OUT_DIR_PLOT}")

In [ ]:
# CELL 29 — Final analytical figures
#
# 1. Log-volatility decomposition: X_obs = X_P + X_R (FTSE, h=1)
#    with annotations of key historical events
# 2. DFA log-log plot: slope = Hurst exponent H for X_obs, X_P, X_R
# 3. RP_Model forecast vs observed X_obs on test set (FTSE, all horizons)
# 4. RMSE benchmark: RP_Model vs HAR vs RW vs SMA20 vs EWMA94

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR_PLOT = f"{BASE_DIR}/grafici_finali_tesi"
os.makedirs(OUT_DIR_PLOT, exist_ok=True)

IDX_MAIN = "FTSE"
COL_MAIN = "FTSE_ret"
H_DECOMP = 1

C_OBS  = "#2166AC"
C_P    = "#1A9641"
C_R    = "#D73027"
C_RP   = "#FF7F0E"
C_HAR  = "#348ABD"
C_RW   = "#7B2D8B"
C_SMA  = "#FBC15E"
C_EWMA = "#8EBA42"

HORIZONS = [1, 5, 20, 60]
INDICI   = ["FTSE", "GDAXI", "GSPC", "HSI", "N225"]

def log_rv_past(r, h, eps=EPS):
    rv = np.sqrt(r.pow(2).rolling(h, min_periods=h).mean())
    return np.log(rv.clip(lower=eps))

def hurst_dfa_plot(series, min_scale=10, n_scales=18, max_scale_cap=500, min_segments=6):
    x = pd.Series(series).dropna().astype(float).values
    N = len(x)
    if N < 300: return [], [], np.nan
    max_scale = min(N // 10, max_scale_cap)
    scales = np.unique(np.floor(np.logspace(np.log10(min_scale), np.log10(max_scale), n_scales)).astype(int))
    y = np.cumsum(x - x.mean())
    used, F = [], []
    for s in scales:
        nseg = N // s
        if nseg < min_segments: continue
        y_cut = y[:nseg * s].reshape(nseg, s)
        t = np.arange(s)
        mse = [np.mean((seg - np.polyval(np.polyfit(t, seg, 1), t))**2) for seg in y_cut]
        Fs = np.sqrt(np.mean(mse))
        if np.isfinite(Fs) and Fs > 0:
            used.append(s)
            F.append(Fs)
    if len(F) < 6: return used, F, np.nan
    alpha = np.polyfit(np.log(used), np.log(F), 1)[0]
    return used, F, float(alpha)

# Load data
ret_all  = pd.read_csv(f"{BASE_DIR}/indici_log_returns.csv",
                       parse_dates=["Date"], index_col="Date").sort_index()
train    = pd.read_csv(f"{BASE_DIR}/indici_log_returns_train_70.csv",
                       parse_dates=["Date"], index_col="Date").sort_index()
bench_df = pd.read_csv(f"{BASE_DIR}/benchmark_HAR.csv")

FC_DIR_HAR = f"{BASE_DIR}/forecast_test_HAR"
FC_DIR_RP  = f"{BASE_DIR}/forecast_test_RP"

r_ftse  = ret_all[COL_MAIN].dropna()
x_obs_1 = log_rv_past(r_ftse, H_DECOMP)
span_p  = max(L_PERSIST_BASE, C_SPAN * H_DECOMP)
x_p_1   = x_obs_1.ewm(span=span_p, adjust=False, min_periods=10).mean()
x_r_1   = x_obs_1 - x_p_1


# ============================================================
# PLOT 1 — Log-volatility decomposition: X_obs = X_P + X_R
# ============================================================

fig1, axes1 = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig1.suptitle(
    f"Log-volatility decomposition: {IDX_MAIN} (h={H_DECOMP})\n"
    r"$X_{obs} = X_P + X_R$",
    fontsize=13, fontweight="bold"
)

axes1[0].plot(x_obs_1.index, x_obs_1, lw=0.7, color=C_OBS,
              alpha=0.75, label=r"$X_{obs}$ (log-sigma)")
axes1[0].plot(x_p_1.index, x_p_1, lw=2.0, color=C_P,
              label=r"$X_P$ (persistent — slow EWM filter)")
axes1[0].axvline(train.index.max(), color="black",
                 lw=1.2, ls="--", alpha=0.5, label="Train/Test split")
axes1[0].set_ylabel("log-sigma", fontsize=9)
axes1[0].legend(fontsize=9, loc="lower right")
axes1[0].grid(True, alpha=0.2)

axes1[1].fill_between(x_r_1.index, x_r_1, 0, where=x_r_1 >= 0,
                       color=C_R, alpha=0.5,
                       label=r"$X_R > 0$ (vol above persistent level)")
axes1[1].fill_between(x_r_1.index, x_r_1, 0, where=x_r_1 < 0,
                       color=C_OBS, alpha=0.5,
                       label=r"$X_R < 0$ (vol below persistent level)")
axes1[1].axhline(0, color="black", lw=0.8)
axes1[1].axvline(train.index.max(), color="black",
                 lw=1.2, ls="--", alpha=0.5)
axes1[1].set_ylabel(r"$X_R$ (rough component)", fontsize=9)
axes1[1].legend(fontsize=9, loc="lower right")
axes1[1].grid(True, alpha=0.2)

axes1[2].plot(r_ftse.index, r_ftse, lw=0.5, color="gray", alpha=0.6)
axes1[2].axvline(train.index.max(), color="black",
                 lw=1.2, ls="--", alpha=0.5, label="Train/Test split")
axes1[2].set_ylabel("Log-returns", fontsize=9)
axes1[2].set_xlabel("Date", fontsize=9)
axes1[2].legend(fontsize=9, loc="lower right")
axes1[2].grid(True, alpha=0.2)

events = {
    "GFC\n2008":        "2008-09-15",
    "COVID\n2020":      "2020-03-16",
    "Rate hikes\n2022": "2022-01-01"
}
for label, date in events.items():
    d = pd.Timestamp(date)
    if x_obs_1.index[0] < d < x_obs_1.index[-1]:
        for ax in axes1:
            ax.axvline(d, color="darkorange", lw=1.0, ls=":", alpha=0.7)
        axes1[0].text(
            d, axes1[0].get_ylim()[1] * 0.95, label,
            fontsize=7, color="darkorange", ha="center", va="top"
        )

fig1.tight_layout()
fig1.savefig(f"{OUT_DIR_PLOT}/grafico_decomposizione_XP_XR.png",
             dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig1)
print("Saved: grafico_decomposizione_XP_XR.png")


# ============================================================
# PLOT 2 — DFA log-log: slope = Hurst exponent H (tutti gli indici)
# ============================================================

EWMA_LAMBDA = 0.94  # RiskMetrics standard

fig2, axes2 = plt.subplots(2, 3, figsize=(18, 10))
axes2 = axes2.flatten()
fig2.suptitle("DFA(1) log-log — Hurst exponent H per index", fontsize=14, fontweight="bold")

COL_MAP = {idx: f"{idx}_ret" for idx in INDICI}

for idx_i, IDX in enumerate(INDICI):
    ax  = axes2[idx_i]
    col = COL_MAP[IDX]
    r   = ret_all[col].dropna()

    # EWMA proxy — identico a Cell 12
    ewma_var = r.pow(2).ewm(alpha=1 - EWMA_LAMBDA, adjust=False, min_periods=20).mean()
    x_obs_i  = 0.5 * np.log(ewma_var.clip(lower=EPS))

    # Decomposizione — span identico a Cell 12
    span_p_i = max(L_PERSIST_BASE, C_SPAN * 1)
    x_p_i    = x_obs_i.ewm(span=span_p_i, adjust=False, min_periods=10).mean()
    x_r_i    = x_obs_i - x_p_i

    tmp = pd.concat({"X_obs": x_obs_i, "X_R": x_r_i, "X_P": x_p_i}, axis=1).dropna()

    series_dfa_i = {
        r"$X_{obs}$":          (tmp["X_obs"], C_OBS, "o"),
        r"$X_P$ (persistent)": (tmp["X_P"],   C_P,   "s"),
        r"$X_R$ (rough)":      (tmp["X_R"],   C_R,   "^"),
    }

    for label, (series, color, marker) in series_dfa_i.items():
        sc_u, F_v, alpha = hurst_dfa_plot(series.values)
        if len(sc_u) < 4:
            continue
        H = alpha if alpha <= 1 else alpha - 1
        H  = float(np.clip(H, 0, 1))
        lx = np.log10(sc_u)
        ly = np.log10(F_v)
        ax.scatter(lx, ly, color=color, marker=marker, s=40, zorder=5)
        m_fit, b_fit = np.polyfit(lx, ly, 1)
        xl = np.linspace(lx.min(), lx.max(), 100)
        ax.plot(xl, m_fit * xl + b_fit, color=color, lw=2,
                label=f"{label}  H={H:.3f}")

    ax.set_title(IDX, fontsize=11, fontweight="bold")
    ax.set_xlabel(r"$\log_{10}$(scale $s$)", fontsize=9)
    ax.set_ylabel(r"$\log_{10}(F(s))$", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.25)

    # PNG individuale
    fig_s, ax_s = plt.subplots(figsize=(8, 5))
    fig_s.suptitle(f"DFA(1) log-log — {IDX}", fontsize=11, fontweight="bold")
    for label, (series, color, marker) in series_dfa_i.items():
        sc_u, F_v, alpha = hurst_dfa_plot(series.values)
        if len(sc_u) < 4:
            continue
        H = alpha if alpha <= 1 else alpha - 1
        H  = float(np.clip(H, 0, 1))
        lx = np.log10(sc_u)
        ly = np.log10(F_v)
        ax_s.scatter(lx, ly, color=color, marker=marker, s=50, zorder=5)
        m_fit, b_fit = np.polyfit(lx, ly, 1)
        xl = np.linspace(lx.min(), lx.max(), 100)
        ax_s.plot(xl, m_fit * xl + b_fit, color=color, lw=2,
                  label=f"{label}  H={H:.3f}")
    ax_s.set_xlabel(r"$\log_{10}$(scale $s$)", fontsize=10)
    ax_s.set_ylabel(r"$\log_{10}(F(s))$", fontsize=10)
    ax_s.legend(fontsize=10)
    ax_s.grid(True, alpha=0.25)
    fig_s.tight_layout()
    fig_s.savefig(f"{OUT_DIR_PLOT}/grafico_DFA_loglog_{IDX}.png", dpi=160, bbox_inches="tight")
    plt.close(fig_s)
    print(f"Saved: grafico_DFA_loglog_{IDX}.png")

axes2[5].set_visible(False)
fig2.tight_layout()
fig2.savefig(f"{OUT_DIR_PLOT}/grafico_DFA_loglog_all.png", dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig2)
print("Saved: grafico_DFA_loglog_all.png")


# ============================================================
# PLOT 3 — RP_Model forecast vs observed X_obs (FTSE, test set)
#           All four horizons
# ============================================================

fig3, axes3 = plt.subplots(2, 2, figsize=(16, 9), sharey=False)
axes3 = axes3.flatten()
fig3.suptitle(
    f"RP_Model forecast vs observed log-sigma — {IDX_MAIN} (test set)",
    fontsize=13, fontweight="bold"
)

for i, h in enumerate(HORIZONS):
    ax      = axes3[i]
    fcsv_rp = f"{FC_DIR_RP}/forecast_test_RP_{IDX_MAIN}_h{h}.csv"

    if not os.path.exists(fcsv_rp):
        ax.set_title(f"h={h} — file not found")
        continue

    fc_rp = pd.read_csv(fcsv_rp, parse_dates=["Date"],
                         index_col="Date").sort_index()

    if "xhat_rp" not in fc_rp.columns or "x_obs" not in fc_rp.columns:
        ax.set_title(f"h={h} — columns missing")
        continue

    fc_rp = fc_rp.dropna(subset=["xhat_rp", "x_obs"])

    ax.plot(fc_rp.index, fc_rp["x_obs"], lw=0.9, color=C_OBS,
            alpha=0.85, label=r"Observed $X_{obs}$")
    ax.plot(fc_rp.index, fc_rp["xhat_rp"], lw=1.2, color=C_RP,
            alpha=0.85, ls="--",
            label=r"RP_Model forecast $\hat{X}$")

    row_rp = bench_df[
        (bench_df["Indice"]    == IDX_MAIN) &
        (bench_df["Horizon_h"] == h)        &
        (bench_df["Model"]     == "RP_Model")
    ]
    if len(row_rp) > 0:
        rmse = float(row_rp["RMSE_logsigma"].values[0])
        corr = float(row_rp["Corr"].values[0])
        ax.set_title(
            f"h = {h}   |   RMSE = {rmse:.4f}   Corr = {corr:.4f}",
            fontsize=9
        )
    else:
        ax.set_title(f"h = {h}", fontsize=9)

    ax.set_ylabel("log-sigma", fontsize=8)
    ax.legend(fontsize=8, loc="lower right")
    ax.grid(True, alpha=0.2)

fig3.tight_layout()
fig3.savefig(f"{OUT_DIR_PLOT}/grafico_RP_forecast_vs_obs.png",
             dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig3)
print("Saved: grafico_RP_forecast_vs_obs.png")


# ============================================================
# PLOT 4 — RMSE benchmark: RP_Model vs HAR vs RW vs SMA20 vs EWMA94
# ============================================================

MODELS_B    = ["RP_Model", "HAR_Model", "RW", "SMA20", "EWMA94"]
COL_MODELS = {
    "RP_Model":  C_RP,
    "HAR_Model": C_HAR,
    "RW":        C_RW,
    "SMA20":     C_SMA,
    "EWMA94":    C_EWMA
}

fig4, axes4 = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
axes4 = axes4.flatten()
fig4.suptitle(
    "RMSE log-sigma on test set: RP_Model vs benchmarks\n"
    "(lower bar = better)",
    fontsize=13, fontweight="bold"
)

x = np.arange(len(INDICI))
w = 0.15

for i, h in enumerate(HORIZONS):
    ax  = axes4[i]
    sub = bench_df[bench_df["Horizon_h"] == h]

    for j, m in enumerate(MODELS_B):
        vals = [
            float(sub[(sub["Indice"] == idx) &
                       (sub["Model"]  == m)]["RMSE_logsigma"].values[0])
            if len(sub[(sub["Indice"] == idx) &
                        (sub["Model"]  == m)]) > 0 else np.nan
            for idx in INDICI
        ]
        off  = (j - len(MODELS_B) / 2 + 0.5) * w
        bars = ax.bar(x + off, vals, w,
                      color=COL_MODELS[m], alpha=0.88,
                      edgecolor="white", linewidth=0.4)
        if m == "RP_Model":
            for b in bars:
                b.set_edgecolor("black")
                b.set_linewidth(1.5)

    ax.set_title(f"h = {h}", fontsize=10, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(INDICI, fontsize=9)
    ax.set_ylabel("RMSE log-sigma" if i % 2 == 0 else "")
    ax.grid(True, alpha=0.2, axis="y")

    rp_wins = sum(
        1 for idx in INDICI
        if (
            len(sub[(sub["Indice" ] == idx) &
                     (sub["Model"]  == "RP_Model")]) > 0 and
            sub[(sub["Indice" ] == idx) &
                 (sub["Model"]  == "RP_Model")]["RMSE_logsigma"].values[0] ==
            min(sub[sub["Indice"] == idx]["RMSE_logsigma"].values)
        )
    )
    ax.text(
        0.98, 0.97, f"RP wins: {rp_wins}/5",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=9, fontweight="bold", color=C_RP,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
                  edgecolor=C_RP, alpha=0.8)
    )

handles4 = [
    plt.Rectangle((0, 0), 1, 1, color=COL_MODELS[m], label=m)
    for m in MODELS_B
]
fig4.legend(handles=handles4, loc="lower center", ncol=5,
            fontsize=10, bbox_to_anchor=(0.5, -0.03))
fig4.tight_layout()
fig4.savefig(f"{OUT_DIR_PLOT}/grafico_RMSE_benchmark_RP.png",
             dpi=160, bbox_inches="tight")
plt.show(); plt.close(fig4)
print("Saved: grafico_RMSE_benchmark_RP.png")

print(f"\nAll plots saved to: {OUT_DIR_PLOT}")